# One-Phonon Diffuse Scattering — Fit to Experimental Data (6o2h, P1)

One rigid body per unit cell (the lysozyme molecule), 6 degrees of freedom per
cell: a rotation $\Omega$ and a translation $v$. The crystal's phonons are the
normal modes of a Born–von Kármán lattice of these rigid bodies, coupled through
a small set of pairwise contact springs. This notebook fits the contact
stiffnesses to the experimental diffuse-scattering maps deposited at
[CXIDB ID 128](https://www.cxidb.org/id-128.html)
(`triclinic_lysozyme_maps.h5`), associated with

> Meisburger, S. P., Case, D. A. & Ando, N. *Diffuse X-ray Scattering from
> Correlated Motions in a Protein Crystal.* Nat. Commun. 11, 1271 (2020).

**Pipeline:** map metadata → atomic model and mass matrix → hybrid electron
density → solvent contrast → molecular transform $F,L$ → coupling vector
$G(\mathbf q)$ → contact detection, framing, and the geometric prior → dynamical
matrix → halo-profile and stratified mid-zone sampling of the experimental map →
refinement of the contact-stiffness shape with the intensity scale profiled →
sloppiness analysis → predicted vs. deposited ADPs → mode animations.

**The ADPs are the independent check, so nothing in the refinement uses them.**
In particular the fitted $K$ is not rescaled to make the predicted mean $B$ match
the deposited mean: that would make the headline comparison a tautology, and it
would also over-soften $K$, since the deposited $B$ contains internal and
substitutional disorder that a rigid-body lattice model cannot reproduce. The
predicted mean $B$ is expected to fall *below* the deposited mean, and how far
below is a result.

The $K$/scale degeneracy is instead removed from the parameter space — the
refinement runs on the normalized *shape* of $K$ — and closed with **units**: the
deposited map is in electron scattering per unit cell, and $I=G^\dagger D^{-1}G$
with $k_BT=1$ and $G$ in $e/\mathrm{Å}$ is in the same units, so $s\equiv1$ is a
physical statement rather than an arbitrary choice.

**Required input files** (same directory as this notebook): `6o2h.cif`,
`6o2h-sf.cif`, and `triclinic_lysozyme_maps.h5` (1.94 GB, from CXIDB).

In [ ]:
import io
import pathlib
from itertools import product

import numpy as np
import h5py
import gemmi
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.lines import Line2D
import scipy.constants as const
from scipy.spatial import cKDTree
from scipy.linalg import eigh
from scipy.ndimage import zoom as nd_zoom
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components as sc_connected_components
from scipy.optimize import minimize
from scipy.spatial.transform import Rotation as Rot
import imageio
import imageio.v2 as iio2
from skimage.measure import marching_cubes
from IPython.display import Image as IPImage, display
import gc

np.set_printoptions(precision=4, suppress=True)

def fig_to_image(fig, dpi=72):
    """Rasterize a matplotlib figure to an RGB array, for building GIFs."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', facecolor=fig.get_facecolor())
    buf.seek(0)
    return iio2.imread(buf)[:, :, :3]


In [ ]:
PDB_FILE = pathlib.Path("6o2h.cif")
SF_CIF   = pathlib.Path("6o2h-sf.cif")
H5_FILE  = pathlib.Path("triclinic_lysozyme_maps.h5")

# CXIDB 128 stores three Bragg-subtracted "processed" estimates of the diffuse
# component plus a raw total-scattering map and simulated comparison maps. The
# variational estimate is the one used for the lattice-dynamics fit in the
# original paper, so it is the default here.
MAP_GROUP = "maps/processed/variational"

D_MIN_MODEL    = 1.5    # Å — resolution for the visualization/reference density
RATE_MODEL     = 1.5
CONTACT_CUTOFF = 4.0    # Å — atom-atom cutoff defining a rigid-body contact
MIN_CONTACTS   = 5      # minimum atom-atom pairs for a contact to count
MAX_IMAGE      = 1      # search ±MAX_IMAGE unit cells (verified sufficient below)
TEMPERATURE    = 300.0  # K — used only for the frequency-unit conversion

# --- Resolution range for the fit -----------------------------------------
# Set deliberately rather than by an SNR heuristic, for two reasons:
#   * mean(I)/mean(sigma) per shell is not an SNR. Diffuse intensity rises
#     steeply with |q| (~q²|F|²), so the ratio climbs with resolution mostly
#     because the NUMERATOR grows, and any rule based on it selects the finest
#     shell available.
#   * high resolution is where this model is least valid. At ~2 Å the diffuse
#     scattering from a protein crystal is dominated by internal and side-chain
#     motion, not rigid-body lattice dynamics -- a central finding of the paper
#     this data comes from -- and it is also where the one-phonon approximation
#     degrades (multi-phonon terms grow as (q·u)^4).
# D_FIT_MIN is thus a model-validity statement. The SNR scan is retained below
# as a diagnostic, and R/CC are reported per resolution shell so the resolution
# at which the model stops working is measured rather than assumed.
D_FIT_MIN, D_FIT_MAX = 3.0, 8.0

N_FIT_MAX     = 6000    # mid-zone points used in the refinement
HOLDOUT_FRAC  = 0.15    # fraction held out for R_free
N_RES_STRATA  = 6       # equal-count resolution strata for the mid-zone sample
N_HALO_STRATA = 8       # equal-count Bragg-distance strata for the mid-zone sample

# A voxel counts as Bragg-adjacent only when it is close to a reciprocal-lattice
# point in ALL THREE indices. Expressed as a Cartesian radius, which is the
# physical criterion; an index-unit threshold interacts badly with the map's
# (1/11, 1/11, 1/13) sampling.
BRAGG_EXCL_CART = 0.010     # Å⁻¹

# --- Halo-profile term -----------------------------------------------------
# Rather than down-weighting individual voxels by distance from a Bragg peak,
# the halo information enters as an azimuthally-and-lattice-averaged PROFILE.
# Averaging beats the noise down by ~sqrt(N), turning ~10^6 noisy voxels (a
# fifth of them negative) into a few dozen well-determined numbers, and the
# resulting I ~ |q-G|^-2 shape with its anisotropic prefactor reads out the
# long-wavelength limit of D(q) directly -- exactly the part this data can
# determine. It also sidesteps the negative-intensity problem entirely, since
# shell averages are positive wherever there is real signal.
HALO_R_MAX    = 0.060   # Å⁻¹ — outer radius of the halo region
N_HALO_SHELLS = 14      # radial shells
N_HALO_DIRS   = 6       # angular sectors, so the profile keeps its anisotropy
HALO_WEIGHT   = 1.0     # relative weight of the halo term vs. the mid-zone term

# --- Prior and regularization ---------------------------------------------
# 1 k_BT/Å² = 0.414 N/m per atom pair, a reasonable order of magnitude for a weak
# non-covalent contact (vdW ≈ 1 N/m, H-bond ≈ 10-100 N/m). Only the prior's
# SHAPE (contact-count scaling, κ_R/κ_T ratio, patch anisotropy) is load-bearing;
# the overall magnitude is refined away by the shape normalization below.
PRIOR_K_PER_PAIR = 1.0
LAMBDA_PRIOR     = 3e-1   # weight on the geodesic distance to the prior. This
                          # is what gives the poorly-determined directions a
                          # restoring force; set it from the held-out scan below
                          # the refinement, not by eye.

REG_EIG_FLOOR = 1e-10     # relative eigenvalue floor in the matrix logarithm used
                          # by the geodesic regularizer


# --- Molecular transform from the experimental density ---------------------
# The transform is computed from the hybrid map (measured |F|, model phases),
# unwrapped onto a single molecule. D_MIN_TRANSFORM sets the density grid's
# sampling and should be comfortably finer than the resolution at which the
# diffuse model is used: masking one molecule out of a periodic map convolves
# its transform with the mask transform, so nearby |q| mix and a cutoff sitting
# exactly at the diffuse limit would bite.
D_MIN_TRANSFORM = 2.0     # Angstrom - resolution cutoff for the transform density
RATE_TRANSFORM  = 2.5     # grid oversampling rate (spacing ~ d_min/rate)
UNWRAP_IMAGE    = 1       # lattice images searched when assigning voxels to a
                          # molecule; verified sufficient at run time
TRANSFORM_MEM_MB  = 512   # memory budget for the scattered-point transform

# --- Bulk solvent ----------------------------------------------------------
# The object that moves rigidly is the CONTRAST density rho_prot - rho_sol*M,
# not the raw density: when a molecule displaces, the hole it occupies in the
# solvent moves with it, while the uniform bulk does not. SOLVENT_PROBE sets the
# excluded-volume envelope (van der Waals surface plus probe), SOLVENT_EDGE
# softens it -- a hard mask edge rings in Fourier space -- and SOLVENT_SHELL is
# the extra margin used when reading the flat solvent level off the map.
SOLVENT_PROBE = 1.1       # Angstrom beyond the vdW surface: the envelope
SOLVENT_EDGE  = 0.7       # Angstrom: tanh softening width of the envelope
SOLVENT_SHELL = 1.5       # Angstrom further out: margin for the level estimate

CHUNK_EVAL = 20000      # q-points per batch in the vectorized evaluators
BZ_NGRID   = 20         # Brillouin-zone grid for the ADP integral (convergence
                        # is checked explicitly, never assumed)
SIGMA_FLOOR_PCT = 5.0   # percentile floor on sigma, so a few anomalously small
                        # error bars cannot dominate the whole objective

np.set_printoptions(precision=4, suppress=True)

In [ ]:
st = gemmi.read_structure(str(PDB_FILE))
st.remove_hydrogens()
st.remove_waters()
st.setup_entities()

cell = st.cell
a1 = np.array(cell.orthogonalize(gemmi.Fractional(1,0,0)).tolist())
a2 = np.array(cell.orthogonalize(gemmi.Fractional(0,1,0)).tolist())
a3 = np.array(cell.orthogonalize(gemmi.Fractional(0,0,1)).tolist())
A_orth  = np.column_stack([a1, a2, a3])       # orthogonalization matrix
B_recip = 2*np.pi * np.linalg.inv(A_orth).T   # reciprocal lattice (columns)

print(f"Cell: {cell.a:.3f}×{cell.b:.3f}×{cell.c:.3f} Å  "
      f"α={cell.alpha:.2f} β={cell.beta:.2f} γ={cell.gamma:.2f}°")
print(f"Space group: {st.spacegroup_hm}   Volume: {cell.volume:.1f} Å³")


## Experimental Map Metadata

The deposited stiffness model must be evaluated on **exactly** the same
$(h,k,l)$ grid the experimental map is sampled on for the two to be
compared voxel-by-voxel rather than merely visually. CXIDB 128 stores each
map group's grid as `grid_ori` (fractional Miller index of the first array
element) and `grid_delta` (spacing between array elements along each axis) —
i.e. the reciprocal-lattice vectors are subdivided by an independent integer
factor $(P_1,P_2,P_3)$ along $\mathbf a^*,\mathbf b^*,\mathbf c^*$
($13\times11\times11$ in the original deposition). Everything below reads
$(P_1,P_2,P_3)$ directly from the file's own metadata — rather than assuming
a value — and uses it consistently for the model's own fine reciprocal grid,
so that a model-evaluated point and an experimental voxel at the same
$(h,k,l)$ are the same physical point, not merely nearby ones.


In [ ]:
def _h5_floats(x):
    """Robustly pull one or more Python floats out of an HDF5 attribute,
    regardless of whether h5py hands it back as a bare Python/NumPy scalar,
    a 0-d array, or a length-N array (MATLAB-written attributes, as here,
    are commonly stored as arrays even for a single value)."""
    return [float(v) for v in np.atleast_1d(np.asarray(x)).ravel()]

def _h5_float(x):
    return _h5_floats(x)[0]

def _h5_str(x):
    """Robustly pull a Python str out of an HDF5 attribute that may come
    back as bytes, a NumPy bytes_/str_ scalar, or a length-1 array of any
    of those."""
    v = np.atleast_1d(np.asarray(x)).ravel()[0]
    return v.decode() if isinstance(v, bytes) else str(v)

with h5py.File(H5_FILE, 'r') as f:
    xattrs = dict(f['/crystal'].attrs)
    grp = f[f'/{MAP_GROUP}']
    grid_size  = tuple(int(round(v)) for v in _h5_floats(grp.attrs['grid_size']))
    grid_ori   = tuple(_h5_floats(grp.attrs['grid_ori']))
    grid_delta = tuple(_h5_floats(grp.attrs['grid_delta']))
    map_description = _h5_str(grp.attrs['Description']) if 'Description' in grp.attrs else MAP_GROUP
    map_units = _h5_str(grp.attrs['units']) if 'units' in grp.attrs else 'unknown'

    # The grid_size/grid_ori/grid_delta attributes describe the dataset's
    # axes in a fixed order, but the dataset's own on-disk axis order isn't
    # guaranteed to match: HDF5 files written from MATLAB (column-major)
    # commonly come out axis-reversed once read by h5py (row-major), so the
    # array's actual .shape can be grid_size reversed rather than grid_size
    # itself. Reconcile against the real dataset shape rather than trusting
    # the attribute order blindly.
    actual_shape = tuple(grp['I'].shape)
    if actual_shape == grid_size:
        pass
    elif actual_shape == grid_size[::-1]:
        print(f"NOTE: dataset 'I' shape {actual_shape} is axis-reversed relative to "
              f"the grid_size/grid_ori/grid_delta attribute order {grid_size} -- "
              f"reversing the metadata to match the file's actual on-disk axis order.")
        grid_size, grid_ori, grid_delta = grid_size[::-1], grid_ori[::-1], grid_delta[::-1]
    else:
        raise ValueError(f"Dataset 'I' shape {actual_shape} matches neither the "
                          f"grid_size attribute {grid_size} nor its reverse -- "
                          f"inspect '/{MAP_GROUP}' in the file manually.")
    assert tuple(grp['sigma'].shape) == actual_shape, \
        "'I' and 'sigma' datasets have different shapes -- unexpected file layout"

cell_h5 = dict(a=_h5_float(xattrs['a']), b=_h5_float(xattrs['b']), c=_h5_float(xattrs['c']),
               alpha=_h5_float(xattrs['alpha']), beta=_h5_float(xattrs['beta']),
               gamma=_h5_float(xattrs['gamma']), sg=int(round(_h5_float(xattrs['spaceGroupNumber']))))

print(f"h5 crystal: a={cell_h5['a']:.3f} b={cell_h5['b']:.3f} c={cell_h5['c']:.3f}  "
      f"α={cell_h5['alpha']:.2f} β={cell_h5['beta']:.2f} γ={cell_h5['gamma']:.2f}  "
      f"SG#{cell_h5['sg']}")
print(f"Map group '/{MAP_GROUP}': {map_description}  [{map_units}]")
print(f"grid_size={grid_size}  grid_ori={grid_ori}  grid_delta={grid_delta}")

for name, dep, exp in [('a',cell.a,cell_h5['a']), ('b',cell.b,cell_h5['b']), ('c',cell.c,cell_h5['c']),
                        ('alpha',cell.alpha,cell_h5['alpha']), ('beta',cell.beta,cell_h5['beta']),
                        ('gamma',cell.gamma,cell_h5['gamma'])]:
    rel_err = abs(dep-exp)/max(abs(dep), 1e-6)
    flag = '  <-- MISMATCH' if rel_err > 5e-3 else ''
    print(f"  {name}: deposited {dep:.4f}  vs  h5 {exp:.4f}{flag}")

# P1,P2,P3: integer subdivision of each reciprocal axis, read from the data
# itself rather than assumed.
P1 = int(round(1.0/grid_delta[0]))
P2 = int(round(1.0/grid_delta[1]))
P3 = int(round(1.0/grid_delta[2]))
assert abs(1.0/P1 - grid_delta[0]) < 1e-4, "grid_delta[0] is not a clean 1/integer subdivision"
assert abs(1.0/P2 - grid_delta[1]) < 1e-4, "grid_delta[1] is not a clean 1/integer subdivision"
assert abs(1.0/P3 - grid_delta[2]) < 1e-4, "grid_delta[2] is not a clean 1/integer subdivision"
print(f"q-resolution matched to the experimental map: (P1,P2,P3) = ({P1},{P2},{P3})")

def hkl_index(h, k, l):
    """Nearest (i, j, k_idx) integer array index for fractional Miller
    index (h,k,l) on this map group's grid; also returns how far off the
    nearest grid point actually was (should be ~0 for points our own model
    grid produces)."""
    fi = (h - grid_ori[0]) / grid_delta[0]
    fj = (k - grid_ori[1]) / grid_delta[1]
    fk = (l - grid_ori[2]) / grid_delta[2]
    i, j, kx = round(fi), round(fj), round(fk)
    err = max(abs(fi-i), abs(fj-j), abs(fk-kx))
    return i, j, kx, err

def in_bounds(i, j, k):
    return (0 <= i < grid_size[0]) and (0 <= j < grid_size[1]) and (0 <= k < grid_size[2])


## Electron Density and the Molecular Transform

The density is the **hybrid map**: experimentally measured structure-factor
amplitudes with phases from the refined model. Compared with an isolated-atom
form-factor table it carries bonding density, ordered solvent, and the real static
and dynamic disorder.

It is also the appropriate density on physical grounds. The hybrid map is the
crystallographic **average** density — the instantaneous molecular density
convolved with the displacement distribution, $F_{\rm avg}=F_0e^{-W}$. The
one-phonon intensity requires $|F_0|^2e^{-2W}$, which is exactly
$|F_{\rm avg}|^2$, so the Debye–Waller factor enters with no separate bookkeeping
and no choice about whether to apply the deposited ADPs.

$F(000)$ is not a measured reflection, so the map integrates to zero over the
cell: it is the true density minus the **cell average** $\bar\rho=Z_{\rm cell}/V$.
That is distinct from the bulk solvent level, which is measured from the map in
the solvent section below.

### Unwrapping onto a single molecule

The grid holds the **crystal** density
$\rho_c(\mathbf r)=\sum_{\mathbf n}\rho_{\rm mol}(\mathbf r-\mathbf T_{\mathbf n})$
folded into one box. Transforming it directly gives

$$F_{\rm cell}(\mathbf q)=\sum_{\mathbf n}e^{-i\mathbf q\cdot\mathbf T_{\mathbf n}}
\int_{\rm box-\mathbf T_{\mathbf n}}\rho_{\rm mol}(\mathbf r')
e^{-i\mathbf q\cdot\mathbf r'}\,d^3r'$$

At **integer** $hkl$ every phase factor is 1 and $F_{\rm cell}=F_{\rm mol}$. At
the **fractional** $hkl$ this pipeline runs on they are not, and the parts of the
molecule folded across a cell boundary return with the wrong phase. No roll of the
grid avoids this for a protein, whose molecular region tiles space.

Each voxel is therefore assigned to the molecular image it belongs to — nearest
atom surface over all lattice images — giving it an integer translation
$\mathbf n_v$. Its position in the molecule's own frame is
$\mathbf r_v-\mathbf T_{\mathbf n_v}$, so

$$F_{\rm mol}(\mathbf q)=\sum_v\rho_v\,
e^{-i\mathbf q\cdot\mathbf r_v}\;e^{+2\pi i\,(h,k,l)\cdot\mathbf n_v}\,dV$$

The correction is a per-voxel phase, equal to 1 at integer $hkl$, so this reduces
to the ordinary structure factor there. $\mathbf n_v$ takes only a handful of
distinct values, so the sum splits into a small number of ordinary transforms, one
per translation group — which also lets each group use the separable axis-by-axis
contraction. Checked below against brute-force summation over explicitly unwrapped
coordinates.

*Provenance.* The density-based approach follows Meisburger, Case & Ando; the
grouping-by-translation formulation here is reconstructed from the physics rather
than transcribed, so the specifics are worth checking against the paper and the
`mdx2` source.

In [ ]:
# --- Atomic model: positions, masses, deposited ADPs, mass matrix ----------
# Built FIRST, because r_cm_at is the reference point for G, for D, and for the
# ADP projection alike -- the mass matrix M = diag(J, m I) is block-diagonal only
# about the centre of mass, and the (Omega, v) split of a rigid displacement
# depends on which point you reference it to.
ATOMIC_MASS = {'C':12.011,'N':14.007,'O':15.999,'S':32.06,'P':30.974,
               'SE':78.96,'H':1.008,'FE':55.845,'ZN':65.38,'CA':40.078}

masses, apos, b_exp, u_exp, has_aniso, elem_names, Zs = [], [], [], [], [], [], []
for ch in st[0]:
    for res in ch:
        for atom in res:
            masses.append(ATOMIC_MASS.get(atom.element.name.upper(), 12.0))
            apos.append(atom.pos.tolist())
            b_exp.append(atom.b_iso)
            u_exp.append(atom.aniso.as_mat33().tolist())
            has_aniso.append(atom.aniso.nonzero())
            elem_names.append(atom.element.name)
            Zs.append(atom.element.atomic_number)
masses = np.array(masses); apos = np.array(apos); b_exp = np.array(b_exp)
u_exp = np.array(u_exp); has_aniso = np.array(has_aniso)
Z_model = float(np.sum(Zs))          # electrons in the deposited model
all_pos = apos                      # same array, used by the contact search below
n_atoms = len(apos)

m_total = masses.sum()
r_cm_at = (masses[:, None]*apos).sum(0)/m_total     # THE reference point
dr_a    = apos - r_cm_at
r2      = (dr_a**2).sum(1)
J       = (masses[:, None, None]*(r2[:, None, None]*np.eye(3)[None]
          - dr_a[:, :, None]*dr_a[:, None, :])).sum(0)
M_mat   = np.block([[J, np.zeros((3,3))], [np.zeros((3,3)), m_total*np.eye(3)]])
Msq     = np.linalg.cholesky(M_mat)
Msq_inv = np.linalg.inv(Msq)

kBT_SI     = const.k * TEMPERATURE
omega_unit = np.sqrt(kBT_SI / (const.atomic_mass * (1e-10)**2))
freq_unit  = omega_unit / (2*np.pi) / 1e12   # THz per sqrt(reduced stiffness/mass)

print(f"{n_atoms} atoms   total mass {m_total:.0f} amu")
print(f"Centre of mass r_cm_at = ({r_cm_at[0]:.3f}, {r_cm_at[1]:.3f}, {r_cm_at[2]:.3f}) Å")
print(f"Deposited ANISOU records present for {has_aniso.mean():.1%} of atoms")
print(f"Deposited mean B = {b_exp.mean():.2f} Å²")
print(f"Frequency unit: 1 (k_BT/amu/Å²)^½ = {freq_unit:.3f} THz")
print(f"Model electron count Z = {Z_model:.0f} e⁻ "
      f"(no hydrogens in the deposited model, so ~12-14% low)")


In [ ]:
# --- Hybrid density: measured |F|, model phases ----------------------------
# Phases come from the calculated model density; amplitudes are the experimental
# ones wherever a measurement exists. F(000) is not measured, so the result has
# zero mean -- it is a CONTRAST density about the cell average, which is exactly
# what scatters at q != 0 (see markdown above).

def _good_fft_size(n):
    """Smallest even 2,3,5-smooth integer >= n (FFT-friendly)."""
    n = max(4, int(np.ceil(n)))
    while True:
        m = n
        for p in (2, 3, 5):
            while m % p == 0:
                m //= p
        if m == 1 and n % 2 == 0:
            return n
        n += 1

# --- Measured amplitudes ---------------------------------------------------
doc_sf = gemmi.cif.read(str(SF_CIF))
rb     = gemmi.as_refln_blocks(doc_sf)[0]
miller = np.array(rb.make_miller_array())
F_meas = np.array(rb.make_float_array("F_meas_au"))
ok = np.isfinite(F_meas) & (F_meas > 0)
miller, F_meas = miller[ok], F_meas[ok]

q_refl = miller @ B_recip.T
d_refl = 2*np.pi/np.maximum(np.linalg.norm(q_refl, axis=1), 1e-9)
use = d_refl >= D_MIN_TRANSFORM
mil_u, Fm_u = miller[use], F_meas[use]

# --- Transform grid --------------------------------------------------------
# Sampling is set by D_MIN_TRANSFORM, which should be comfortably finer than the
# resolution at which the diffuse model is used -- masking a molecule out of a
# periodic map convolves its transform with the mask transform, so nearby |q|
# mix and a cutoff exactly at the diffuse limit would bite.
#
# The grid must also satisfy n_i >= 2*h_max_i + 1 or distinct reflections fold
# onto the same point. h_max is taken from the actual indices rather than from
# a/d_min, which under-estimates it for a triclinic cell.
h_lim = np.abs(mil_u).max(axis=0)
n_tr = [_good_fft_size(max(RATE_TRANSFORM*L/D_MIN_TRANSFORM, 2*hm + 1))
        for L, hm in zip((cell.a, cell.b, cell.c), h_lim)]
nu, nv, nw = n_tr
spacing = np.array([cell.a/nu, cell.b/nv, cell.c/nw])

# --- Model phases ----------------------------------------------------------
dc = gemmi.DensityCalculatorX()
dc.d_min = D_MIN_MODEL; dc.rate = RATE_MODEL
dc.set_grid_cell_and_spacegroup(st)
dc.initialize_grid()
dc.add_model_density_to_grid(st[0])
dc.grid.symmetrize_sum()
rho_model = np.array(dc.grid)                   # also reused for the isosurface
nu_m, nv_m, nw_m = rho_model.shape
assert min(nu_m, nv_m, nw_m) >= 2*h_lim.max() + 1, "model grid too coarse for phase lookup"
F_calc_grid = np.fft.ifftn(rho_model)

ph = np.angle(F_calc_grid[mil_u[:,0] % nu_m, mil_u[:,1] % nv_m, mil_u[:,2] % nw_m])
F_hyb = Fm_u*np.exp(1j*ph)

# rho(r) = (1/V) sum_h F_h exp(-2 pi i h.r) -- numpy's forward fftn convention.
F_grid = np.zeros((nu, nv, nw), dtype=complex)
F_grid[mil_u[:,0] % nu, mil_u[:,1] % nv, mil_u[:,2] % nw] = F_hyb/cell.volume
F_grid[(-mil_u[:,0]) % nu, (-mil_u[:,1]) % nv, (-mil_u[:,2]) % nw] = F_hyb.conj()/cell.volume
rho = np.fft.fftn(F_grid).real
dV  = cell.volume/rho.size

print(f"Transform grid: {nu}x{nv}x{nw} = {rho.size} voxels, "
      f"spacing {spacing.round(2)} A  (D_MIN_TRANSFORM = {D_MIN_TRANSFORM} A)")
print(f"Reflections used: {use.sum()} of {len(miller)}  "
      f"({d_refl[use].min():.2f}-{d_refl[use].max():.2f} A), |h|max = {tuple(h_lim)}")
print(f"Density RMS {rho.std():.4f} e-/A^3; cell integral "
      f"{rho.sum()*dV:.3f} e- (~0 by construction, F(000) unmeasured)")

# Missing low-order terms distort the molecular envelope far more than missing
# high-order ones, so report the low-resolution completeness explicitly.
n_low = int((d_refl[use] > 8.0).sum())
print(f"Reflections beyond 8 A: {n_low} "
      f"({'SPARSE -- the envelope may be distorted' if n_low < 30 else 'adequate'})")


In [ ]:
# --- Unwrap the periodic density onto a single molecule --------------------
# Every voxel is assigned to the molecular image whose atom surface it sits
# nearest, over all lattice translations in +-UNWRAP_IMAGE. That assignment is a
# fundamental domain of the lattice: one voxel per orbit, so no density is
# double-counted or lost, and the images tile space.
#
# The assignment yields an integer translation n_v per voxel. In the molecule's
# own frame the voxel sits at r_v - T_{n_v}, so the transform picks up a per-voxel
# phase exp(+2*pi*i*(h,k,l).n_v) -- exactly 1 at integer hkl, which is why this
# reduces to the ordinary structure factor where it must.
#
# Assignment is by distance to the atom SURFACE (centre distance minus the van
# der Waals radius) rather than to the atom centre, so a large atom does not lose
# surface voxels to a small one nearby.

VDW_RADIUS = {'C':1.70,'N':1.55,'O':1.52,'S':1.80,'P':1.80,'H':1.20,
              'SE':1.90,'FE':2.00,'ZN':1.39,'CA':2.31,'MG':1.73,'CL':1.75}
vdw_at = np.array([VDW_RADIUS.get(e.upper(), 1.70) for e in elem_names])

ii = np.arange(nu)/nu; jj = np.arange(nv)/nv; kk = np.arange(nw)/nw
Ig, Jg, Kg = np.meshgrid(ii, jj, kk, indexing='ij')
r_grid = (A_orth @ np.stack([Ig, Jg, Kg]).reshape(3, -1)).T      # (N_vox, 3)
N_VOX  = len(r_grid)

_imgs = np.array([(i, j, k) for i in range(-UNWRAP_IMAGE, UNWRAP_IMAGE+1)
                            for j in range(-UNWRAP_IMAGE, UNWRAP_IMAGE+1)
                            for k in range(-UNWRAP_IMAGE, UNWRAP_IMAGE+1)])
_T_img   = _imgs @ A_orth.T
_pos_img = np.vstack([apos + T for T in _T_img])
_img_of  = np.repeat(np.arange(len(_imgs)), n_atoms)
_vdw_img = np.tile(vdw_at, len(_imgs))

# Query enough neighbours that the surface-distance winner is certainly among
# them, then re-rank by surface distance.
_tree_img = cKDTree(_pos_img)
_K_NEAR = 8
_dc, _ic = _tree_img.query(r_grid, k=_K_NEAR, workers=-1)
_ds_cand = _dc - _vdw_img[_ic]                      # distance to each atom SURFACE
_best    = np.argmin(_ds_cand, axis=1)
_rows    = np.arange(N_VOX)
_idx_near  = _ic[_rows, _best]
d_surface  = _ds_cand[_rows, _best]                 # (N_vox,) can be negative inside atoms
grp_img    = _img_of[_idx_near]
n_of_vox   = _imgs[grp_img]
r_mol      = r_grid - n_of_vox @ A_orth.T           # molecule-frame positions

groups  = np.unique(grp_img)
GRP_IDX = [np.where(grp_img == g)[0] for g in groups]
GRP_N   = _imgs[groups].astype(float)

print(f"Voxels assigned to {len(groups)} distinct lattice images "
      f"(of {len(_imgs)} searched, UNWRAP_IMAGE={UNWRAP_IMAGE})")
for g in groups:
    m = grp_img == g
    print(f"   n={tuple(_imgs[g])}: {m.sum():7d} voxels ({100*m.mean():5.1f}%)")

# Sufficiency of UNWRAP_IMAGE: a large share of voxels in NON-ZERO images is
# expected and harmless -- the molecule genuinely reaches into its neighbours.
# The meaningful test is whether widening the search by one shell would claim any
# voxels at all, so the assignment is repeated with UNWRAP_IMAGE+1 and the
# fraction landing beyond the original radius is reported.
def _assign_images(max_img):
    im  = np.array([(i, j, k) for i in range(-max_img, max_img+1)
                              for j in range(-max_img, max_img+1)
                              for k in range(-max_img, max_img+1)])
    pos = np.vstack([apos + T for T in im @ A_orth.T])
    dc, ic = cKDTree(pos).query(r_grid, k=_K_NEAR, workers=-1)
    ds = dc - np.tile(vdw_at, len(im))[ic]
    b  = np.argmin(ds, axis=1)
    return im, np.repeat(np.arange(len(im)), n_atoms)[ic[_rows, b]], ds[_rows, b]

_im_ext, _g_ext, _ = _assign_images(UNWRAP_IMAGE + 1)
_beyond = float((np.abs(_im_ext[_g_ext]).max(axis=1) > UNWRAP_IMAGE).mean())
if _beyond > 1e-3:
    print(f"NOTE: widening the search to {UNWRAP_IMAGE+1} shells claims {_beyond:.2%} "
          f"of voxels; raise UNWRAP_IMAGE.")
else:
    print(f"Widening the search to {UNWRAP_IMAGE+1} shells claims {_beyond:.3%} of "
          f"voxels: UNWRAP_IMAGE={UNWRAP_IMAGE} is sufficient.")

_span = r_mol.max(0) - r_mol.min(0)
print(f"Unwrapped molecular extent: {_span.round(1)} Å "
      f"(cell is {cell.a:.1f}×{cell.b:.1f}×{cell.c:.1f} Å -- the unwrapped molecule "
      f"is expected to be LARGER than the cell in at least one direction)")


### Bulk Solvent: the Excluded-Volume Contrast

**What moves.** Write the crystal density as protein plus a flat bulk solvent
filling the complement of the molecular envelope $M$:

$$\rho_c(\mathbf r)=\sum_{\mathbf n}\rho_{\rm prot}(\mathbf r-\mathbf T_{\mathbf n})
+\rho_{\rm sol}\Bigl[1-\sum_{\mathbf n}M(\mathbf r-\mathbf T_{\mathbf n})\Bigr]
=\underbrace{\rho_{\rm sol}}_{\text{static, uniform}}
+\sum_{\mathbf n}\underbrace{\bigl[\rho_{\rm prot}-\rho_{\rm sol}M\bigr]}
_{\textstyle\rho_{\rm eff},\ \text{moves rigidly}}(\mathbf r-\mathbf T_{\mathbf n})$$

When a molecule displaces, its atoms move and so does the hole it occupies in the
solvent — water rearranges behind it on a timescale far shorter than a phonon
period. The uniform bulk term does not move. The object whose motion generates
one-phonon diffuse scattering is therefore the **contrast** density
$\rho_{\rm eff}=\rho_{\rm prot}-\rho_{\rm sol}M$, the same quantity that governs
solution scattering.

**Negative regions.** Inside the envelope, wherever the protein's own density
falls below the bulk solvent level — interatomic voids, and the shell between the
flat-solvent boundary and the first atoms — the contrast is negative: those
regions hold fewer electrons than the solvent they displaced. Along a line running
from bulk solvent into the protein the profile is a flat plateau at
$\rho_{\rm sol}$, a dip **below** it through the excluded-volume shell, then a
rise to the atomic peaks. The first diagnostic panel below is that profile, and
it is the direct check on the subtraction.

**Babinet.** The solvent occupies the *complement* of the envelope, and
$\mathcal F[1]=\delta(\mathbf q)$, so

$$\mathcal F\bigl[\rho_{\rm sol}(1-M)\bigr]
=\rho_{\rm sol}V\,\delta_{\mathbf q,0}-\rho_{\rm sol}\mathcal F[M]
\;\xrightarrow{\ \mathbf q\neq0\ }\;-\rho_{\rm sol}\,\mathcal F[M]$$

Complementary regions scatter identically apart from the direct beam. That
equivalence is what allows solvent spread over the whole complement to be
represented as a **negative** density confined inside the envelope, making the
moving object compactly supported. It is the same flat bulk-solvent model used in
refinement, where $F_{\rm sol}=-k_{\rm sol}e^{-B_{\rm sol}s^2}F_{\rm mask}$.
Transforming a density with negative regions needs nothing special — the Fourier
transform is linear. Babinet justifies the *form* of $\rho_{\rm eff}$, not its
evaluation.

**Consequences for scale.** The contrast electron count is roughly
$Z_{\rm prot}-\rho_{\rm sol}V_{\rm mask}$, several times smaller than
$Z_{\rm prot}$, so $|F|$ at low $q$ drops by that factor and intensity by its
square. The effect is confined to low resolution: at high $hkl$ the envelope
transform $\mathcal F[M]$ is negligible and $F_{\rm eff}\to F_{\rm meas}$, which
the resolution-binned check in the transform cell confirms.

The envelope also makes the image-assignment boundary harmless. That boundary lies
in bulk solvent where $M\approx0$, so voxels with an ambiguous assignment
contribute nothing. At crystal contacts the envelopes of neighbouring molecules
overlap, and the nearest-surface rule splits the shared density between them.

In [ ]:
# --- Flat bulk-solvent level ----------------------------------------------
# Read off the map itself, in the map's own offset, so no assumption about
# absolute scale is needed. "Deep solvent" = beyond the probe shell and a further
# margin, to stay clear of the transition region at the envelope boundary.
_deep = d_surface > (SOLVENT_PROBE + SOLVENT_SHELL)
if _deep.sum() < 0.005*N_VOX:
    print(f"WARNING: only {_deep.sum()} voxels ({100*_deep.mean():.3f}%) are deep "
          f"solvent; falling back to the outermost 2% by surface distance.")
    _deep = d_surface > np.quantile(d_surface, 0.98)
rho_sol_map = float(np.median(rho.ravel()[_deep]))

print(f"Deep-solvent voxels: {_deep.sum()} ({100*_deep.mean():.1f}% of the cell), "
      f"beyond {SOLVENT_PROBE + SOLVENT_SHELL:.1f} Å from any atom surface")
print(f"Flat solvent level in the map: rho_sol = {rho_sol_map:+.4f} e⁻/Å³")

# --- Excluded-volume envelope ---------------------------------------------
# Standard bulk-solvent mask: within SOLVENT_PROBE of an atom's van der Waals
# surface. Softened with a tanh ramp of width SOLVENT_EDGE rather than left as a
# step -- a hard mask edge rings in Fourier space, and the soft ramp is the real
# -space counterpart of the B_sol smearing applied to F_mask in refinement.
M_soft = 0.5*(1.0 - np.tanh((d_surface - SOLVENT_PROBE)/SOLVENT_EDGE))
V_mask = float(M_soft.sum()*dV)

# --- The density that moves rigidly ---------------------------------------
rho_eff = ((rho.ravel() - rho_sol_map)*M_soft).reshape(rho.shape)

# Absolute scale. The map is rho_true - rho_bar with rho_bar = Z_cell/V, and the
# cell holds protein plus solvent: Z_cell = Z_prot + rho_sol_abs*V_solv. With
# rho_sol_abs = rho_sol_map + rho_bar, that pair solves to
#     rho_bar = (Z_prot + rho_sol_map*V_solv)/V_mask
# The result should land near 0.33 e-/A^3 for water; the deposited model has no
# hydrogens, so Z_prot is low by ~12% and this check reads low with it.
V_solv      = cell.volume - V_mask
rho_bar     = (Z_model + rho_sol_map*V_solv)/V_mask
rho_sol_abs = rho_sol_map + rho_bar
print(f"Implied cell-average density:     {rho_bar:.4f} e⁻/Å³")
print(f"Implied absolute solvent density: {rho_sol_abs:.4f} e⁻/Å³ "
      f"(bulk water ≈ 0.335, no H in the model)")

_inside   = M_soft > 0.5
_neg_frac = float((rho_eff.ravel()[_inside] < 0).mean())
Q_contrast = float(rho_eff.sum()*dV)

print(f"\nEnvelope volume: {V_mask:.0f} Å³ ({100*V_mask/cell.volume:.0f}% of the cell)")
print(f"Contrast electrons  ∫rho_eff dV        = {Q_contrast:8.1f} e⁻")
print(f"  Z_model - rho_sol_abs*V_mask         = {Z_model - rho_sol_abs*V_mask:8.1f} e⁻")
print(f"  raw model electron count             = {Z_model:8.1f} e⁻")
print(f"  contrast/raw ratio: {Q_contrast/Z_model:.3f} in |F|, "
      f"{(Q_contrast/Z_model)**2:.4f} in intensity at low q")
# Since the map integrates to zero over the cell, ∫rho_eff dV = -rho_sol*V_cell
# exactly IF the map is flat at rho_sol throughout the solvent region. The gap
# between the two measures how well that flat-solvent assumption holds.
print(f"  flat-solvent identity  -rho_sol*V_cell = {-rho_sol_map*cell.volume:8.1f} e⁻  "
      f"(gap {100*abs(Q_contrast + rho_sol_map*cell.volume)/abs(Q_contrast):.0f}%)")
print(f"Negative-density fraction inside the envelope: {_neg_frac:.1%}")
print("  (nonzero is CORRECT: excluded-volume regions hold fewer electrons than")
print("   the solvent they displaced -- see the profile below.)")

# --- Weight channels for the transform ------------------------------------
# [rho_eff, rho_eff*(x-x_cm), rho_eff*(y-y_cm), rho_eff*(z-z_cm)] x dV, so one
# matrix product per translation group yields F and all three components of L.
# The lever arm uses the MOLECULE-FRAME position and the MASS centre of mass.
_w = rho_eff.ravel()*dV
_d = r_mol - r_cm_at
WL_grid = np.column_stack([_w, _d[:,0]*_w, _d[:,1]*_w, _d[:,2]*_w])

_rho_cm = ((r_mol*_w[:, None]).sum(0)/_w.sum()) if abs(_w.sum()) > 1e-9 else np.full(3, np.nan)
print(f"\nContrast-density centroid: {_rho_cm.round(2)} Å")
print(f"Atomic centre of mass:     {r_cm_at.round(2)} Å")
print(f"  separation: {np.linalg.norm(_rho_cm - r_cm_at):.2f} Å  "
      f"(a few Å is normal for a contrast density; tens of Å means the unwrapping "
      f"or the mask failed)")

# --- The profile Meisburger sketched --------------------------------------
_edges = np.linspace(-1.5, min(6.0, d_surface.max()), 60)
_cen   = 0.5*(_edges[:-1] + _edges[1:])
_bi    = np.clip(np.digitize(d_surface, _edges)-1, 0, len(_cen)-1)
_prof_raw = np.array([rho.ravel()[_bi==b].mean() if (_bi==b).any() else np.nan
                      for b in range(len(_cen))])
_prof_eff = np.array([rho_eff.ravel()[_bi==b].mean() if (_bi==b).any() else np.nan
                      for b in range(len(_cen))])
_prof_m   = np.array([M_soft[_bi==b].mean() if (_bi==b).any() else np.nan
                      for b in range(len(_cen))])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
ax = axes[0]
ax.plot(_cen, _prof_raw, lw=1.8, color='steelblue', label=r'hybrid map $\rho$')
ax.axhline(rho_sol_map, color='crimson', ls='--', lw=1.2,
           label=r'flat solvent level $\rho_{sol}$')
ax.axhline(0, color='0.7', lw=0.8)
ax.axvline(SOLVENT_PROBE, color='k', ls=':', lw=1, label='envelope boundary')
ax.set_xlabel('distance from nearest atom surface (Å)')
ax.set_ylabel(r'$\rho$ (e$^-$/Å$^3$)')
ax.set_title('Density profile: protein → solvent'); ax.legend(fontsize=7)

ax = axes[1]
ax.plot(_cen, _prof_eff, lw=1.8, color='darkgreen', label=r'contrast $\rho_{eff}$')
ax.fill_between(_cen, 0, np.where(_prof_eff < 0, _prof_eff, 0), color='crimson',
                alpha=0.3, label='excluded-volume deficit')
ax.axhline(0, color='k', lw=0.9)
ax.axvline(SOLVENT_PROBE, color='k', ls=':', lw=1)
ax.set_xlabel('distance from nearest atom surface (Å)')
ax.set_ylabel(r'$\rho_{eff}$ (e$^-$/Å$^3$)')
ax.set_title('What moves rigidly (note the negative shell)'); ax.legend(fontsize=7)

ax = axes[2]
ax.hist(rho_eff.ravel()[_inside], bins=120, color='slateblue')
ax.axvline(0, color='k', lw=1)
ax.set_yscale('log')
ax.set_xlabel(r'$\rho_{eff}$ inside the envelope (e$^-$/Å$^3$)')
ax.set_ylabel('voxels')
ax.set_title(f'Contrast distribution ({_neg_frac:.0%} negative)')
plt.tight_layout(); plt.savefig('solvent_contrast.png', dpi=150); plt.show()

print("Left panel is the profile to check against the physical picture: a flat")
print("plateau in bulk solvent, a dip BELOW it through the excluded-volume shell,")
print("then the rise to the atomic peaks. If the plateau is not flat, or the")
print("dashed line does not sit on it, the solvent level is misestimated.")


## Coupling Vector $G(\mathbf{q})$

$$G = \begin{pmatrix}G_R\\G_T\end{pmatrix}
    = \begin{pmatrix}i\mathbf{q}\times L(\mathbf{q})\\
                      i\mathbf{q}\,F(\mathbf{q})\end{pmatrix},
\quad F = \int\rho_{\rm eff}\,e^{-i\mathbf{q}\cdot\mathbf{r}}d^3r,
\quad L = \int(\mathbf{r}-\mathbf{r}_{\rm cm})\,\rho_{\rm eff}\,
          e^{-i\mathbf{q}\cdot\mathbf{r}}d^3r$$

with $\rho_{\rm eff}$ the unwrapped single-molecule contrast density and
$\mathbf r_{\rm cm}=$ `r_cm_at`, the atomic centre of mass — the same reference
point the dynamical matrix uses. That choice is forced: $M=\mathrm{diag}(J,mI)$ is
block-diagonal only about the centre of mass, and the $(\Omega,v)$ split of a
rigid displacement depends on the reference point, so $G$ and $D$ must share one.

$G$ is complex, and $I=G^\dagger D^{-1}G$ is a Hermitian quadratic form in a
complex Hermitian $D$; both stay complex throughout.

In [ ]:
# --- Molecular transform F(q), L(q) from the unwrapped density -------------
#
#   F(q) = sum_v rho_v e^{-i q.r_v} e^{+2 pi i (h,k,l).n_v} dV
#   L(q) = sum_v rho_v (r_v - T_{n_v} - r_cm) e^{-i q.r_v} e^{+2 pi i (h,k,l).n_v} dV
#
# Split by translation group: within a group the phase correction is a single
# scalar, so each group is an ordinary Fourier sum over a subset of the grid.

def F_L_batch(h_arr, k_arr, l_arr, chunk=None):
    """F and L at arbitrary fractional Miller indices. Memory is bounded by a
    (chunk x group-size) phase matrix; chunk is derived from a fixed budget."""
    h_arr = np.asarray(h_arr, float); k_arr = np.asarray(k_arr, float)
    l_arr = np.asarray(l_arr, float)
    N = len(h_arr)
    F_out = np.zeros(N, complex); L_out = np.zeros((N, 3), complex)
    if N == 0:
        return F_out, L_out
    chunk = chunk or max(1, int(TRANSFORM_MEM_MB*1024**2 / (16*max(N_VOX, 1))))
    for s in range(0, N, chunk):
        sl = slice(s, min(s+chunk, N))
        hkl = np.column_stack([h_arr[sl], k_arr[sl], l_arr[sl]])
        q = hkl @ B_recip.T                                    # (n,3)
        for gi, idx in enumerate(GRP_IDX):
            if len(idx) == 0:
                continue
            ph_g = np.exp(2j*np.pi*(hkl @ GRP_N[gi]))          # (n,) group phase
            out  = np.exp(-1j*(q @ r_grid[idx].T)) @ WL_grid[idx]   # (n,4)
            F_out[sl] += ph_g*out[:, 0]
            L_out[sl] += ph_g[:, None]*out[:, 1:4]
    return F_out, L_out

def F_L_plane(h_pts, k_pts, l_val):
    """Same transform on a full (h,k) plane, by three sequential axis
    contractions instead of one dense (M x N_vox) phase matrix.

    q.r factorizes exactly on a lattice grid:
        q.r = 2 pi (h i/nu + k j/nv + l m/nw)
    so contracting one axis at a time costs O(N_vox + nH nu nv + nH nK nv)
    instead of O(nH nK N_vox) -- about three orders of magnitude for a full
    experimental plane, which is what makes a full-plane comparison affordable.
    """
    h_pts = np.asarray(h_pts, float); k_pts = np.asarray(k_pts, float)
    nH, nK = len(h_pts), len(k_pts)
    Ei = np.exp(-2j*np.pi*np.outer(h_pts, np.arange(nu))/nu)     # (nH, nu)
    Ej = np.exp(-2j*np.pi*np.outer(np.arange(nv), k_pts)/nv)     # (nv, nK)
    Ek = np.exp(-2j*np.pi*l_val*np.arange(nw)/nw)                # (nw,)
    F = np.zeros((nH, nK), complex); L = np.zeros((nH, nK, 3), complex)
    for gi, idx in enumerate(GRP_IDX):
        if len(idx) == 0:
            continue
        ph_g = np.exp(2j*np.pi*(h_pts[:, None]*GRP_N[gi, 0]
                                + k_pts[None, :]*GRP_N[gi, 1]
                                + l_val*GRP_N[gi, 2]))           # (nH, nK)
        for ch in range(4):
            w = np.zeros(N_VOX); w[idx] = WL_grid[idx, ch]
            block = ((Ei @ (w.reshape(nu, nv, nw) @ Ek)) @ Ej)   # (nH, nK)
            if ch == 0:
                F += ph_g*block
            else:
                L[:, :, ch-1] += ph_g*block
    return F, L

# --- Verification ----------------------------------------------------------
# 1. At integer hkl the group phases are all exactly 1, so the transform reduces
#    to an ordinary cell structure factor. It is computed from the CONTRAST
#    density, so |F| should approach |F_meas| at high resolution -- where the
#    envelope transform F[M] is negligible -- and fall below it at low
#    resolution, where the solvent subtraction removes real amplitude. Binning
#    the ratio by resolution is therefore a direct check on both the transform
#    and the solvent model.
_rng_v = np.random.default_rng(0)
_sel = _rng_v.choice(len(mil_u), size=min(2000, len(mil_u)), replace=False)
_Fi, _ = F_L_batch(mil_u[_sel,0], mil_u[_sel,1], mil_u[_sel,2])
_ratio = np.abs(_Fi)/Fm_u[_sel]
_dres  = 2*np.pi/np.maximum(np.linalg.norm(mil_u[_sel] @ B_recip.T, axis=1), 1e-9)
_edges = np.quantile(_dres, np.linspace(0, 1, 7))
print(f"At integer hkl, |F_contrast| / |F_meas| by resolution "
      f"({len(_sel)} reflections):")
for _b in range(len(_edges)-1):
    _m = (_dres >= _edges[_b]) & (_dres <= _edges[_b+1])
    if _m.sum() < 10:
        continue
    print(f"   {_edges[_b]:6.2f}-{_edges[_b+1]:6.2f} Å ({_m.sum():5d} refl):  "
          f"median ratio {np.median(_ratio[_m]):.3f}")
print("   (approaches 1 at high resolution; the low-resolution shortfall is the")
print("    solvent subtraction removing amplitude the bulk contributed)")

# 2. At FRACTIONAL hkl, check the group-phase construction against brute-force
#    summation over explicitly unwrapped coordinates. A transform of the
#    periodic grid WITHOUT the unwrapping phase is evaluated alongside, to show
#    the size of the effect the phase corrects.
_hf = np.array([1 + 3/11, -2 + 5/11, 0.5])
_kf = np.array([0.5, 1 - 4/11, -1 + 2/13])
_lf = np.array([-1 + 2/13, 0.25, 1 + 6/13])
_Fg, _Lg = F_L_batch(_hf, _kf, _lf)
_qf = np.column_stack([_hf, _kf, _lf]) @ B_recip.T
_ph_brute = np.exp(-1j*(_qf @ r_mol.T))                      # unwrapped positions
_F_brute = _ph_brute @ (rho_eff.ravel()*dV)
_F_naive = np.exp(-1j*(_qf @ r_grid.T)) @ (rho_eff.ravel()*dV)   # no unwrap phase
print("\nAt fractional hkl:")
print(f"   grouped vs brute-force unwrapped: max rel. error "
      f"{np.abs(_Fg-_F_brute).max()/np.abs(_F_brute).max():.3e}")
print(f"   periodic grid, no unwrap phase:   max rel. error "
      f"{np.abs(_F_naive-_F_brute).max()/np.abs(_F_brute).max():.3e}  (size of the effect)")
assert np.abs(_Fg-_F_brute).max() < 1e-8*np.abs(_F_brute).max(), \
    "group-phase unwrapping disagrees with brute force"

# 3. The plane path must agree with the scattered-point path.
_hp = np.arange(-3, 4) + 2/11
_kp = np.arange(-2, 3) + 3/11
_Fp, _Lp = F_L_plane(_hp, _kp, 4/13)
_HH, _KK = np.meshgrid(_hp, _kp, indexing='ij')
_Fb, _Lb = F_L_batch(_HH.ravel(), _KK.ravel(), np.full(_HH.size, 4/13))
print(f"   plane vs scattered path: max rel. error "
      f"{np.abs(_Fp.ravel()-_Fb).max()/np.abs(_Fb).max():.3e}")
assert np.abs(_Fp.ravel()-_Fb).max() < 1e-8*np.abs(_Fb).max()
print("OK: transform checks pass.")


def G_at_hkl_batch(h_arr, k_arr, l_arr):
    """Vectorized G(q) at arrays of fractional Miller indices. (N, 6) complex."""
    h_arr = np.asarray(h_arr, float); k_arr = np.asarray(k_arr, float)
    l_arr = np.asarray(l_arr, float)
    q  = (h_arr[:, None]*B_recip[:, 0] + k_arr[:, None]*B_recip[:, 1]
          + l_arr[:, None]*B_recip[:, 2])
    iq = 1j*q
    F, L = F_L_batch(h_arr, k_arr, l_arr)
    return np.concatenate([np.cross(iq, L), iq*F[:, None]], axis=1)

def G_at_hkl(h, k, l):
    return G_at_hkl_batch([h], [k], [l])[0]

def G_plane(h_pts, k_pts, l_val):
    """G(q) over a full (h,k) plane via the separable transform. (nH, nK, 6)."""
    h_pts = np.asarray(h_pts, float); k_pts = np.asarray(k_pts, float)
    F, L = F_L_plane(h_pts, k_pts, l_val)
    HH, KK = np.meshgrid(h_pts, k_pts, indexing='ij')
    q = (HH[..., None]*B_recip[:, 0] + KK[..., None]*B_recip[:, 1]
         + l_val*B_recip[:, 2])
    iq = 1j*q
    return np.concatenate([np.cross(iq, L), iq*F[..., None]], axis=-1)

_Gt = G_at_hkl(1 + 5/11, -2 + 3/11, 4/13)
print("G at a representative fractional hkl (complex, 6 components):")
print("  |Re G| =", np.abs(_Gt.real).round(1))
print("  |Im G| =", np.abs(_Gt.imag).round(1))
assert np.abs(_Gt.imag).max() > 1e-6*np.abs(_Gt).max(), \
    "G must be complex; a purely real G means the transform lost its phase"
print("OK: G retains a substantial imaginary part, as it must.")


## Rigid-Body Contacts

Two unit-cell images are in contact if any of their atoms come within
`CONTACT_CUTOFF`. Born–von Kármán translational symmetry means the contact
between cell 0 and cell $\mathbf n$ depends only on $\mathbf n$, and the two
directions of a contact ($\mathbf n$ and $-\mathbf n$) are the same physical
spring seen from either side — which is why 12 directed images collapse to 6
independent stiffnesses.

Note this is *not* a point-group symmetry reduction: in $P1$ there is no point
symmetry at all, so the six contacts are genuinely **distinct**, not
"symmetry-distinct". They share nothing, and each carries its own free
$6\times6$ matrix.

For each contact we record the centroid of the atom pairs actually within the
cutoff — the physical location of the contact patch — which sets both the
contact-local reference frame and the geometric prior on $K$.


In [ ]:
tree = cKDTree(all_pos)

def canonical(n):
    return min(n, tuple(-x for x in n))

def _find_interfaces(max_image):
    found = {}
    for n_tup in product(range(-max_image, max_image+1), repeat=3):
        if n_tup == (0,0,0):
            continue
        R_n = sum(n_tup[i]*[a1,a2,a3][i] for i in range(3))
        idx_lists = tree.query_ball_point(all_pos + R_n, CONTACT_CUTOFF)
        n_c = sum(len(idx) for idx in idx_lists)
        if n_c >= MIN_CONTACTS:
            found[n_tup] = {'R_n': R_n, 'n_contacts': n_c, 'idx_lists': idx_lists}
    return found

raw_interfaces = _find_interfaces(MAX_IMAGE)

# Verify MAX_IMAGE is actually large enough rather than assuming it. One of the
# detected contacts here sits at |R_n| ≈ 47 Å, so the molecule is long enough
# that second-shell images are worth ruling out explicitly.
_outer = {n: v for n, v in _find_interfaces(MAX_IMAGE+1).items()
          if max(abs(x) for x in n) > MAX_IMAGE}
if _outer:
    print(f"WARNING: {len(_outer)} contact(s) found beyond MAX_IMAGE={MAX_IMAGE}: "
          f"{sorted(_outer)} -- increase MAX_IMAGE.")
else:
    print(f"Checked shell {MAX_IMAGE+1}: no additional contacts, MAX_IMAGE={MAX_IMAGE} is sufficient.")

unique = {}
for n_tup, info in raw_interfaces.items():
    c = canonical(n_tup)
    if c not in unique:
        unique[c] = dict(info, n_tup=n_tup)

def contact_midpoints(R_n, idx_lists):
    """Midpoints (lab frame) of every atom pair within CONTACT_CUTOFF."""
    j_idx = np.concatenate([np.full(len(ii), j, int) for j, ii in enumerate(idx_lists) if len(ii)]) \
            if any(len(ii) for ii in idx_lists) else np.zeros(0, int)
    i_idx = np.concatenate([np.asarray(ii, int) for ii in idx_lists if len(ii)]) \
            if any(len(ii) for ii in idx_lists) else np.zeros(0, int)
    return 0.5*(all_pos[i_idx] + all_pos[j_idx] + R_n)

for c, info in unique.items():
    info['midpoints']  = contact_midpoints(info['R_n'], info['idx_lists'])
    info['contact_pt'] = info['midpoints'].mean(0)

shell_order = sorted(unique.keys())
print(f"\nDetected {len(unique)} distinct contacts ({len(raw_interfaces)} directed images):")
for c in shell_order:
    info = unique[c]
    d = info['midpoints'] - info['contact_pt']
    rg = np.sqrt((d**2).sum(1).mean())
    info['rho_g'] = rg          # patch gyration radius: the contact's own length scale
    print(f"  n={c}  |R_n|={np.linalg.norm(info['R_n']):6.2f} Å  "
          f"atom-pairs={info['n_contacts']:3d}  patch gyration radius={rg:5.2f} Å")


## Reduced Units

Stiffnesses are expressed directly in units of $k_BT$, so $k_BT=1$ throughout.

One consequence is worth stating because it is a free correctness test: in the
classical limit $\langle u_{\mathbf q}u_{\mathbf q}^\dagger\rangle=k_BT\,
D(\mathbf q)^{-1}$, which contains **no mass matrix at all**. $M$ therefore
affects only the plotted frequencies — never $I(\mathbf q)$, never the ADPs. At
room temperature the classical limit is amply justified: the branches sit below
$\sim0.4$ THz, i.e. $\hbar\omega\lesssim1.6$ meV against $k_BT=25.9$ meV.

This also fixes the absolute units of $I_{\rm model}$. With $k_BT=1$ and $G$ in
$e/\mathrm{Å}$, $G^\dagger D^{-1}G$ is in electrons² per unit cell — the same
scale the deposited map reports ("electron scattering per unit cell"). That
coincidence is what lets the $K$/scale degeneracy be closed without touching the
ADPs; see the refinement section.

In [ ]:
# The mass matrix was built alongside the atomic model above (it has to be, since
# r_cm_at is also G's reference point). Report it here and check the classical
# limit explicitly.
print(f"Moment of inertia J (amu·Å²), eigenvalues: {np.linalg.eigvalsh(J).round(0)}")
print(f"Total mass: {m_total:.0f} amu")
print(f"Frequency unit: 1 (k_BT/amu/Å²)^½ = {freq_unit:.3f} THz")

hbar_over_kT_THz = const.hbar*2*np.pi*1e12/(const.k*TEMPERATURE)
print(f"\nClassical-limit check: ħω/k_BT = {hbar_over_kT_THz:.4f} × (ω in THz)")
print(f"  at 0.4 THz -> {0.4*hbar_over_kT_THz:.3f};  at 2 THz -> {2*hbar_over_kT_THz:.3f}")
print("  Both « 1, so the classical (equipartition) form is appropriate.")

## Contact Frames, the Geometric Prior, and the Stiffness Parameterization

Each contact's $6\times6$ stiffness $K$ is specified in a **local contact frame**:
origin at the measured contact centroid, $z$-axis along $\mathbf R_{\mathbf n}$. A
fixed rigid shift-and-rotation, computed once from the structure and never fit,
re-expresses $K$ at the far body's own centre of mass, which is the representation
the dynamical matrix uses.

**The prior.** Model the contact as $n$ independent isotropic point springs of
stiffness $k$ at the atom-pair midpoints $\mathbf m_i$, with
$\mathbf d_i=\mathbf m_i-\bar{\mathbf m}$:

$$E=\tfrac{k}{2}\sum_i\bigl|\mathbf v+\boldsymbol\Omega\times\mathbf d_i\bigr|^2
\;\Longrightarrow\;
K_{TT}=k\,n\,I_3,\qquad
K_{RR}=k\sum_i\bigl(|\mathbf d_i|^2I-\mathbf d_i\mathbf d_i^{\mathsf T}\bigr),\qquad
K_{TR}=0$$

the last exactly, since $\sum_i\mathbf d_i=0$ at the centroid. This costs no extra
parameters and supplies four things at once:

* **scaling with the number of atom–atom contacts**, which spans 9 to 63 here;
* **the $\kappa_R/\kappa_T$ ratio**, of order $\rho_g^2$ with $\rho_g$ the patch
  gyration radius. The two are not interchangeable — one is $k_BT/\mathrm{rad}^2$,
  the other $k_BT/\mathrm{Å}^2$ — and getting the ratio wrong by $\rho_g^2$ makes
  the librational springs far too soft relative to the translational ones;
* **the patch anisotropy**: a flat contact is automatically soft about axes lying
  in its own plane;
* **a vanishing local-frame $T$–$R$ block**, rather than one imposed by hand.

Refinement is regularized toward this prior. In a sloppy model the alternative is
that the poorly-determined directions — most of the 21 per contact, including
every overall magnitude — are held by nothing and drift to whichever boundary the
optimizer reaches first.

### The regularizer metric

$K$ mixes units: in $(\boldsymbol\Omega,\mathbf v)$ the rotational block is
$k_BT/\mathrm{rad}^2$, the translational block $k_BT/\mathrm{Å}^2$, and the cross
block $k_BT/(\mathrm{rad}\cdot\mathrm{Å})$. A Frobenius norm of
$K-K^{\rm prior}$ would add squared quantities differing by powers of length, and
would weight the rotational block more heavily by $\sim\rho_g^4$.

The penalty is instead the **affine-invariant (geodesic) distance** on the cone of
positive-definite matrices,

$$d^2\bigl(K,K^{\rm prior}\bigr)
=\bigl\|\log\bigl(K_{\rm prior}^{-1/2}KK_{\rm prior}^{-1/2}\bigr)\bigr\|_F^2
=\sum_i\bigl(\log s_i\bigr)^2$$

with $s_i$ the generalized eigenvalues of the pencil $(K,K^{\rm prior})$. The
prior carries exactly the units of $K$, so the ratio is dimensionless and no
length scale has to be introduced. Three further properties:

* **Coordinate-invariant.** Under $K\to TKT^{\mathsf T}$ — which is what moving the
  reference point from the contact centroid to the far body's centre of mass, or
  rotating the local axes, amounts to — the penalty is unchanged, because the
  prior transforms identically. The penalty therefore does not depend on a
  bookkeeping choice.
* **Symmetric in stiff/soft.** A factor of two too stiff costs exactly what a
  factor of two too soft costs.
* **Barrier at the boundary.** $d\to\infty$ as $K$ approaches singular, so each
  contact stays positive definite without a bound constraint.

**Stability.** $K=LL^{\mathsf T}$ with $L$ lower-triangular and otherwise free is
positive semidefinite for any real $L$, so the optimizer runs unconstrained. That
is sufficient for the whole crystal, because each contact enters the dynamical
matrix only as

$$D_{\mathbf n}(\mathbf q)=M_{\mathbf n}(\mathbf q)^\dagger K_{\mathbf n}
M_{\mathbf n}(\mathbf q),\qquad
M_{\mathbf n}(\mathbf q)=A_{\mathbf n}-e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}I$$

which is manifestly Hermitian positive semidefinite.

In [ ]:
def skew(v):
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])

def contact_frame(R_n):
    """Orthonormal frame with z along R_n."""
    ez = R_n / np.linalg.norm(R_n)
    perp = np.array([1.,0.,0.]) if abs(ez[0]) < 0.9 else np.array([0.,1.,0.])
    ex = np.cross(perp, ez); ex /= np.linalg.norm(ex)
    return np.column_stack([ex, np.cross(ez, ex), ez])

def Gc_inv_matrix(c):
    """Inverse of the rigid-shift adjoint that moves a reference point by c."""
    G = np.eye(6); G[3:6, 0:3] = -skew(c)
    return G

def K_local_to_lab(K_local, R_frame, c_local):
    """K_local is expressed at the contact centroid, in local-frame axes.
    Returns K expressed at the far body's own centre of mass, in lab axes."""
    Gc_inv = Gc_inv_matrix(c_local)
    K_shifted = Gc_inv.T @ K_local @ Gc_inv
    Gamma = np.block([[R_frame, np.zeros((3,3))],[np.zeros((3,3)), R_frame]])
    return Gamma @ K_shifted @ Gamma.T

for c in shell_order:
    info = unique[c]
    R_frame = contact_frame(info['R_n'])
    d_c     = info['contact_pt'] - r_cm_at        # contact offset from body-0 COM
    info['R_frame'] = R_frame
    info['c_local'] = R_frame.T @ (info['R_n'] - d_c)
    info['Gamma']   = np.block([[R_frame, np.zeros((3,3))],
                                [np.zeros((3,3)), R_frame]])

TRIL_I, TRIL_J = np.tril_indices(6)
N_TRIL   = len(TRIL_I)                # 21
N_SHELL  = len(shell_order)
N_PARAMS = N_SHELL * N_TRIL

def theta_to_Lmap(theta):
    Lmap, idx = {}, 0
    for c in shell_order:
        L = np.zeros((6,6))
        L[TRIL_I, TRIL_J] = theta[idx:idx+N_TRIL]
        Lmap[c] = L
        idx += N_TRIL
    return Lmap

def Lmap_to_theta(Lmap):
    theta, idx = np.zeros(N_PARAMS), 0
    for c in shell_order:
        theta[idx:idx+N_TRIL] = Lmap[c][TRIL_I, TRIL_J]
        idx += N_TRIL
    return theta

def build_K(Lmap):
    """Cholesky factors -> (K in the local contact frame, K in the lab frame at
    each contact's far-body COM, the representation dynamical_matrix expects)."""
    K_local, K_lab = {}, {}
    for c in shell_order:
        Kl = Lmap[c] @ Lmap[c].T
        K_local[c] = Kl
        K_lab[c] = K_local_to_lab(Kl, unique[c]['R_frame'], unique[c]['c_local'])
    return K_local, K_lab

# --- Geometric prior: n isotropic point springs at the atom-pair midpoints ---
def contact_patch_stiffness(info, k_per_pair):
    d = info['midpoints'] - info['contact_pt']
    n = len(d)
    K = np.zeros((6,6))
    K[3:6, 3:6] = k_per_pair * n * np.eye(3)                        # translation
    K[0:3, 0:3] = k_per_pair * ((d**2).sum()*np.eye(3) - d.T @ d)   # rotation
    return K                                                        # T-R block = 0

def prior_Lmap(k_per_pair=PRIOR_K_PER_PAIR, ridge_frac=1e-3):
    Lmap, K_prior_local = {}, {}
    for c in shell_order:
        info = unique[c]
        Gamma = info['Gamma']
        Kl = Gamma.T @ contact_patch_stiffness(info, k_per_pair) @ Gamma
        Kl = 0.5*(Kl + Kl.T)
        Kl += ridge_frac * np.trace(Kl)/6 * np.eye(6)   # keep strictly PD
        K_prior_local[c] = Kl
        Lmap[c] = np.linalg.cholesky(Kl)
    return Lmap, K_prior_local

L_PRIOR, K_PRIOR_LOCAL = prior_Lmap()
_, K_LAB_PRIOR = build_K(L_PRIOR)

# --- Regularizer: geodesic distance to the geometric prior ------------------
# K mixes units -- K_RR is k_BT/rad^2, K_RT is k_BT/(rad*A), K_TT is k_BT/A^2 --
# so a Frobenius norm of (K - K_prior) would add squared quantities differing by
# powers of length, and would weight the rotational block more heavily by
# ~rho_g^4. The penalty is instead the affine-invariant (geodesic) distance on
# the cone of positive-definite matrices,
#
#     d^2 = || log( K_prior^{-1/2} K K_prior^{-1/2} ) ||_F^2 = sum_i (log s_i)^2
#
# with s_i the generalized eigenvalues of the pencil (K, K_prior). The prior
# carries exactly the units of K, so the ratio is dimensionless. It is also
# invariant under K -> T K T^T, hence independent of whether K is referenced at
# the contact centroid or the far body's centre of mass and of the choice of
# local axes; symmetric in stiff/soft; and divergent as K approaches singular,
# which keeps each contact positive definite without a bound constraint.

def _sqrtm_inv_sym(A):
    w, V = np.linalg.eigh(0.5*(A + A.T))
    w = np.maximum(w, REG_EIG_FLOOR*max(w.max(), 1e-300))
    return (V*w**-0.5) @ V.T

REG_PISQRT = {c: _sqrtm_inv_sym(K_PRIOR_LOCAL[c]) for c in shell_order}

def _affine_core(K, Minv):
    """Eigen-data of P = K_prior^{-1/2} K K_prior^{-1/2}, the dimensionless
    relative stiffness."""
    P = Minv @ K @ Minv; P = 0.5*(P + P.T)
    w, U = np.linalg.eigh(P)
    w = np.maximum(w, REG_EIG_FLOOR*max(w.max(), 1e-300))
    return w, U, np.log(w)

# lam defaults to None and is looked up at CALL time, not at def time. A default
# of `lam=LAMBDA_PRIOR` would bind the value once, when the cell is executed, so
# the scan further down -- which rebinds the module-level LAMBDA_PRIOR in a loop
# -- would silently run every value at whatever lambda was set here.
def regularizer_prior(K_local, lam=None):
    """lam * sum_n d^2(K_n, K_n^prior) over the contacts present in K_local."""
    lam = LAMBDA_PRIOR if lam is None else lam
    return lam*sum(np.sum(_affine_core(K_local[c], REG_PISQRT[c])[2]**2)
                   for c in K_local)

def regularizer_prior_dK(K_local, lam=None):
    """dR/dK per contact (symmetric): 2*lam * Minv (P^{-1} log P) Minv."""
    lam = LAMBDA_PRIOR if lam is None else lam
    out = {}
    for c in K_local:
        Minv = REG_PISQRT[c]
        w, U, lg = _affine_core(K_local[c], Minv)
        out[c] = 2*lam*(Minv @ ((U*(lg/w)) @ U.T) @ Minv)
    return out

# The penalty should cost the same for equal RELATIVE perturbations of the
# rotational and translational blocks. A ratio far from 1 would mean one block
# is effectively unregularized.
_c0 = shell_order[0]; _Kp0 = K_PRIOR_LOCAL[_c0]
_pR = _Kp0.copy(); _pR[0:3,0:3] *= 1.10
_pT = _Kp0.copy(); _pT[3:6,3:6] *= 1.10
_aR = regularizer_prior({_c0: _pR}, lam=1.0)
_aT = regularizer_prior({_c0: _pT}, lam=1.0)
print(f"\nGeodesic regularizer, penalty for a 10% relative change in each block")
print(f"of contact {_c0} (rho_g = {unique[_c0]['rho_g']:.2f} Å):")
print(f"   rotational {_aR:.4e}   translational {_aT:.4e}   ratio {_aR/_aT:.3f}")

print(f"Free parameters: {N_SHELL} contacts × {N_TRIL} Cholesky entries = {N_PARAMS}\n")
print(f"Geometric prior at k_per_pair = {PRIOR_K_PER_PAIR} k_BT/Å² "
      f"(= {PRIOR_K_PER_PAIR*0.414:.2f} N/m per atom pair):")
for c in shell_order:
    Kl = K_PRIOR_LOCAL[c]
    kt, kr = np.trace(Kl[3:6,3:6])/3, np.trace(Kl[0:3,0:3])/3
    print(f"  {str(c):14s} κ_T={kt:8.2f} k_BT/Å²   κ_R={kr:9.1f} k_BT/rad²   "
          f"ratio={kr/kt:6.1f} Å²   |T-R|={np.abs(Kl[0:3,3:6]).max():.2e}")
print("\n(The ratio column is the point: a flat isotropic guess sets it to 1.)")


## Dynamical Matrix

For contact $\mathbf n$ with stiffness $K_{\mathbf n}$ (expressed at the far
body's own centre of mass) and adjoint
$A_{\mathbf n}=\left(\begin{smallmatrix}I&0\\-[\mathbf R_{\mathbf n}]_\times&I
\end{smallmatrix}\right)$,

$$D_{\mathbf n}(\mathbf q)=M_{\mathbf n}(\mathbf q)^\dagger K_{\mathbf n}
M_{\mathbf n}(\mathbf q),\qquad
M_{\mathbf n}(\mathbf q)=A_{\mathbf n}-e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}I$$

and $D=\sum_{\mathbf n}D_{\mathbf n}$ over the 6 independent contacts, both
directions of each already included. Assembling $D$ in this factored form keeps it
exactly Hermitian positive semidefinite in floating point, so cancellation cannot
push an acoustic eigenvalue negative.

$D(\mathbf q)$ is complex Hermitian, not real symmetric:
$\mathrm{Im}\,D_{\mathbf n}=\sin(\mathbf q\cdot\mathbf R_{\mathbf n})
\bigl(K_{\mathbf n}A_{\mathbf n}-(K_{\mathbf n}A_{\mathbf n})^{\mathsf T}\bigr)$.

**Zero modes at $\Gamma$.** $D(\mathbf q)u=0$ iff $M_{\mathbf n}(\mathbf q)u=0$ for
every $\mathbf n$, given $K_{\mathbf n}\succ0$. At $\mathbf q=0$ that reads
$\mathbf R_{\mathbf n}\times\boldsymbol\Omega=0$ for all $\mathbf n$, which for
three independent $\mathbf R_{\mathbf n}$ forces $\boldsymbol\Omega=0$ with
$\mathbf v$ free. Exactly 3 eigenvalues vanish at $\Gamma$ — the pure translations
— and the 3 librational eigenvalues sit at whatever rest frequency the rotational
stiffness sets. A Born–von Kármán lattice with fixed $\mathbf R_{\mathbf n}$ is not
rotationally invariant, so a global rotation of the crystal is not a zero mode of
this model. That is a limitation of the model worth naming.

In [ ]:
def ad_translation(R_n):
    A = np.eye(6); A[3:6, 0:3] = -skew(R_n); return A

_I6 = np.eye(6)

def dynamical_matrix_batch(q_batch, unique, K_lab):
    """D(q) = Σ_n M_n(q)^† K_n M_n(q) over a batch of Cartesian q. (N,6,6) complex."""
    q_batch = np.atleast_2d(np.asarray(q_batch, float))
    D = np.zeros((len(q_batch), 6, 6), dtype=complex)
    for c, info in unique.items():
        R_n = info['R_n']; K = K_lab[c]
        A_n = ad_translation(R_n)
        phase = np.exp(1j*(q_batch @ R_n))                        # (N,)
        M = A_n[None,:,:] - phase[:,None,None]*_I6[None,:,:]      # (N,6,6)
        D += np.einsum('nji,jk,nkl->nil', M.conj(), K, M)
    return D

def dynamical_matrix(q_cart, unique, K_lab):
    return dynamical_matrix_batch(np.asarray(q_cart, float)[None,:], unique, K_lab)[0]

def mass_weighted_eigs(D_batch):
    """Eigenvalues of the pencil (D, M), via the Cholesky congruence. Complex
    Hermitian throughout -- taking .real here would be a different matrix."""
    Dw = np.einsum('ij,njk,lk->nil', Msq_inv, D_batch, Msq_inv)
    return np.linalg.eigvalsh(Dw)

# --- Sanity check at Gamma -------------------------------------------------
D0  = dynamical_matrix(np.zeros(3), unique, K_LAB_PRIOR)
ev0 = mass_weighted_eigs(D0[None])[0]
print("D(q=0) mass-weighted eigenvalues:", ev0.round(8))
print("  -> exactly 3 acoustic zeros expected; the upper 3 are the librational")
print("     rest frequencies and are NOT required to vanish (see markdown).")
assert np.sum(np.abs(ev0) < 1e-8*max(abs(ev0).max(), 1e-30)) == 3, \
    "expected exactly 3 zero modes at Gamma"

# --- Hermiticity / PSD check at a generic q --------------------------------
_qc = B_recip @ np.array([0.31, -0.17, 0.43])
_Dc = dynamical_matrix(_qc, unique, K_LAB_PRIOR)
print(f"\nAt a generic q: ||Im D|| / ||Re D|| = "
      f"{np.linalg.norm(_Dc.imag)/np.linalg.norm(_Dc.real):.4f}  (must be > 0)")
print(f"  Hermitian: {np.allclose(_Dc, _Dc.conj().T)}   "
      f"min eigenvalue: {np.linalg.eigvalsh(_Dc).min():.4e} (must be >= 0)")
print(f"  lowest eigenvalues: {np.linalg.eigvalsh(_Dc)[:3].round(4)}")


## Phonon Band Structure

Eigenvalues of the mass-weighted dynamical matrix along a path through the
Brillouin zone, using the geometric prior as the starting model. The
eigenproblems are batched — a few hundred complex-Hermitian $6\times6$ solves per
call — because this function is called once per frame of the convergence movie
later.

The printed band range is a reference point for the fit: a refined model in which
one branch collapses toward zero while others rise by an order of magnitude
indicates the optimizer has reached a boundary rather than an interior optimum.

In [ ]:
HSP = {'Γ':np.array([0.,0.,0.]),'X':np.array([.5,0.,0.]),
       'Y':np.array([0.,.5,0.]),'Z':np.array([0.,0.,.5])}
path_labels = ['Γ','X','Γ','Y','Γ','Z','Γ']
N_seg = 60

q_frac, tick_idx = [], [0]
for seg in range(len(path_labels)-1):
    p0, p1 = HSP[path_labels[seg]], HSP[path_labels[seg+1]]
    last = (seg == len(path_labels)-2)
    for t in np.linspace(0, 1, N_seg, endpoint=last):
        q_frac.append(p0*(1-t)+p1*t)
    if not last: tick_idx.append(len(q_frac))
tick_idx.append(len(q_frac)-1)

q_frac = np.array(q_frac)
q_cart = (B_recip @ q_frac.T).T
x = np.concatenate([[0], np.cumsum(np.linalg.norm(np.diff(q_cart,axis=0),axis=1))])

def bandstructure_freqs(K_lab, qs=None):
    """Batched and fully complex. ~50x faster than the per-q Python loop."""
    qs = q_cart if qs is None else qs
    ev = mass_weighted_eigs(dynamical_matrix_batch(qs, unique, K_lab))
    return np.sqrt(np.maximum(ev, 0)) * freq_unit

freqs_bs = bandstructure_freqs(K_LAB_PRIOR)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(x, freqs_bs, color='steelblue', lw=1.2)
for ti in tick_idx: ax.axvline(x[ti], color='k', lw=0.7)
ax.set_xticks([x[ti] for ti in tick_idx]); ax.set_xticklabels(path_labels)
ax.set_ylabel('Frequency (THz)'); ax.set_xlim(x[0],x[-1]); ax.set_ylim(0)
ax.set_title('Rigid-Body Phonon Band Structure — 6o2h (P1), geometric prior')
plt.tight_layout(); plt.savefig('band_structure_prior.png', dpi=150); plt.show()

print(f"Librational rest frequencies at Γ: {freqs_bs[0][3:].round(4)} THz")
print(f"Band range across the path: {freqs_bs[freqs_bs>0].min():.4f} – {freqs_bs.max():.4f} THz")
print("A physically sensible fit should keep this spread modest. A refined model")
print("in which one branch collapses toward zero while others rise by an order of")
print("magnitude is a sign the optimizer has found a boundary, not an optimum.")

## Reading the Experimental Map

Two jobs in one streaming pass: a **diagnostic** sweep — mean intensity, mean
$\sigma$, median $I/\sigma$, finite-voxel coverage and negative fraction against
resolution — and the **collection** of candidate fitting voxels. One $h$-index
slice at a time, so peak memory stays at one $(n_k\times n_l)$ plane regardless of
map size, and the ~1 GB of `I` and `sigma` is read once rather than twice.

**Resolution range.** `D_FIT_MIN` and `D_FIT_MAX` are set deliberately in the
parameters cell rather than chosen by an SNR heuristic, because
$\overline I/\overline\sigma$ per shell is not an SNR: diffuse intensity rises
steeply with $|q|$ (roughly as $q^2|F|^2$), so the ratio climbs with resolution
mostly because the *numerator* grows, and any rule based on it selects the finest
shell available. That is the wrong end of the range — at ~2 Å the diffuse
scattering from a protein crystal is dominated by internal and side-chain motion
rather than rigid-body lattice dynamics, the one-phonon approximation is
degrading, and Debye–Waller suppression is strongest. `D_FIT_MIN` is therefore a
model-validity statement. The median of $I/\sigma$ is collected as the honest
per-voxel statistic, and $CC$ is reported per resolution shell later so the
resolution at which the model stops working is measured rather than assumed.

**Bragg exclusion.** A voxel counts as Bragg-adjacent only when it is close to a
reciprocal-lattice point in all three indices, tested on the Cartesian distance
`BRAGG_EXCL_CART`. An index-unit threshold would interact badly with the map's
$(1/11,1/11,1/13)$ sampling.

In [ ]:
def _bragg_offsets(h_frac, k_frac_all, l_frac_all):
    """Signed offsets to the nearest reciprocal-lattice point, and the Cartesian
    distance to it. Purely geometric -- it knows nothing about I_obs or its sign,
    which is what makes it a legitimate basis for stratifying or weighting.
    Selecting on the sign of the quantity being fit would not be."""
    dh = h_frac - round(h_frac)
    dk = k_frac_all - np.round(k_frac_all)
    dl = l_frac_all - np.round(l_frac_all)
    dq = (dh*B_recip[:,0][None,None,:] + dk[:,:,None]*B_recip[:,1][None,None,:]
          + dl[:,:,None]*B_recip[:,2][None,None,:])
    return dh, dk, dl, np.linalg.norm(dq, axis=-1)

D_SCAN_MAX = 12.0
N_RES_BINS = 40

def scan_and_collect(d_lo, d_hi, n_res_bins=N_RES_BINS):
    """One streaming pass: resolution diagnostics AND candidate-voxel collection.

    Collects everything in the [d_lo, d_hi] shell that is not Bragg-adjacent --
    both the near-Bragg voxels that feed the halo profile and the mid-zone
    voxels that are stratified and subsampled below.
    """
    inv_d_edges = np.linspace(0.0, 2*np.pi/1.0, n_res_bins+1)
    inv_d_mid   = 0.5*(inv_d_edges[:-1] + inv_d_edges[1:])
    sum_I   = np.zeros(n_res_bins); sum_sig = np.zeros(n_res_bins)
    cnt_fin = np.zeros(n_res_bins, np.int64); cnt_tot = np.zeros(n_res_bins, np.int64)
    cnt_neg = np.zeros(n_res_bins, np.int64)
    snr_samples = [[] for _ in range(n_res_bins)]

    q_lo, q_hi = 2*np.pi/d_hi, 2*np.pi/d_lo
    hs, ks, ls, Is, Ss, Bds = [], [], [], [], [], []
    rng_thin = np.random.default_rng(0)

    nH, nK_, nL_ = grid_size
    with h5py.File(H5_FILE, 'r') as f:
        Ids, Sds = f[f'{MAP_GROUP}/I'], f[f'{MAP_GROUP}/sigma']
        jj_idx, kk_idx = np.meshgrid(np.arange(nK_), np.arange(nL_), indexing='ij')
        k_frac_all = grid_ori[1] + jj_idx*grid_delta[1]
        l_frac_all = grid_ori[2] + kk_idx*grid_delta[2]
        for i in range(nH):
            h_frac = grid_ori[0] + i*grid_delta[0]
            q  = (h_frac*B_recip[:,0][None,None,:]
                  + k_frac_all[:,:,None]*B_recip[:,1][None,None,:]
                  + l_frac_all[:,:,None]*B_recip[:,2][None,None,:])
            qn = np.linalg.norm(q, axis=-1)
            I_slice = Ids[i,:,:]; S_slice = Sds[i,:,:]
            finite  = np.isfinite(I_slice) & np.isfinite(S_slice) & (S_slice > 0)

            bin_idx = np.clip(np.digitize(qn, inv_d_edges)-1, 0, n_res_bins-1)
            np.add.at(cnt_tot, bin_idx.ravel(), 1)
            fb = bin_idx[finite]
            np.add.at(cnt_fin, fb, 1)
            np.add.at(sum_I,   fb, I_slice[finite].astype(np.float64))
            np.add.at(sum_sig, fb, S_slice[finite].astype(np.float64))
            np.add.at(cnt_neg, bin_idx[finite & (I_slice < 0)], 1)
            if finite.any():
                r = I_slice[finite]/S_slice[finite]
                thin = rng_thin.random(len(r)) < 0.01
                for b_, v_ in zip(fb[thin], r[thin]):
                    snr_samples[b_].append(float(v_))

            dh, dk, dl, bragg_d = _bragg_offsets(h_frac, k_frac_all, l_frac_all)
            ok = finite & (qn >= q_lo) & (qn <= q_hi) & (bragg_d >= BRAGG_EXCL_CART)
            if not ok.any():
                continue
            sel = np.where(ok)
            hs.append(np.full(len(sel[0]), h_frac))
            ks.append(k_frac_all[sel]); ls.append(l_frac_all[sel])
            Is.append(I_slice[sel].astype(np.float64))
            Ss.append(S_slice[sel].astype(np.float64))
            Bds.append(bragg_d[sel])

    with np.errstate(invalid='ignore', divide='ignore'):
        mean_I     = np.where(cnt_fin > 0, sum_I/np.maximum(cnt_fin,1), np.nan)
        mean_sigma = np.where(cnt_fin > 0, sum_sig/np.maximum(cnt_fin,1), np.nan)
        snr_proxy  = mean_I/mean_sigma
        neg_frac   = np.where(cnt_fin > 0, cnt_neg/np.maximum(cnt_fin,1), np.nan)
    med_snr = np.array([np.median(s) if len(s) > 20 else np.nan for s in snr_samples])
    d_mid   = np.where(inv_d_mid > 1e-6, 2*np.pi/np.maximum(inv_d_mid,1e-6), np.inf)

    hkl = (np.column_stack([np.concatenate(hs), np.concatenate(ks), np.concatenate(ls)])
           if hs else np.zeros((0,3)))
    return (dict(d_mid=d_mid, snr_proxy=snr_proxy, med_snr=med_snr,
                 finite_frac=cnt_fin/np.maximum(cnt_tot,1), neg_frac=neg_frac,
                 cnt_finite=cnt_fin),
            hkl,
            np.concatenate(Is) if Is else np.zeros(0),
            np.concatenate(Ss) if Ss else np.zeros(0),
            np.concatenate(Bds) if Bds else np.zeros(0))

_diag, hkl_pool, I_pool, sigma_pool, bragg_dist_pool = scan_and_collect(D_FIT_MIN, D_FIT_MAX)
d_mid        = _diag['d_mid']
snr_proxy    = _diag['snr_proxy']
med_snr      = _diag['med_snr']
finite_frac  = _diag['finite_frac']
neg_frac_res = _diag['neg_frac']
cnt_finite   = _diag['cnt_finite']

print(f"{len(hkl_pool)} candidate voxels in {D_FIT_MIN}-{D_FIT_MAX} Å "
      f"(one streaming pass; I and sigma each read once)")
print(f"Negative fraction in the candidate pool: {(I_pool < 0).mean():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
finite_d = np.isfinite(d_mid) & (d_mid <= D_SCAN_MAX) & (cnt_finite > 0)

axes[0].plot(d_mid[finite_d], snr_proxy[finite_d], 'o-', ms=3,
             label=r'$\overline{I}/\overline{\sigma}$ (misleading)')
axes[0].plot(d_mid[finite_d], med_snr[finite_d], 's-', ms=3, color='darkgreen',
             label=r'median $I/\sigma$ (honest)')
axes[0].axhline(1.0, color='r', ls='--', lw=1)
axes[0].axvspan(D_FIT_MIN, D_FIT_MAX, color='steelblue', alpha=0.2, label='fitting range')
axes[0].set_xlabel('resolution $d$ (Å)'); axes[0].set_ylabel('signal-to-noise')
axes[0].invert_xaxis(); axes[0].legend(fontsize=8)
axes[0].set_title('Signal-to-noise vs. resolution')

axes[1].plot(d_mid[finite_d], finite_frac[finite_d], 'o-', ms=3, color='darkorange')
axes[1].axvspan(D_FIT_MIN, D_FIT_MAX, color='steelblue', alpha=0.2)
axes[1].set_xlabel('resolution $d$ (Å)'); axes[1].set_ylabel('finite-voxel fraction')
axes[1].invert_xaxis(); axes[1].set_title('Data coverage vs. resolution')

axes[2].plot(d_mid[finite_d], neg_frac_res[finite_d], 'o-', ms=3, color='crimson')
axes[2].axhline(0.5, color='k', ls=':', lw=1)
axes[2].axvspan(D_FIT_MIN, D_FIT_MAX, color='steelblue', alpha=0.2)
axes[2].set_xlabel('resolution $d$ (Å)'); axes[2].set_ylabel(r'fraction with $I<0$')
axes[2].invert_xaxis(); axes[2].set_title('Negative-intensity fraction vs. resolution')

plt.tight_layout(); plt.savefig('resolution_snr_scan.png', dpi=150); plt.show()

print(f"Fitting range (set by model validity, not by SNR): {D_FIT_MIN}-{D_FIT_MAX} Å")
_in = finite_d & (d_mid >= D_FIT_MIN) & (d_mid <= D_FIT_MAX)
if _in.any():
    print(f"  median I/σ there:        {np.nanmin(med_snr[_in]):.2f} – {np.nanmax(med_snr[_in]):.2f}")
    print(f"  negative fraction there: {np.nanmin(neg_frac_res[_in]):.1%} – "
          f"{np.nanmax(neg_frac_res[_in]):.1%}")
print("\nThe two SNR curves diverge toward high resolution: mean(I)/mean(sigma)")
print("keeps climbing because the numerator grows, while the per-voxel median")
print("does not. That gap is why the fitting range is set by model validity.")

## Negative Intensities

A substantial fraction of voxels carry $I_{\rm obs}<0$: Compton and
detector-geometry corrections are subtracted during processing, and noise pushes
weak pixels below zero. Three points govern how they are handled here.

1. **Least squares against negative data is not ill-behaved.** If
   $I_{\rm obs}=I_{\rm true}+\varepsilon$ with $\mathbb E[\varepsilon]=0$, then
   minimizing $\sum w(I_{\rm model}-I_{\rm obs})^2$ is a consistent estimator of
   $I_{\rm true}$ even where most observations are negative. Individual residuals
   are large but unbiased, and $1/\sigma^2$ down-weights them correctly.

2. **Filtering on the sign of $I_{\rm obs}$ would bias the fit.** Keeping only
   voxels where noise pushed $I_{\rm obs}$ above zero, while discarding the
   equally informative ones it pushed below, inflates the retained sample's mean
   above the truth. It would also improve $R$ and $R_{\rm free}$ mechanically,
   since the points guaranteed to contribute large residuals are exactly the ones
   removed. Negative points are kept throughout.

3. **Negatives mark low SNR, and that is the real hazard.** A high negative
   fraction means SNR $\lesssim1$. Such voxels carry weight but no information,
   and because $I_{\rm model}=G^\dagger D^{-1}G\ge0$ always, the least-squares
   optimum for a region whose weighted mean is $\approx0$ is $I_{\rm model}\to0$,
   which the model reaches by driving $K$ stiff. The signature is $R\to1$
   *together with* $CC\to0$ — a model predicting nothing, which $R$ alone cannot
   distinguish from a merely poor fit. Both are reported throughout.

The response is to fit where there is signal — the halo profile plus a stratified
mid-zone sample — and to report the effective sample size so a nominal point count
cannot disguise a fit resting on few observations.

### Halo Profile and Stratified Mid-Zone Sampling

Two complementary sets of observations, for two parts of the model.

**Halo profile.** $I_{\rm obs}$ averaged over (radial shell × angular sector) bins
of the offset from the nearest reciprocal-lattice point, pooled across all such
points in range. Averaging beats the noise down by $\sim\sqrt N$, turning
$\sim10^6$ noisy voxels into a few dozen well-determined numbers, and the
$I\sim|\mathbf q-\mathbf G|^{-2}$ shape with its anisotropic prefactor is a direct
readout of the long-wavelength limit of $D(\mathbf q)$ — the acoustic and elastic
information this data can genuinely determine. Sectoring rather than pure radial
averaging preserves the anisotropy, which is where the information about *which*
contacts are stiff lives. The shell averages are positive wherever there is real
signal, so the negative-intensity issue does not arise for this term.

**Stratified mid-zone sample.** Equal-count quotas over a (resolution ×
Bragg-distance) grid of strata, then pure inverse-variance weights.

Expressing the halo preference through *sampling* rather than through a
multiplicative weight matters because the number of voxels at Bragg distance $d$
grows as $d^2$: a uniformly-drawn pool is dominated by large $d$, and a
short-length-scale weight would discard most of it rather than reweighting it,
collapsing the effective sample size well below what 126 parameters require.

The effective sample size $(\sum w)^2/\sum w^2$ is printed next to every nominal
count from here on.

In [ ]:
def effective_sample_size(w):
    """ESS = (Σw)²/Σw². A nominal N means nothing once weights are heterogeneous."""
    w = np.asarray(w, float)
    return w.sum()**2/np.sum(w**2)

def make_weights(sigma):
    """Inverse-variance weights with a percentile floor on sigma, so a handful of
    anomalously small error bars cannot dominate the whole objective."""
    s = np.asarray(sigma, float)
    floor = np.percentile(s[np.isfinite(s) & (s > 0)], SIGMA_FLOOR_PCT)
    return 1.0/np.maximum(s, floor)**2

# --- Diagnostic: is the negative-intensity pattern really halo-related? ----
nbins = 12
bd_edges = np.linspace(0, np.percentile(bragg_dist_pool, 99), nbins+1)
bd_cen   = 0.5*(bd_edges[:-1]+bd_edges[1:])
bd_idx   = np.clip(np.digitize(bragg_dist_pool, bd_edges)-1, 0, nbins-1)
neg_frac_bd = np.array([(I_pool[bd_idx==b] < 0).mean() if (bd_idx==b).any() else np.nan
                        for b in range(nbins)])
mean_I_bd   = np.array([I_pool[bd_idx==b].mean() if (bd_idx==b).any() else np.nan
                        for b in range(nbins)])
cnt_bd      = np.array([(bd_idx==b).sum() for b in range(nbins)])

fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].plot(bd_cen, neg_frac_bd, 'o-', color='crimson')
axes[0].set_xlabel(r'distance to nearest Bragg point (Å$^{-1}$)')
axes[0].set_ylabel('fraction with $I_{obs}<0$'); axes[0].set_title('Negative fraction')
axes[1].plot(bd_cen, mean_I_bd, 'o-', color='steelblue')
axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel(r'distance to nearest Bragg point (Å$^{-1}$)')
axes[1].set_ylabel(r'mean $I_{obs}$'); axes[1].set_title('Mean intensity')
axes[2].semilogy(bd_cen, np.maximum(cnt_bd,1), 'o-', color='darkgreen')
axes[2].set_xlabel(r'distance to nearest Bragg point (Å$^{-1}$)')
axes[2].set_ylabel('voxel count'); axes[2].set_title(r'Population grows as $d^2$')
plt.tight_layout(); plt.savefig('bragg_distance_diagnostic.png', dpi=150); plt.show()

print("The right-hand panel is why a multiplicative halo weight fails: the pool is")
print("dominated by large Bragg distance, so a short-length-scale Gaussian weight")
print("discards most of the sample rather than reweighting it.")

# --- Halo profile ---------------------------------------------------------
def build_halo_profile(hkl, I, sigma, bragg_d, r_max=HALO_R_MAX,
                       n_shell=N_HALO_SHELLS, n_dir=N_HALO_DIRS, min_count=30):
    """Weighted mean I in (radial shell × angular sector) bins of the offset from
    the nearest reciprocal-lattice point, pooled over all such points.

    Returns representative (h,k,l) -- the weighted centroid of each bin -- so the
    profile feeds the same forward model as the mid-zone points, together with
    the bin mean and its standard error.
    """
    if len(hkl) == 0:
        return np.zeros((0,3)), np.zeros(0), np.zeros(0), np.zeros(0, int)
    q  = hkl @ B_recip.T
    dq = q - (np.round(hkl) @ B_recip.T)
    sel = (bragg_d > BRAGG_EXCL_CART) & (bragg_d < r_max)
    if sel.sum() == 0:
        return np.zeros((0,3)), np.zeros(0), np.zeros(0), np.zeros(0, int)
    hkl_s, I_s, sig_s, bd_s, dq_s = hkl[sel], I[sel], sigma[sel], bragg_d[sel], dq[sel]
    u = dq_s/np.linalg.norm(dq_s, axis=1, keepdims=True)
    theta = np.arccos(np.clip(u[:,2], -1, 1)); phi = np.arctan2(u[:,1], u[:,0])
    n_th = max(1, int(round(np.sqrt(n_dir)))); n_ph = max(1, n_dir//n_th)
    a_idx = (np.clip((theta/np.pi*n_th).astype(int), 0, n_th-1)*n_ph
             + np.clip(((phi+np.pi)/(2*np.pi)*n_ph).astype(int), 0, n_ph-1))
    r_edges = np.linspace(BRAGG_EXCL_CART, r_max, n_shell+1)
    r_idx = np.clip(np.digitize(bd_s, r_edges)-1, 0, n_shell-1)
    key = r_idx*(n_th*n_ph) + a_idx
    w = make_weights(sig_s)
    pts, mean, err, cnt = [], [], [], []
    for k_ in np.unique(key):
        m = key == k_
        if m.sum() < min_count:
            continue
        W = w[m].sum()
        pts.append(np.average(hkl_s[m], axis=0, weights=w[m]))
        mean.append(float(np.sum(w[m]*I_s[m])/W))
        err.append(float(1.0/np.sqrt(W)))
        cnt.append(int(m.sum()))
    if not pts:
        return np.zeros((0,3)), np.zeros(0), np.zeros(0), np.zeros(0, int)
    return np.array(pts), np.array(mean), np.array(err), np.array(cnt)

hkl_halo, I_halo, sig_halo, cnt_halo = build_halo_profile(
    hkl_pool, I_pool, sigma_pool, bragg_dist_pool)
print(f"\nHalo profile: {len(hkl_halo)} (shell × sector) bins from {cnt_halo.sum()} voxels")
if len(hkl_halo):
    print(f"  {cnt_halo.min()}-{cnt_halo.max()} voxels per bin;  "
          f"all bin means positive: {bool((I_halo > 0).all())}")
    print("  (shell averaging removes the negative-intensity problem for this term)")

# --- Stratified mid-zone sample -------------------------------------------
rng = np.random.default_rng(7)
q_norm_pool = np.linalg.norm(hkl_pool @ B_recip.T, axis=1)
idx_all = np.where(bragg_dist_pool >= HALO_R_MAX)[0]

q_bins = np.quantile(q_norm_pool[idx_all], np.linspace(0,1,N_RES_STRATA+1))
b_bins = np.quantile(bragg_dist_pool[idx_all], np.linspace(0,1,N_HALO_STRATA+1))
qi = np.clip(np.digitize(q_norm_pool[idx_all], q_bins)-1, 0, N_RES_STRATA-1)
bi = np.clip(np.digitize(bragg_dist_pool[idx_all], b_bins)-1, 0, N_HALO_STRATA-1)
strata_key = qi*N_HALO_STRATA + bi
per = max(1, N_FIT_MAX//(N_RES_STRATA*N_HALO_STRATA))
picked = [rng.choice(idx_all[strata_key==s_], size=min(per, int((strata_key==s_).sum())),
                     replace=False)
          for s_ in range(N_RES_STRATA*N_HALO_STRATA) if (strata_key==s_).any()]
sub_idx = np.concatenate(picked)

hkl_mid, I_mid, sigma_mid = hkl_pool[sub_idx], I_pool[sub_idx], sigma_pool[sub_idx]
train_mask = rng.random(len(sub_idx)) > HOLDOUT_FRAC

q_train, I_train, sigma_train = hkl_mid[train_mask], I_mid[train_mask], sigma_mid[train_mask]
q_test,  I_test,  sigma_test  = hkl_mid[~train_mask], I_mid[~train_mask], sigma_mid[~train_mask]
w_train_mid = make_weights(sigma_train)

print(f"\nMid-zone sample: {len(sub_idx)} points over {N_RES_STRATA}×{N_HALO_STRATA} strata")
print(f"  training {train_mask.sum()} (effective {effective_sample_size(w_train_mid):.0f}), "
      f"held out {(~train_mask).sum()}")
print(f"  negative fraction in training: {(I_train < 0).mean():.1%} "
      f"(kept -- filtering on sign would bias the fit)")

In [ ]:
def well_conditioned_batch(K_lab, hkl, min_eig_frac=1e-4):
    """D(q) is exactly singular only at reciprocal-lattice points, but a point can
    still land near a degenerate direction, where the plain linear solve the
    objective uses is ill-conditioned.

    Evaluated at the PRIOR K -- i.e. at the actual operating point the refinement
    starts from -- rather than at an unrelated placeholder, and vectorized.
    """
    if len(hkl) == 0:
        return np.zeros(0, bool)
    q = (hkl[:,0:1]*B_recip[:,0] + hkl[:,1:2]*B_recip[:,1] + hkl[:,2:3]*B_recip[:,2])
    ev = np.linalg.eigvalsh(dynamical_matrix_batch(q, unique, K_lab))
    emax, emin = ev[:,-1], ev[:,0]
    return (emax > 0) & (emin/np.where(emax > 0, emax, 1.0) > min_eig_frac)

keep_tr = well_conditioned_batch(K_LAB_PRIOR, q_train)
keep_te = well_conditioned_batch(K_LAB_PRIOR, q_test)
keep_ha = well_conditioned_batch(K_LAB_PRIOR, hkl_halo)
print(f"conditioning screen: training {keep_tr.sum()}/{len(keep_tr)}, "
      f"held out {keep_te.sum()}/{len(keep_te)}, halo {keep_ha.sum()}/{len(keep_ha)}")

q_train, I_train, sigma_train = q_train[keep_tr], I_train[keep_tr], sigma_train[keep_tr]
q_test,  I_test,  sigma_test  = q_test[keep_te],  I_test[keep_te],  sigma_test[keep_te]
hkl_halo, I_halo, sig_halo, cnt_halo = (hkl_halo[keep_ha], I_halo[keep_ha],
                                        sig_halo[keep_ha], cnt_halo[keep_ha])
w_train_mid = make_weights(sigma_train)

### Assembling the Objective

The halo bins and the mid-zone points enter the same weighted least-squares sum.
Each halo bin carries the standard error of its own average, so its natural weight
is $1/\mathrm{SE}^2$ — large, since each bin condenses thousands of voxels. The
halo block is then renormalized so the two terms carry comparable total weight
before `HALO_WEIGHT` scales it, otherwise the balance would depend on how many
bins the profile happened to produce. Set `HALO_WEIGHT` to 0 to fit the mid-zone
alone, or raise it to lean harder on the acoustic limit.

In [ ]:
if len(hkl_halo):
    w_halo = HALO_WEIGHT/np.maximum(sig_halo, 1e-12)**2
    w_halo = w_halo*(w_train_mid.sum()/max(w_halo.sum(), 1e-30))
    q_fit = np.vstack([q_train, hkl_halo])
    I_fit = np.concatenate([I_train, I_halo])
    w_fit = np.concatenate([w_train_mid, w_halo])
    is_halo = np.concatenate([np.zeros(len(q_train), bool), np.ones(len(hkl_halo), bool)])
else:
    w_halo = np.zeros(0)
    q_fit, I_fit, w_fit = q_train, I_train, w_train_mid
    is_halo = np.zeros(len(q_train), bool)

ess = effective_sample_size(w_fit)
print(f"Objective: {len(q_fit)} observations "
      f"({len(q_train)} mid-zone + {len(hkl_halo)} halo bins)")
print(f"  effective sample size: {ess:.0f}")
print(f"  weight share: mid-zone {100*w_train_mid.sum()/w_fit.sum():.0f}%, "
      f"halo {100*w_halo.sum()/w_fit.sum() if len(w_halo) else 0:.0f}%")
print(f"  free parameters: {N_PARAMS} (the shape normalization removes 1 of them)")
if ess < 5*N_PARAMS:
    print("\n*** WARNING: fewer than 5 effective observations per parameter. Expect")
    print("    the prior to dominate; read the sloppiness spectrum before quoting K. ***")

$G(\mathbf q)$ depends only on the structure and $\mathbf q$, never on the contact
stiffnesses being fit, so it is computed once for the fitting and held-out sets
rather than recomputed at every refinement iteration.

In [ ]:
G_fit  = G_at_hkl_batch(q_fit[:,0],  q_fit[:,1],  q_fit[:,2])
G_test = (G_at_hkl_batch(q_test[:,0], q_test[:,1], q_test[:,2])
          if len(q_test) else np.zeros((0,6), complex))
print(f"Cached G(q) for {len(G_fit)} fitting and {len(G_test)} held-out points")

### Two Intensity Code Paths

The refinement objective evaluates $I=G^\dagger D^{-1}G$ with a batched linear
solve; the maps and the Brillouin-zone integral use an eigendecomposition with a
pseudo-inverse. The assertion below requires them to agree, so that the figures
and the predicted ADPs describe the same model that was fit.

In [ ]:
def diffuse_intensity_batch(K_lab, h_arr, k_arr, l_arr, chunk=None,
                            min_eig_frac=1e-6, G_arr=None):
    """I(q) = G^† D(q)^+ G, vectorized.

    FULL COMPLEX G and D. D is periodic in the reciprocal lattice, so D(G)=D(0)
    at every Bragg position: the 3 acoustic eigenvalues vanish there and a
    pseudo-inverse keeps I finite. Immediately off a Bragg spot the acoustic
    eigenvalues grow as |δq|², producing the I ~ |q-G|^-2 halos.
    """
    chunk = chunk or CHUNK_EVAL
    h_arr = np.asarray(h_arr, float); k_arr = np.asarray(k_arr, float)
    l_arr = np.asarray(l_arr, float)
    N = len(h_arr); I_out = np.zeros(N)
    for s in range(0, N, chunk):
        sl = slice(s, min(s+chunk, N))
        qb = (h_arr[sl,None]*B_recip[:,0] + k_arr[sl,None]*B_recip[:,1]
              + l_arr[sl,None]*B_recip[:,2])
        Db = dynamical_matrix_batch(qb, unique, K_lab)
        Gb = G_arr[sl] if G_arr is not None else G_at_hkl_batch(h_arr[sl], k_arr[sl], l_arr[sl])
        ev, evec = np.linalg.eigh(Db)                      # complex Hermitian
        emax = np.clip(ev.max(axis=1, keepdims=True), 1e-15, None)
        pos = ev > (min_eig_frac*emax)
        ev_inv = np.where(pos, 1.0/np.where(pos, ev, 1.0), 0.0)
        proj = np.einsum('nji,nj->ni', evec.conj(), Gb) * ev_inv
        DinvG = np.einsum('nij,nj->ni', evec, proj)
        I_out[sl] = np.real(np.einsum('ni,ni->n', Gb.conj(), DinvG))
    return I_out

def forward_I(K_lab, h, k, l, G=None):
    """Single-point I(q) by a plain (non-pseudo-inverse) solve -- used where
    D(q) is known to be comfortably invertible, i.e. away from Bragg points."""
    q = h*B_recip[:,0] + k*B_recip[:,1] + l*B_recip[:,2]
    if G is None:
        G = G_at_hkl(h, k, l)
    D = dynamical_matrix(q, unique, K_lab)
    return float(np.real(np.conj(G) @ np.linalg.solve(D, G)))

# CONSISTENCY CHECK. Two independent code paths evaluate I(q): the refinement
# objective uses forward_I's linear solve, while the maps and the Brillouin-zone
# integral use diffuse_intensity_batch's eigendecomposition with a pseudo-inverse.
# They must agree, or the figures and the predicted ADPs would describe a
# different model from the one being fit.
_hq = np.array([1.37, -2.13, 0.61, 3.05, -1.44])
_kq = np.array([0.22,  1.81, -2.4, 0.13,  2.77])
_lq = np.array([0.45, -0.33, 1.12, -2.06, 0.88])
_Ia = diffuse_intensity_batch(K_LAB_PRIOR, _hq, _kq, _lq)
_Ib = np.array([forward_I(K_LAB_PRIOR, h, k, l) for h, k, l in zip(_hq, _kq, _lq)])
_rel = np.abs(_Ia - _Ib)/np.maximum(np.abs(_Ib), 1e-30)
print(f"Batched vs. single-point I(q): max relative difference {_rel.max():.3e}")
assert _rel.max() < 1e-8, "the two intensity code paths disagree -- see markdown"
print("OK: both intensity paths agree, so the plotted model is the fitted model.")


## Where the Fitted q-Points Sit

The left panel shows all fitting points in true Cartesian $\mathbf q$; the two
right panels overlay them on the experimental map at $l=0$ and $l\approx0.5$,
single 2-D slices read directly from the HDF5 file. The overlay selects a slab of
finite thickness in $l$ rather than an exact layer, so the sampling is visible
relative to the halos.

In [ ]:
def nearest_l_index(l_target):
    return int(round((l_target - grid_ori[2])/grid_delta[2]))

def read_hk_plane(l_index):
    with h5py.File(H5_FILE, 'r') as f:
        I_plane = f[f'{MAP_GROUP}/I'][:, :, l_index]
    h_pts = grid_ori[0] + np.arange(grid_size[0])*grid_delta[0]
    k_pts = grid_ori[1] + np.arange(grid_size[1])*grid_delta[1]
    return h_pts, k_pts, np.asarray(I_plane)

l_idx_0    = nearest_l_index(0.0)
l_idx_half = nearest_l_index(0.5)
h_pts_bg0,  k_pts_bg0,  I_bg0  = read_hk_plane(l_idx_0)
h_pts_bg05, k_pts_bg05, I_bg05 = read_hk_plane(l_idx_half)
l_layer_0  = grid_ori[2] + l_idx_0*grid_delta[2]
l_layer_05 = grid_ori[2] + l_idx_half*grid_delta[2]

SLAB = 4*grid_delta[2]      # slab half-thickness for the overlay

qc_fit  = q_fit  @ B_recip.T
qc_test = q_test @ B_recip.T

fig = plt.figure(figsize=(16, 5))
ax3d = fig.add_subplot(1, 3, 1, projection='3d')
ax3d.scatter(*qc_fit[~is_halo].T, s=6, alpha=0.4, color='steelblue', label='mid-zone')
if is_halo.any():
    ax3d.scatter(*qc_fit[is_halo].T, s=22, alpha=0.9, color='limegreen', label='halo bins')
ax3d.scatter(*qc_test.T, s=6, alpha=0.5, color='orangered', label='held out')
ax3d.set_xlabel('$q_x$'); ax3d.set_ylabel('$q_y$'); ax3d.set_zlabel('$q_z$')
ax3d.set_title('Sampled q-points (Å$^{-1}$)'); ax3d.legend(fontsize=8)

def overlay_panel(ax, h_pts, k_pts, I_bg, l_layer):
    pos = I_bg[np.isfinite(I_bg) & (I_bg > 0)]
    vmax = np.percentile(pos, 97) if len(pos) else 1.0
    I_disp = np.log1p(np.clip(np.nan_to_num(I_bg, nan=0.0), 0, vmax))/np.log(10)
    ax.imshow(I_disp.T, origin='lower', cmap='gray',
              extent=[h_pts[0], h_pts[-1], k_pts[0], k_pts[-1]], aspect='auto', alpha=0.85)
    for pts, color, label in [(q_fit, 'steelblue', 'fitting'), (q_test, 'orangered', 'held out')]:
        sel = np.abs(pts[:,2] - l_layer) < SLAB
        ax.scatter(pts[sel,0], pts[sel,1], s=8, color=color, edgecolor='none',
                   alpha=0.6, label=f'{label} ({sel.sum()})')
    ax.set_xlabel('h'); ax.set_ylabel('k')
    ax.set_title(f'l ≈ {l_layer:.3f} ± {SLAB:.3f}  (experimental map)')
    ax.legend(fontsize=7, loc='upper right')

overlay_panel(fig.add_subplot(1,3,2), h_pts_bg0,  k_pts_bg0,  I_bg0,  l_layer_0)
overlay_panel(fig.add_subplot(1,3,3), h_pts_bg05, k_pts_bg05, I_bg05, l_layer_05)
plt.tight_layout(); plt.savefig('sampled_q_points.png', dpi=150); plt.show()

## Refining $K$ to the Experimental Data

$$\mathcal L(\theta,s)=\sum_{\mathbf q}w(\mathbf q)
\bigl[s\,I_{\rm model}(\mathbf q;\theta)-I_{\rm obs}(\mathbf q)\bigr]^2
+\lambda\sum_{\mathbf n}\bigl\|\log\bigl(K_{\mathbf n,\rm prior}^{-1/2}
K_{\mathbf n}K_{\mathbf n,\rm prior}^{-1/2}\bigr)\bigr\|_F^2$$

with $w=1/\sigma_{\rm exp}^2$ from the deposited map's own per-voxel error
estimate (floored at a low percentile), and the second term the geodesic distance
to the geometric prior. Because that distance is invariant under
$K\to TKT^{\mathsf T}$, it is the same whether evaluated on local-frame or
lab-frame stiffnesses.

### The $K$/scale degeneracy

$D$ is linear in $K$, so $K\to\lambda K$ sends $I_{\rm model}\to I_{\rm model}/\lambda$
exactly, and the predicted ADPs rescale the same way. With $s$ free, the
combination $\lambda s$ is therefore exactly unidentifiable from the diffuse map.

Two ways of handling that fail. Profiling $s$ freely at every iteration is
mathematically consistent but makes the data loss exactly invariant to the overall
magnitude of $K$, leaving that direction controlled only by the regularizer, which
prefers the prior — so the magnitude is set by the prior rather than the data.
Pinning $s$ once from an arbitrary starting $K$ does not remove the degeneracy at
all; it picks an arbitrary point on it, and the data then constrains $K$ only
relative to a number chosen in advance.

A third approach — rescaling the fitted $K$ afterwards so the predicted mean $B$
matches the deposited mean — is worse than either. It makes the headline
validation a tautology, and it over-softens $K$, since the deposited $B$ contains
internal and substitutional disorder on top of the lattice motion.

**What is done instead.** The degenerate direction is removed from the parameter
space: the refinement runs on the normalized *shape* of $K$, with
$\sum_{\mathbf n}\mathrm{tr}\,K_{\mathbf n}$ held fixed and $s$ profiled in closed
form each call. There is then no runaway direction, and the regularizer is a mild
bias rather than the only thing holding the magnitude. Afterwards the gauge is
closed with **units**: with $k_BT=1$ and $G$ in $e/\mathrm{Å}$, $I_{\rm model}$ is
in electrons² per unit cell, the same scale the deposited map reports, so
$s\equiv1$. That fixes $K$ absolutely using no information from the ADPs.

The profiled $s^\star$ at the fitted shape is itself a result: a value far from 1
says something real about the model — missing non-lattice scattering, multi-phonon
terms, or a normalization difference in the deposited map — and is reported rather
than absorbed.

**Order of operations.** `LAMBDA_PRIOR` is selected by cross-validation *before*
the main refinement, and everything downstream — the fitted $K$, the distance
diagnostics, the Gauss-Newton spectrum, the predicted ADPs — uses the selected
value. The default in the parameters cell only seeds the scan.

### Gradient and optimizer

$dI=-v^\dagger dD\,v$ with $v=D^{-1}G$, and $D$ is linear in each $K_{\mathbf n}$,
so $dI/dK_{\mathbf n}$ is closed-form. The regularizer contributes
$2\lambda K_{\rm prior}^{-1/2}(P^{-1}\log P)K_{\rm prior}^{-1/2}$ with
$P=K_{\rm prior}^{-1/2}KK_{\rm prior}^{-1/2}$. Both chain through the fixed
lab-frame transform and $K=LL^{\mathsf T}$; checked below against finite
differences. With $s$ profiled, the envelope theorem means its dependence on
$\theta$ contributes no extra term.

L-BFGS-B with the analytic gradient, unconstrained: the Cholesky
parameterization makes every point in parameter space a valid model. The loss is
vectorized over the whole point batch. Both `maxiter` and `maxfun` are set
explicitly, since L-BFGS-B caps function evaluations separately and its default
binds first on a problem like this.

**Two convergence questions, two tests.** *Is the fit converged?* Once the loss
stops improving by more than about one $\chi^2$ unit there is nothing left to
learn — the noise on $\chi^2$ itself is $\sqrt{2\,\mathrm{dof}}$, so smaller
changes cannot alter any statement about the model. That trailing-window test is
the stopping rule. *Is $K$ converged?* Not the same question: along a flat
direction the loss barely changes while $K$ travels a long way, so both the
projected gradient and the loss can look settled while the stiffnesses are still
in motion.

The refinement therefore runs L-BFGS-B in warm-started segments and reports both
per segment: the loss, split into data and penalty so it is comparable across
$\lambda$, and the **geodesic step** — how far $K$ moved, in the same metric the
regularizer uses, where 0.01 is a 1% change in one stiffness eigenvalue of one
contact.

When the loss criterion fires while the geodesic step is still large, the fit is
statistically converged but $K$ is not: the reported stiffnesses are one point in
a nearly flat region rather than a point estimate. The notebook says so
explicitly. That region is what the prior and the sloppiness spectrum describe,
and a larger `LAMBDA_PRIOR` shrinks it — which the scan below settles on held-out
data.

In [ ]:
def normalize_shape(Lmap, target=None):
    """Project onto the fixed-normalization shape manifold: Σ_n tr(K_n) = target.
    This removes the single degenerate direction from the parameter space, so the
    overall magnitude of K is carried by the profiled scale s alone and is not a
    free parameter competing with it."""
    if target is None:
        target = 6.0*len(Lmap)
    tot = sum(np.trace(L @ L.T) for L in Lmap.values())
    f = np.sqrt(target/tot)
    return {c: f*L for c, L in Lmap.items()}

def _model_raw(theta, q_arr, G_arr):
    """I_model before the overall scale, plus the solved v = D^-1 G and the
    Cartesian q batch (both reused by the gradient)."""
    Lmap = theta_to_Lmap(theta)
    K_local, K_lab = build_K(Lmap)
    q_batch = (q_arr[:,0:1]*B_recip[:,0] + q_arr[:,1:2]*B_recip[:,1]
               + q_arr[:,2:3]*B_recip[:,2])
    D_batch = dynamical_matrix_batch(q_batch, unique, K_lab)
    V = np.linalg.solve(D_batch, G_arr[:,:,None])[:,:,0]
    return np.real(np.sum(np.conj(G_arr)*V, axis=1)), V, q_batch, Lmap, K_local

def profiled_scale(theta, q_arr, G_arr, I_obs, weights):
    """Closed-form weighted least-squares scale. Valid via the envelope theorem;
    exposed standalone so it can be recomputed once from the FIXED fitted shape
    for reporting, always from the fitting set, never from the held-out set."""
    Iraw, _, _, _, _ = _model_raw(theta, np.asarray(q_arr, float), np.asarray(G_arr))
    denom = np.sum(weights*Iraw**2)
    return float(np.sum(weights*Iraw*I_obs)/denom) if denom > 0 else 1.0

def loss_and_grad(theta, q_arr, G_arr, I_obs, weights):
    """Weighted least squares with s profiled in closed form, plus the
    prior-centred regularizer. Vectorized over the whole observation batch."""
    q_arr = np.asarray(q_arr, float); G_arr = np.asarray(G_arr)
    I_obs = np.asarray(I_obs); weights = np.asarray(weights)
    Iraw, V, q_batch, Lmap, K_local = _model_raw(theta, q_arr, G_arr)

    denom = np.sum(weights*Iraw**2)
    s = float(np.sum(weights*Iraw*I_obs)/denom) if denom > 0 else 1.0
    resid = s*Iraw - I_obs
    data_loss = float(np.sum(weights*resid**2))
    coeff = 2.0*weights*resid*s     # envelope theorem: ds/dtheta contributes nothing

    N_mat = {c: Gc_inv_matrix(unique[c]['c_local']) @ unique[c]['Gamma'].T
             for c in shell_order}
    grad_L = {}
    for c in shell_order:
        R_n = unique[c]['R_n']
        A_n = ad_translation(R_n)
        phase = np.exp(1j*(q_batch @ R_n))
        U  = V @ A_n.T
        w1 = U - phase[:,None]*V
        w2 = V - phase.conj()[:,None]*U
        Graw = -(np.einsum('ni,nj->nij', U.conj(), w1) + np.einsum('ni,nj->nij', V.conj(), w2))
        G_A  = np.einsum('ij,njk,lk->nil', N_mat[c], Graw, N_mat[c])
        dL_c = np.real(np.einsum('nij,jk->nik', G_A + G_A.transpose(0,2,1), Lmap[c]))
        grad_L[c] = np.einsum('n,nij->ij', coeff, dL_c)

    reg  = regularizer_prior(K_local)
    dRdK = regularizer_prior_dK(K_local)
    for c in shell_order:
        grad_L[c] += (dRdK[c] + dRdK[c].T) @ Lmap[c]
        grad_L[c] = np.tril(grad_L[c])

    return data_loss + reg, Lmap_to_theta(grad_L)

In [ ]:
# Gradient check: analytic vs. central finite differences. The scale is profiled
# inside loss_and_grad, so this also exercises the envelope-theorem term being
# correctly absent.
rng_chk = np.random.default_rng(42)
theta_chk = Lmap_to_theta(normalize_shape(L_PRIOR))*(1 + 0.05*rng_chk.standard_normal(N_PARAMS))
n_chk = min(40, len(q_fit))
q_chk, G_chk, I_chk, w_chk = q_fit[:n_chk], G_fit[:n_chk], I_fit[:n_chk], w_fit[:n_chk]

loss0, grad0 = loss_and_grad(theta_chk, q_chk, G_chk, I_chk, w_chk)
eps, errs = 1e-6, []
for idx in rng_chk.choice(N_PARAMS, size=min(8, N_PARAMS), replace=False):
    tp = theta_chk.copy(); tp[idx] += eps
    tm = theta_chk.copy(); tm[idx] -= eps
    fd = (loss_and_grad(tp, q_chk, G_chk, I_chk, w_chk)[0]
          - loss_and_grad(tm, q_chk, G_chk, I_chk, w_chk)[0])/(2*eps)
    errs.append(abs(fd - grad0[idx])/max(abs(fd), 1e-6))
print(f"Gradient check, max relative error over {len(errs)} random directions: {max(errs):.2e}")
assert max(errs) < 1e-3

In [ ]:
# --- Convergence criteria --------------------------------------------------
# Two different questions, and they need two different tests.
#
#   Is the FIT converged?  Statistically, once the loss stops improving by more
#   than about one chi-squared unit there is nothing left to learn: the noise on
#   chi-squared itself is sqrt(2*dof), so smaller changes cannot alter any
#   statement about the model. This is the stopping rule.
#
#   Is K converged?  Not the same thing. Along a flat direction the loss barely
#   moves while K travels a long way, so the projected gradient -- and the loss
#   itself -- can look settled while the stiffnesses are still in motion. The
#   geodesic step measures that directly, in the same metric the regularizer
#   uses: a step of 0.01 is a 1% change in one stiffness eigenvalue of one
#   contact. It is reported every segment and is the diagnostic that says whether
#   the reported K is a point estimate or one point in a flat region.

def geodesic_step(theta_a, theta_b):
    """Largest geodesic distance any single contact's K moved between two
    iterates. exp(d) is the factor along the worst-affected direction."""
    Ka, _ = build_K(theta_to_Lmap(theta_a))
    Kb, _ = build_K(theta_to_Lmap(theta_b))
    worst = 0.0
    for c in shell_order:
        Minv = _sqrtm_inv_sym(Ka[c])
        P = Minv @ Kb[c] @ Minv
        w = np.linalg.eigvalsh(0.5*(P + P.T))
        w = np.maximum(w, REG_EIG_FLOOR*max(w.max(), 1e-300))
        worst = max(worst, float(np.sqrt(np.sum(np.log(w)**2))))
    return worst

BLOCK     = 500     # iterations per segment
MAX_BLOCK = 80      # segments
LOSS_TOL  = 1.0     # stop when the loss improves by less than this (chi-squared
LOSS_WIN  = 5       # units) over LOSS_WIN consecutive segments
GEO_TOL   = 1e-2    # also stop if K itself stops moving this much per segment

def refine(theta_start, args, block=BLOCK, max_block=MAX_BLOCK,
           loss_tol=LOSS_TOL, loss_win=LOSS_WIN, geo_tol=GEO_TOL, verbose=True):
    """L-BFGS-B in warm-started segments, stopping on the trailing-window loss
    improvement, with the geodesic step reported alongside as a diagnostic.

    Segments rather than one call so both quantities can be measured between
    them; ftol and gtol are zero within a segment so it always runs its full
    block. L-BFGS-B rebuilds its curvature estimate each segment, which costs a
    few percent of the iterations.
    """
    theta = np.asarray(theta_start, float).copy()
    hist  = [theta.copy()]
    nit = nfev = 0
    rows, why = [], "max_block"
    if verbose:
        print(f"   {'seg':>3s} {'iters':>7s} {'data loss':>12s} {'penalty':>9s} "
              f"{'K moved':>10s}")
    for b in range(max_block):
        res = minimize(loss_and_grad, theta, args=args, jac=True, method='L-BFGS-B',
                       options={'maxiter': block, 'maxfun': 4*block,
                                'ftol': 0.0, 'gtol': 0.0},
                       callback=lambda xk: hist.append(xk.copy()))
        step = geodesic_step(theta, res.x)
        theta = res.x; nit += res.nit; nfev += res.nfev
        K_local, _ = build_K(theta_to_Lmap(theta))
        pen = regularizer_prior(K_local)
        rows.append((nit, res.fun, res.fun - pen, pen, step))
        if verbose:
            print(f"   {b+1:3d} {nit:7d} {res.fun-pen:12.6g} {pen:9.4f} {step:10.3e}")
        if step < geo_tol:
            why = "geodesic"; break
        if len(rows) > loss_win and (rows[-1-loss_win][1] - rows[-1][1]) < loss_tol:
            why = "loss"; break
    return theta, hist, nit, nfev, rows, why


In [ ]:
def fit_stats(K_lab, q_arr, G_arr, I_arr, scale=1.0):
    """R-factor AND the linear correlation coefficient.

    R alone is a poor summary once observations can be negative: Σ|I_obs| in the
    denominator is inflated by the magnitude of negatives, and R → 1 is exactly
    what a model predicting ~0 produces -- so R cannot tell 'poor fit' from
    'no fit'. CC separates those cases and is the statistic the diffuse-scattering
    literature reports.
    """
    Imodel = scale*diffuse_intensity_batch(K_lab, q_arr[:,0], q_arr[:,1], q_arr[:,2],
                                           G_arr=np.asarray(G_arr))
    R  = np.sum(np.abs(Imodel - I_arr))/np.sum(np.abs(I_arr))
    CC = np.corrcoef(Imodel, I_arr)[0,1] if len(I_arr) > 2 else np.nan
    return R, CC, Imodel

def geodesic_per_contact(theta_a, theta_b):
    Ka, _ = build_K(theta_to_Lmap(theta_a))
    Kb, _ = build_K(theta_to_Lmap(theta_b))
    out = {}
    for c in shell_order:
        Minv = _sqrtm_inv_sym(Ka[c])
        P = Minv @ Kb[c] @ Minv
        w = np.linalg.eigvalsh(0.5*(P + P.T))
        w = np.maximum(w, REG_EIG_FLOOR*max(w.max(), 1e-300))
        out[c] = float(np.sqrt(np.sum(np.log(w)**2)))
    return out

THETA_PRIOR = Lmap_to_theta(normalize_shape(L_PRIOR))
print("Helpers ready; LAMBDA_PRIOR is selected below before the refinement.")


### Choosing `LAMBDA_PRIOR`

The regularizer weight decides whether the poorly-determined directions have a
restoring force at all. Too small and the optimizer slides along them
indefinitely — the geodesic step never falls below tolerance, because there is
nothing to converge to. Too large and the penalty overrides the stiff directions
the data does determine.

It is selected **before** the main refinement, so everything downstream uses the
selected value. Selection is by **$k$-fold cross-validation on $R$**, using no
ADP information, so the ADP comparison further down stays an independent check.
Cross-validation rather than a single held-out split because the noise on $R$
from one split is comparable to the variation across $\lambda$. The three panels show generalization
against $\lambda$, how far the fit moves, and whether the geodesic step reaches
tolerance at all.

In [ ]:
# --- Cross-validated selection of LAMBDA_PRIOR -----------------------------
LAMBDA_SCAN = [1e-2, 3e-2, 1e-1, 3e-1, 1.0, 3.0, 10.0]
N_FOLDS     = 3        # 1 reproduces a single held-out split
SCAN_BLOCKS = 30       # segment cap per fit during the scan

# The scan rebinds the module-level LAMBDA_PRIOR, which only works because
# regularizer_prior looks it up at call time rather than binding it as a default
# argument. Verify that rather than discovering it from identical rows.
_saved_lambda = LAMBDA_PRIOR
_probe = {shell_order[0]: 1.05*K_PRIOR_LOCAL[shell_order[0]]}
LAMBDA_PRIOR = 1.0;  _p1 = regularizer_prior(_probe)
LAMBDA_PRIOR = 10.0; _p2 = regularizer_prior(_probe)
LAMBDA_PRIOR = _saved_lambda
assert abs(_p2/_p1 - 10.0) < 1e-9, \
    "regularizer_prior is not picking up LAMBDA_PRIOR at call time"

_rng_cv = np.random.default_rng(11)
_fold = np.array_split(_rng_cv.permutation(len(q_fit)), max(N_FOLDS, 1))

def cv_score(lam):
    global LAMBDA_PRIOR
    LAMBDA_PRIOR = lam
    Rs, CCs = [], []
    if N_FOLDS > 1:
        for f in _fold:
            te = np.zeros(len(q_fit), bool); te[f] = True
            th, _, _, _, _, _ = refine(THETA_PRIOR,
                                       (q_fit[~te], G_fit[~te], I_fit[~te], w_fit[~te]),
                                       max_block=SCAN_BLOCKS, verbose=False)
            _s = profiled_scale(th, q_fit[~te], G_fit[~te], I_fit[~te], w_fit[~te])
            _, Ksh = build_K(theta_to_Lmap(th))
            R, CC, _ = fit_stats({c: _s*Ksh[c] for c in Ksh},
                                 q_fit[te], G_fit[te], I_fit[te], 1.0)
            Rs.append(R); CCs.append(CC)
    th, _, nit, _, rw, _ = refine(THETA_PRIOR, (q_fit, G_fit, I_fit, w_fit),
                                  max_block=SCAN_BLOCKS, verbose=False)
    s_star = profiled_scale(th, q_fit, G_fit, I_fit, w_fit)
    _, Ksh = build_K(theta_to_Lmap(th))
    Klab = {c: s_star*Ksh[c] for c in Ksh}
    if N_FOLDS <= 1:
        R, CC, _ = fit_stats(Klab, q_test, G_test, I_test, 1.0)
        Rs, CCs = [R], [CC]
    chi2 = rw[-1][2]/max(len(q_fit) - N_PARAMS, 1)
    return (np.mean(Rs), np.std(Rs), np.mean(CCs), chi2,
            max(geodesic_per_contact(THETA_PRIOR, th).values()),
            s_star, nit, rw[-1][4])

print(f"{N_FOLDS}-fold cross-validation over {len(q_fit)} observations\n")
print(f"{'lambda':>8s} {'chi2/dof':>9s} {'R_cv':>8s} {'±':>7s} {'CC_cv':>9s} "
      f"{'prior→fit':>10s} {'s*':>10s} {'iters':>7s} {'last step':>10s}")
_scan = []
for _lam in LAMBDA_SCAN:
    r = cv_score(_lam); _scan.append((_lam,) + r)
    print(f"{_lam:8.3g} {r[3]:9.4f} {r[0]:8.4f} {r[1]:7.4f} {r[2]:9.5f} "
          f"{r[4]:10.3f} {r[5]:10.4g} {r[6]:7d} {r[7]:10.2e}")
LAMBDA_PRIOR = _saved_lambda

# One-standard-error rule: among the values whose cross-validated R is within one
# standard error of the best, take the LARGEST lambda. The R curve is shallow
# compared with its own noise, so the bare minimum would be fitting the fold
# split; the larger lambda is the more conservative model and the better-posed
# optimization.
_Rs  = np.array([r[1] for r in _scan])
_ses = np.array([r[2] for r in _scan])/max(np.sqrt(N_FOLDS), 1.0)
_i   = int(np.argmin(_Rs)); _thr = _Rs[_i] + _ses[_i]
LAMBDA_PRIOR = float(max(r[0] for r in _scan if r[1] <= _thr))
print(f"\nbest R_cv = {_Rs[_i]:.4f} at lambda = {_scan[_i][0]:g} "
      f"(standard error {_ses[_i]:.4f})")
print(f"one-standard-error rule -> LAMBDA_PRIOR = {LAMBDA_PRIOR:g}")

_fig, _ax = plt.subplots(1, 3, figsize=(15, 4))
_l = [r[0] for r in _scan]
_ax[0].errorbar(_l, _Rs, yerr=_ses, fmt='o-', capsize=3)
_ax[0].axvline(LAMBDA_PRIOR, color='g', lw=1.5, label='selected')
_ax[0].axhline(_thr, color='r', ls=':', lw=1, label='best + 1 s.e.')
_ax[0].set_xscale('log'); _ax[0].set_xlabel(r'$\lambda$')
_ax[0].set_ylabel('cross-validated $R$'); _ax[0].set_title('Generalization')
_ax[0].legend(fontsize=8)
_ax[1].loglog(_l, [r[5] for r in _scan], 'o-')
_ax[1].axvline(LAMBDA_PRIOR, color='g', lw=1.5)
_ax[1].set_xlabel(r'$\lambda$'); _ax[1].set_ylabel('geodesic prior→fit')
_ax[1].set_title('How far the fit moves')
_ax[2].loglog(_l, [max(r[8], 1e-12) for r in _scan], 'o-')
_ax[2].axhline(GEO_TOL, color='r', ls='--', lw=1, label='GEO_TOL')
_ax[2].axvline(LAMBDA_PRIOR, color='g', lw=1.5)
_ax[2].set_xlabel(r'$\lambda$'); _ax[2].set_ylabel('geodesic step, final segment')
_ax[2].set_title('Is the problem well-posed?'); _ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig('lambda_scan.png', dpi=150); plt.show()


### Refinement at the Selected $\lambda$

Everything below — the fitted stiffnesses, the distance diagnostics, the
Gauss-Newton spectrum, the predicted ADPs, the mode animations — uses the
`LAMBDA_PRIOR` chosen above.

In [ ]:
print(f"Refining (segments of {BLOCK} iterations)")
print(f"  stop when the loss improves by < {LOSS_TOL} over {LOSS_WIN} segments, "
      f"or K moves < {GEO_TOL:g} in one")
theta_fit, theta_history, NIT, NFEV, _rows, _why = refine(
    Lmap_to_theta(normalize_shape(L_PRIOR)), (q_fit, G_fit, I_fit, w_fit))
theta0 = theta_history[0]
_, K_LAB_SHAPE = build_K(theta_to_Lmap(theta_fit))

_step = _rows[-1][4]
_dl   = (_rows[-1-LOSS_WIN][1] - _rows[-1][1]) if len(_rows) > LOSS_WIN else float('nan')
print(f"\nStopped after {NIT} iterations ({NFEV} evaluations) on: {_why}")
print(f"  loss improvement over the last {LOSS_WIN} segments: {_dl:.4f} χ² units")
print(f"  K moved {_step:.3e} in the final segment "
      f"(a factor {np.exp(_step):.3f} along the worst direction)")

# --- Close the gauge with UNITS, not with the ADPs -------------------------
# I_model = G^dag D^-1 G with k_BT = 1 and G in e/Å is in electrons² per unit
# cell, the same scale the deposited map reports, so the physical statement is
# s == 1. Absorbing the profiled s into K achieves that using no information
# from the deposited B-factors.
S_STAR = profiled_scale(theta_fit, q_fit, G_fit, I_fit, w_fit)
K_LAB_FIT = {c: S_STAR*K_LAB_SHAPE[c] for c in K_LAB_SHAPE}
S_FIT = 1.0
print(f"  profiled scale s* = {S_STAR:.4g}  (a value far from 1 is a statement")
print( "    about the model, not something to absorb silently)")

chi2_dof = _rows[-1][2]/max(len(q_fit) - N_PARAMS, 1)
print(f"  weighted χ²/dof (training) = {chi2_dof:.4f}")

if _why == "loss" and _step > 10*GEO_TOL:
    print("\nThe FIT is statistically converged but K is NOT: it was still moving")
    print(f"{_step:.2f} per {BLOCK} iterations when the loss stopped improving. The")
    print("reported K is therefore one point in a nearly flat region, not a point")
    print("estimate. That region is what the prior and the sloppiness spectrum")
    print("describe; a larger LAMBDA_PRIOR shrinks it (see the scan below).")
elif _why == "max_block":
    print("\nHit MAX_BLOCK without meeting either criterion -- raise it, or raise")
    print("LAMBDA_PRIOR if the geodesic step is not decaying.")

In [ ]:
# R and CC, at the selected lambda.
R_fit,  CC_fit,  I_model_fit  = fit_stats(K_LAB_FIT, q_fit,  G_fit,  I_fit,  S_FIT)
R_free, CC_free, I_model_test = fit_stats(K_LAB_FIT, q_test, G_test, I_test, S_FIT)
print(f"fitting set:  R = {R_fit:.4f}   CC = {CC_fit:.4f}")
print(f"held out:     R = {R_free:.4f}   CC = {CC_free:.4f}")
if len(hkl_halo):
    Rh, CCh, _ = fit_stats(K_LAB_FIT, hkl_halo, G_fit[is_halo], I_halo, S_FIT)
    print(f"halo bins:    R = {Rh:.4f}   CC = {CCh:.4f}")
print("\nR → 1 together with CC → 0 means the model is predicting essentially")
print("nothing; R → 1 with CC substantial means a scale or shape mismatch instead.")

# --- Per-resolution-shell breakdown: where does the model stop working? ----
q_abs = np.linalg.norm(q_test @ B_recip.T, axis=1)
d_test = 2*np.pi/np.maximum(q_abs, 1e-9)
edges = np.quantile(d_test, np.linspace(0, 1, 7))
print(f"\nHeld-out CC by resolution shell:")
shell_d, shell_cc = [], []
for b in range(len(edges)-1):
    m = (d_test >= edges[b]) & (d_test <= edges[b+1])
    if m.sum() < 20: continue
    cc = np.corrcoef(I_model_test[m], I_test[m])[0,1]
    shell_d.append(0.5*(edges[b]+edges[b+1])); shell_cc.append(cc)
    print(f"  {edges[b]:5.2f}–{edges[b+1]:5.2f} Å  ({m.sum():5d} pts):  CC = {cc:+.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15,4.5))
for ax, Im, Io, title in [(axes[0], I_model_fit, I_fit, 'fitting set'),
                          (axes[1], I_model_test, I_test, 'held-out set')]:
    ax.scatter(Io, Im, s=6, alpha=0.35)
    lim = [min(Io.min(), Im.min())*1.05, max(Io.max(), Im.max())*1.05]
    ax.plot(lim, lim, 'k--', lw=1); ax.axhline(0, color='0.7', lw=0.6); ax.axvline(0, color='0.7', lw=0.6)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('$I_{obs}$ (experimental)'); ax.set_ylabel('$I_{model}$'); ax.set_title(title)
axes[2].plot(shell_d, shell_cc, 'o-', color='darkgreen')
axes[2].axhline(0, color='k', lw=0.8, ls='--')
axes[2].invert_xaxis(); axes[2].set_xlabel('resolution $d$ (Å)'); axes[2].set_ylabel('held-out CC')
axes[2].set_title('Where the model stops working')
plt.tight_layout(); plt.savefig('refinement_fit_quality.png', dpi=150); plt.show()

### Distance Travelled from the Prior

With no ground truth available, the interpretable quantity is how far each contact
moved from its geometric prior, in the metric that respects the units of $K$.
$e^{d}$ is the factor along the worst-affected direction, so $d\approx1$ is a
factor $e$ and $d\approx3$ a factor of 20. Values that large in a direction the
data barely constrains point at `LAMBDA_PRIOR` rather than at the data.

In [ ]:
# --- How far did the fit travel from the prior? ----------------------------
_d_pf = geodesic_per_contact(THETA_PRIOR, theta_fit)
print(f"{'contact':14s} {'prior→fit':>10s} {'worst-direction factor':>24s}")
for c in shell_order:
    print(f"{str(c):14s} {_d_pf[c]:10.3f} {np.exp(_d_pf[c]):24.1f}")
print(f"{'worst':14s} {max(_d_pf.values()):10.3f} {np.exp(max(_d_pf.values())):24.1f}")

## How Many Parameters Does the Data Determine?

126 free parameters, and no reason to expect the data to constrain 126
independent combinations of them.

The Gauss-Newton Hessian of the data term, $H_{\rm data}=J^{\mathsf T}J$ with
$J_{ij}=\sqrt{w_i}\,\partial I_i/\partial\theta_j$, measures how sharply the data
constrains each direction. Its small eigenvalues are not meaningful on their own:
in those directions the fit is held by the prior, not the data.

The informative comparison is therefore against the prior's own curvature
$H_{\rm prior}=\partial^2R/\partial\theta^2$ at the same point. The generalized
eigenvalues of the pencil $(H_{\rm data},H_{\rm prior})$ are dimensionless — above
1 the data constrains that direction more tightly than the prior, below 1 the
prior carries it — so $n_{\rm eff}$ follows without an arbitrary noise-floor
cutoff.

Both panels are shown: the raw data spectrum, whose near-geometric decay over many
decades is the signature of a sloppy model, and the data-to-prior ratio, which is
the one to read $n_{\rm eff}$ off. The $N-n_{\rm eff}$ prior-carried directions are
not measurements and a fitted $K$ should be reported saying so.

In [ ]:
# --- How many parameters does the data determine? --------------------------
# The Gauss-Newton Hessian of the DATA term,
#     H_data = J^T J,   J_ij = sqrt(w_i) dI_i/dtheta_j
# measures how sharply the data constrains each parameter combination. On its own
# its small eigenvalues are meaningless, because in those directions the fit is
# held by the prior rather than the data.
#
# So the informative comparison is against the prior's own curvature, H_prior =
# d^2R/dtheta^2, at the same point. The generalized eigenvalues of the pencil
# (H_data, H_prior) are dimensionless: a value above 1 means the data constrains
# that direction more tightly than the prior does, below 1 means the prior is
# carrying it. n_eff is the count above 1 -- a definition that needs no arbitrary
# noise-floor cutoff.

def weighted_jacobian(theta, q_list, G_list, weights, scale=1.0, eps=1e-5):
    sw = np.sqrt(np.asarray(weights, float))
    q_list = np.asarray(q_list, float); G_arr = np.asarray(G_list)
    qb = (q_list[:,0:1]*B_recip[:,0] + q_list[:,1:2]*B_recip[:,1]
          + q_list[:,2:3]*B_recip[:,2])
    def I_of(th):
        _, K_lab = build_K(theta_to_Lmap(th))
        D = dynamical_matrix_batch(qb, unique, K_lab)
        V = np.linalg.solve(D, G_arr[:,:,None])[:,:,0]
        return scale*np.real(np.sum(np.conj(G_arr)*V, axis=1))
    Jm = np.zeros((len(q_list), len(theta)))
    for j in range(len(theta)):
        tp = theta.copy(); tp[j] += eps
        tm = theta.copy(); tm[j] -= eps
        Jm[:, j] = sw*(I_of(tp) - I_of(tm))/(2*eps)
    return Jm

def reg_grad_theta(theta):
    """dR/dtheta for the geodesic penalty, through K = L L^T."""
    Lmap = theta_to_Lmap(theta)
    K_local, _ = build_K(Lmap)
    dRdK = regularizer_prior_dK(K_local)
    return Lmap_to_theta({c: np.tril((dRdK[c] + dRdK[c].T) @ Lmap[c])
                          for c in shell_order})

def regularizer_hessian(theta, eps=1e-5):
    """d^2R/dtheta^2 by central differences on the analytic gradient."""
    n = len(theta); H = np.zeros((n, n))
    for j in range(n):
        tp = theta.copy(); tp[j] += eps
        tm = theta.copy(); tm[j] -= eps
        H[:, j] = (reg_grad_theta(tp) - reg_grad_theta(tm))/(2*eps)
    return 0.5*(H + H.T)

def report_sloppiness(theta, q_list, G_list, weights, scale=1.0,
                      fname='sloppiness_spectrum.png'):
    Jm = weighted_jacobian(theta, q_list, G_list, weights, scale)
    H_data  = Jm.T @ Jm
    H_prior = 0.5*regularizer_hessian(theta)   # R acts as -2 ln(prior),
                                              # so its precision is R''/2
    Binv    = _sqrtm_inv_sym(H_prior)
    gen     = np.linalg.eigvalsh(0.5*((Binv @ H_data @ Binv)
                                      + (Binv @ H_data @ Binv).T))[::-1]
    ev_data = np.linalg.eigvalsh(H_data)[::-1]
    n_eff   = int(np.sum(gen > 1.0))

    print(f"Data-term spectrum:  λ_max {ev_data[0]:.4e}   λ_min "
          f"{max(ev_data[-1], 0):.4e}   dynamic range "
          f"{ev_data[0]/max(ev_data[-1], 1e-300):.2e}")
    print(f"Data vs prior (generalized eigenvalues of H_data against H_prior):")
    print(f"   above 1 (data-dominated): {n_eff} / {len(gen)}")
    print(f"   largest {gen[0]:.3e},  median {np.median(gen):.3e},  "
          f"smallest {gen[-1]:.3e}")
    print(f"   -> {len(gen)-n_eff} directions are carried by the prior. Report K")
    print( "      with that stated; they are not measurements.")

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].semilogy(np.arange(1, len(ev_data)+1),
                   np.maximum(ev_data, 1e-300)/ev_data[0], 'o-', ms=3)
    ax[0].set_xlabel('index'); ax[0].set_ylabel(r'$\lambda_i/\lambda_1$')
    ax[0].set_title('Data term $J^TJ$ (sloppy spectrum)')
    ax[1].semilogy(np.arange(1, len(gen)+1), np.maximum(gen, 1e-300), 'o-', ms=3)
    ax[1].axhline(1.0, color='r', ls='--', lw=1, label='data = prior')
    ax[1].axvline(n_eff+0.5, color='k', ls=':', lw=1, label=f'$n_{{eff}}$ = {n_eff}')
    ax[1].set_xlabel('index'); ax[1].set_ylabel('data curvature / prior curvature')
    ax[1].set_title(f'What the data determines ($\\lambda$ = {LAMBDA_PRIOR:g})')
    ax[1].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()
    return gen, n_eff


In [ ]:
def bz_covariance(K_lab, n_grid=BZ_NGRID, offset=True, eig_floor=None, chunk=None):
    """Σ = <(Ω,v)(Ω,v)^T>, from integrating D(q)^-1 over the Brillouin zone.

    Two details of the quadrature matter:

    * the grid is OFFSET (Monkhorst-Pack style) so it never lands on Γ. The
      acoustic contribution goes as ∫d³q/q² -- convergent in 3D but easy to
      undersample with a coarse Γ-inclusive grid, and Σ is the ADP prediction;
    * the eigenvalue floor is ABSOLUTE rather than relative to ev.max() at each
      q. With a strongly anisotropic K a genuine small acoustic eigenvalue near
      Γ could otherwise be zeroed merely because ev.max() is large there.
    """
    chunk = chunk or CHUNK_EVAL
    g = (np.arange(n_grid) + (0.5 if offset else 0.0))/n_grid
    Ig, Jg, Kg = np.meshgrid(g, g, g, indexing='ij')
    q_batch = np.column_stack([Ig.ravel(), Jg.ravel(), Kg.ravel()]) @ B_recip.T
    N = len(q_batch)

    if eig_floor is None:
        Dref = dynamical_matrix_batch(q_batch[:min(N, 2000)], unique, K_lab)
        eig_floor = 1e-9 * mass_weighted_eigs(Dref).max()

    Sigma = np.zeros((6,6))
    for s0 in range(0, N, chunk):
        sl = slice(s0, min(s0+chunk, N))
        Db = dynamical_matrix_batch(q_batch[sl], unique, K_lab)
        Dw = np.einsum('ij,njk,lk->nil', Msq_inv, Db, Msq_inv)
        ev, evec = np.linalg.eigh(Dw)
        e_lab = np.einsum('ij,njk->nik', Msq_inv.T, evec)
        mask  = ev > eig_floor
        coeff = np.where(mask, 1.0/np.where(mask, ev, 1.0), 0.0)
        Sigma += np.real(np.einsum('ns,nis,njs->ij', coeff, e_lab, e_lab.conj()))
    return Sigma/N

def atom_adp_tensors(Sigma, positions):
    """Full anisotropic displacement tensor U (Å², 3x3) per atom. Vectorized."""
    d = np.asarray(positions) - r_cm_at
    N = len(d)
    Jr = np.zeros((N, 3, 6))
    Jr[:,0,1] =  d[:,2]; Jr[:,0,2] = -d[:,1]
    Jr[:,1,0] = -d[:,2]; Jr[:,1,2] =  d[:,0]
    Jr[:,2,0] =  d[:,1]; Jr[:,2,1] = -d[:,0]
    Jr[:,0,3] = Jr[:,1,4] = Jr[:,2,5] = 1.0
    return np.einsum('nia,ab,njb->nij', Jr, Sigma, Jr)

def isotropic_B(U):
    return (8*np.pi**2/3) * np.trace(U, axis1=1, axis2=2)

def mean_predicted_B(K_lab, n_grid=BZ_NGRID):
    return isotropic_B(atom_adp_tensors(bz_covariance(K_lab, n_grid=n_grid), apos)).mean()

def check_bz_convergence(K_lab, grids=(8, 12, 16, BZ_NGRID)):
    """Σ is the headline ADP prediction -- never report it without this."""
    print("BZ-grid convergence of the predicted mean B:")
    prev = None
    for n in grids:
        B = mean_predicted_B(K_lab, n_grid=n)
        delta = '' if prev is None else f'   Δ = {100*(B-prev)/prev:+.2f}%'
        print(f"  n_grid={n:3d}³ ({n**3:6d} q-points):  mean B = {B:8.3f} Å²{delta}")
        prev = B


## What the Data Actually Determines

$K$ itself is poorly determined, but that is not the same as saying the data is
uninformative. The quantity measured at each $\mathbf q$ is

$$I(\mathbf q)=G^\dagger D^{-1}G=\mathrm{Tr}\bigl[D(\mathbf q)^{-1}GG^\dagger\bigr]
=\sum_s\frac{|G\cdot e_s|^2}{\omega_s^2}$$

a single projection of $D^{-1}$ onto one rank-1 direction, and weighted as
$1/\omega^2$. Two consequences follow, and they explain everything seen above.

**The data measures the soft sector.** A branch three times stiffer contributes
nine times less intensity. The stiff directions of $K$ are therefore nearly
invisible, which is where the $\sim53$ prior-carried directions come from. It is
not a defect of the sampling — it is what thermal diffuse scattering is.

**$\Sigma$ carries exactly the same weighting.** The displacement covariance
$\Sigma=\langle uu^\dagger\rangle=N^{-1}\sum_{\mathbf q}D(\mathbf q)^{-1}$ is the
same $1/\omega^2$-weighted object, integrated over the zone rather than projected
onto $G$. So the ADPs are determined about as well as the intensities are, even
though $K$ is not. This matters for reading the ADP comparison below as an independent check: it is
a genuinely different functional, integrated over the whole zone rather than
evaluated at the fitted $\mathbf q$, but it probes the same soft sector the fit
was trained on, so it is a weaker test than it first appears.

**Why the band structure looks converged.** $\omega(\mathbf q)$ is 6 numbers per
$\mathbf q$; $D(\mathbf q)$ is 36 real numbers. Matching eigenvalues is a 6-of-36
compression, and matrices related by unitary conjugation share them exactly. So a
band structure can agree closely while $D$ — and hence $K$ — differs a great deal.

Note the degeneracy is in the *data*, not the model: in
$D(\mathbf q)=\sum_{\mathbf n}[A^\dagger KA+K-e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}A^\dagger K-\mathrm{h.c.}]$
the factors $e^{i\mathbf q\cdot\mathbf R_{\mathbf n}}$ are independent functions of
$\mathbf q$ for distinct $\mathbf R_{\mathbf n}$, so $K_{\mathbf n}$ *is* recoverable
from $D$ known everywhere. It is the finite, $1/\omega^2$-weighted sampling that
loses it.

The cell below turns this from an argument into a measurement. Sampling the
posterior $\theta\sim\mathcal N(\hat\theta,\,\mathcal C)$ with
$\mathcal C=(J^{\mathsf T}J+\tfrac12\partial^2R/\partial\theta^2)^{-1}$ and
propagating through, it shows which *physical* quantities survive: the dispersion
with an uncertainty band, the predicted mean $B$, and the per-contact spread in
$K$ measured in the geodesic metric.

In [ ]:
# --- Propagate the posterior to physical observables -----------------------
# Covariance from the Gauss-Newton Hessian of the data term plus the prior's
# curvature. With chi^2 = sum w r^2 the Fisher information is J^T J, and treating
# the penalty R as -2 ln(prior) contributes R''/2, so C = (J^T J + H_R/2)^-1.
# Inflating by chi2/dof allows for model error on top of the quoted sigmas.

N_POST = 60          # posterior samples
INFLATE_BY_CHI2 = True

def posterior_cov_sqrt(theta, q_list, G_list, weights, scale=1.0, chi2_dof=1.0):
    Jm = weighted_jacobian(theta, q_list, G_list, weights, scale)
    H  = Jm.T @ Jm + 0.5*regularizer_hessian(theta)
    H  = 0.5*(H + H.T)
    w, V = np.linalg.eigh(H)
    w = np.maximum(w, 1e-12*w.max())
    f = np.sqrt(chi2_dof) if INFLATE_BY_CHI2 else 1.0
    return f*(V*w**-0.5)          # C^(1/2): samples = theta + Csqrt @ randn

_Csq = posterior_cov_sqrt(theta_fit, q_fit, G_fit, w_fit, scale=S_STAR, chi2_dof=chi2_dof)
_rng_p = np.random.default_rng(3)
_samples = theta_fit[None, :] + (_Csq @ _rng_p.standard_normal((N_PARAMS, N_POST))).T

# --- 1. dispersion ---------------------------------------------------------
_bands = np.stack([bandstructure_freqs(build_K(theta_to_Lmap(t))[1]) for t in _samples])
_lo, _hi = np.percentile(_bands, [2.5, 97.5], axis=0)
_fit_b   = bandstructure_freqs(K_LAB_FIT)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for s in range(6):
    ax[0].fill_between(x, _lo[:, s], _hi[:, s], color='steelblue', alpha=0.25, lw=0)
ax[0].plot(x, _fit_b, color='steelblue', lw=1.2, label='fit')
for ti in tick_idx: ax[0].axvline(x[ti], color='k', lw=0.6)
ax[0].set_xticks([x[ti] for ti in tick_idx]); ax[0].set_xticklabels(path_labels)
ax[0].set_ylabel('Frequency (THz)'); ax[0].set_xlim(x[0], x[-1]); ax[0].set_ylim(0)
ax[0].set_title('Dispersion, 95% posterior band'); ax[0].legend(fontsize=8)

_rel = np.where(_fit_b > 1e-9, 0.5*(_hi - _lo)/np.maximum(_fit_b, 1e-9), np.nan)
for s in range(6):
    ax[1].plot(x, 100*_rel[:, s], lw=1.0, label=f'branch {s}')
for ti in tick_idx: ax[1].axvline(x[ti], color='k', lw=0.6)
ax[1].set_xticks([x[ti] for ti in tick_idx]); ax[1].set_xticklabels(path_labels)
ax[1].set_ylabel('half-width / frequency  (%)'); ax[1].set_yscale('log')
ax[1].set_title('Fractional uncertainty per branch'); ax[1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.savefig('posterior_bands.png', dpi=150); plt.show()

print(f"Median fractional uncertainty, acoustic branches 0-2: "
      f"{100*np.nanmedian(_rel[:, :3]):.2f}%")
print(f"Median fractional uncertainty, librational branches 3-5: "
      f"{100*np.nanmedian(_rel[:, 3:]):.2f}%")

# --- 2. displacement parameters -------------------------------------------
_B = np.array([isotropic_B(atom_adp_tensors(
                   bz_covariance({c: S_STAR*build_K(theta_to_Lmap(t))[1][c]
                                  for c in shell_order}, n_grid=12), apos)).mean()
               for t in _samples[:min(N_POST, 25)]])
print(f"\nMean B: fit {isotropic_B(atom_adp_tensors(bz_covariance(K_LAB_FIT, n_grid=12), apos)).mean():.3f}"
      f"  posterior {np.mean(_B):.3f} ± {np.std(_B):.3f} Å² "
      f"({100*np.std(_B)/np.mean(_B):.1f}%)   deposited {b_exp.mean():.3f} Å²")

# --- 3. the stiffnesses themselves ----------------------------------------
_spread = {c: np.std([geodesic_per_contact(theta_fit, t)[c] for t in _samples[:30]])
           for c in shell_order}
print(f"\n{'contact':14s} {'posterior spread in K':>22s} {'worst-direction factor':>24s}")
for c in shell_order:
    print(f"{str(c):14s} {_spread[c]:22.3f} {np.exp(_spread[c]):24.2f}")
print("\nRead the three together: the dispersion is pinned to a few percent, the")
print("mean B to around ten, while individual contact stiffnesses are uncertain by")
print("factors. The data measures the soft dynamics, not the springs.")


### Reading the Stiff and Sloppy Directions

The eigendirections of $(F,P)$ — data curvature $F=J^{\mathsf T}J$ against prior
curvature $P=\tfrac12\partial^2R/\partial\theta^2$ — are abstract vectors in the
126-dimensional Cholesky space. Three things make them interpretable.

**How much of each is recovered, in closed form.** For a Gaussian linear model the
posterior mean is $\hat\theta=(F+P)^{-1}F\,\theta_{\rm true}$, so along a
generalized eigendirection with eigenvalue $f=F/P$ the recovered fraction is
exactly $f/(1+f)$ — no refit required. The shrinkage curve is the honest answer to
"which directions are stiff": not a binary split but a continuum, and $n_{\rm eff}$
counts $f>1$, i.e. directions recovered more than half. A direction sitting just
above that cutoff is barely measured, so a direction sitting just above the
cutoff is barely measured.

**What each direction is, in $K$.** Mapping $\delta\theta\to\delta K_{\mathbf n}$
and non-dimensionalizing with each contact's gyration radius
($S=\mathrm{diag}(I/\rho_g,I)$, so translational and rotational entries are
comparable), each direction decomposes by **which contact** it lives on, by
**block** ($TT$, $TR$, $RR$), and within each block into an **isotropic** part —
an overall stiffening — and a **deviatoric** part, a change of anisotropy at fixed
magnitude.

**What each direction does, physically.** The induced fractional change in the
acoustic sound speeds near $\Gamma$, in the librational rest frequencies at
$\Gamma$, at the zone boundary, and in the predicted mean $B$. This is the
decomposition that answers the question directly: a direction the data pins down
tightly but which changes no observable is uninteresting, and one that is poorly
determined but moves the sound speeds is a warning.

### What the composition says

Averaged over the two subspaces, the split is sharp:

| subspace | $RR$ | $TR$ | $TT$ |
|---|---|---|---|
| data-determined | 0.02 | 0.41 | 0.57 |
| prior-carried | 0.61 | 0.36 | 0.03 |

**The diffuse data measures translational contact stiffness and barely sees
rotational contact stiffness.** Three effects compound to produce this:

1. $I\propto1/\omega^2$, and the librational branches are the stiff ones.
2. At small reduced $\mathbf q$ the acoustic eigenvectors are *pure translations* —
   $D(0)$ annihilates exactly the three translations — so the brightest part of
   the pattern, the halos, couples only through $G_T=i\mathbf qF$.
3. $L(\mathbf q)\to0$ as $\mathbf q\to0$ (the first moment about the centroid
   vanishes), so $G_R$ is further suppressed exactly where the intensity is
   largest.

The $TR$ block sits in between, at $\sim0.4$ in both subspaces, which is what one
should expect: the contacts are off-centre, so translation–rotation coupling mixes
librational character into the acoustic branches at finite $\mathbf q$ and is
therefore partly visible.

The contact ranking follows the same logic. The stiffest directions are
concentrated on the largest contact, and the sloppiest on the smallest — the
contact with the fewest atom pairs and the smallest patch contributes least to
every mode, so its stiffness is the least observable.

In [ ]:
# Data and prior curvature at the prior, for the direction analysis below.
Jm_p = weighted_jacobian(THETA_PRIOR, q_fit, G_fit, w_fit, scale=S_STAR)
# --- Anatomy of the stiff and sloppy directions ----------------------------
_Fm = Jm_p.T @ Jm_p                      # data curvature at the prior
_Pm = 0.5*regularizer_hessian(THETA_PRIOR)   # prior curvature, same convention
                                             # as the posterior covariance above
_Pi = _sqrtm_inv_sym(_Pm)
_A  = _Pi @ _Fm @ _Pi
_f, _V = np.linalg.eigh(0.5*(_A + _A.T))
_o  = np.argsort(_f)[::-1]
_f, _V = np.maximum(_f[_o], 0.0), _V[:, _o]
_DIRS = _Pi @ _V                          # back to theta coordinates
_shrink = _f/(1.0 + _f)

print(f"recovered fraction f/(1+f):  >90% for {np.sum(_shrink>0.9):3d} directions, "
      f">50% for {np.sum(_shrink>0.5):3d}, <10% for {np.sum(_shrink<0.1):3d}")
print(f"n_eff (f > 1) = {int(np.sum(_f>1))} / {N_PARAMS}")

# --- composition of a direction -------------------------------------------
_S = {c: np.diag(np.r_[np.full(3, 1.0/unique[c]['rho_g']), np.ones(3)])
      for c in shell_order}

def direction_anatomy(direction, eps=1e-4):
    """Where a theta-direction lives in K, non-dimensionalized."""
    tp, tm = THETA_PRIOR + eps*direction, THETA_PRIOR - eps*direction
    Kp, _ = build_K(theta_to_Lmap(tp)); Km, _ = build_K(theta_to_Lmap(tm))
    dK = {c: _S[c] @ ((Kp[c]-Km[c])/(2*eps)) @ _S[c] for c in shell_order}
    tot = sum(np.sum(dK[c]**2) for c in shell_order) or 1.0
    per_contact = {c: np.sum(dK[c]**2)/tot for c in shell_order}
    blocks, iso = {}, {}
    for name, sl in [('RR', slice(0,3)), ('TT', slice(3,6))]:
        b = sum(np.sum(dK[c][sl, sl]**2) for c in shell_order)
        t = sum((np.trace(dK[c][sl, sl])**2)/3 for c in shell_order)
        blocks[name] = b/tot; iso[name] = t/max(b, 1e-300)
    blocks['TR'] = sum(2*np.sum(dK[c][0:3,3:6]**2) for c in shell_order)/tot
    return per_contact, blocks, iso

# --- what a direction does, physically ------------------------------------
# Every direction is rescaled to the SAME geodesic distance before the effect is
# measured. Without that the comparison is meaningless: a unit-norm step in theta
# is a different physical perturbation for every direction, because theta holds
# Cholesky entries whose magnitude runs from ~sqrt(K_TT) ~ 7 to ~sqrt(K_RR) ~ 40.
_q_ac = np.array([[0.04,0,0],[0,0.04,0],[0,0,0.04]]) @ B_recip.T   # near Gamma
_q_zb = (B_recip @ np.array([0.5,0.5,0.5]))[None, :]
EFFECT_D = 0.20        # geodesic distance at which every direction is compared

def scale_to_geodesic(direction, target_d=EFFECT_D, tol=0.01):
    """Amplitude along `direction` giving a geodesic displacement of target_d."""
    lo, hi = 1e-6, 1e4
    for _ in range(80):
        a = np.sqrt(lo*hi)
        try:
            d = max(geodesic_per_contact(THETA_PRIOR, THETA_PRIOR + a*direction).values())
        except np.linalg.LinAlgError:
            hi = a; continue
        if abs(d - target_d) < tol*target_d: return a
        lo, hi = (a, hi) if d < target_d else (lo, a)
    return a

def direction_effect(direction, target_d=EFFECT_D):
    """Fractional change in physical observables for a displacement of fixed
    geodesic size along `direction`. Branches are sorted ascending, so [:3] are
    acoustic and [3:] librational."""
    a = scale_to_geodesic(direction, target_d)
    def obs(th):
        _, Kl = build_K(theta_to_Lmap(th))
        f_ac = bandstructure_freqs(Kl, _q_ac)[:, :3].ravel()   # acoustic near Gamma
        f_G  = bandstructure_freqs(Kl, np.zeros((1,3)))[0][3:] # librational at Gamma
        f_zb = bandstructure_freqs(Kl, _q_zb)[0]               # zone boundary, all 6
        B    = isotropic_B(atom_adp_tensors(bz_covariance(Kl, n_grid=8), apos)).mean()
        return np.r_[f_ac, f_G, f_zb, B]
    p, m = obs(THETA_PRIOR + a*direction), obs(THETA_PRIOR - a*direction)
    return np.abs(p-m)/np.maximum(0.5*np.abs(p+m), 1e-30)

_probe_idx = [0, 1, 2, max(0, int(np.sum(_f>1))-1), N_PARAMS-3, N_PARAMS-2, N_PARAMS-1]
print(f"\nComposition, and the effect of a displacement of geodesic size "
      f"{EFFECT_D} along each direction:")
print(f"{'dir':>4s} {'f':>9s} {'recov':>6s} {'RR':>5s} {'TR':>5s} {'TT':>5s} "
      f"{'iso(TT)':>8s} {'top contact':>14s} {'Δν_ac':>7s} {'Δν_lib':>7s} "
      f"{'Δν_zb':>7s} {'ΔB':>7s}")
for i in _probe_idx:
    d = _DIRS[:, i]/np.linalg.norm(_DIRS[:, i])
    pc, bl, iso = direction_anatomy(d)
    eff = direction_effect(d)
    top = max(pc, key=pc.get)
    print(f"{i:4d} {_f[i]:9.2e} {_shrink[i]:6.1%} {bl['RR']:5.2f} {bl['TR']:5.2f} "
          f"{bl['TT']:5.2f} {iso['TT']:8.2f} {str(top):>14s} "
          f"{np.mean(eff[0:9]):7.1%} {np.mean(eff[9:12]):7.1%} "
          f"{np.mean(eff[12:18]):7.1%} {eff[-1]:7.1%}")
print("   Δν_ac  acoustic branches near Γ (the sound speeds)")
print("   Δν_lib librational rest frequencies at Γ")
print("   Δν_zb  all six branches at the zone boundary")
print("   A direction with small f but large Δν_lib is one the data does NOT")
print("   determine yet which moves the optical branches -- read the dispersion")
print("   uncertainty band with that in mind.")

# --- averaged over the stiff and sloppy halves ----------------------------
_ne = max(int(np.sum(_f > 1)), 1)
print(f"\naveraged composition ({_ne} data-dominated vs {N_PARAMS-_ne} prior-carried):")
for _lab, _rng in [('stiff', range(0, _ne)), ('sloppy', range(_ne, N_PARAMS))]:
    _acc = np.zeros(3); _isotr = 0.0; _pcs = {c: 0.0 for c in shell_order}
    for i in _rng:
        d = _DIRS[:, i]/np.linalg.norm(_DIRS[:, i])
        pc, bl, iso = direction_anatomy(d)
        _acc += np.array([bl['RR'], bl['TR'], bl['TT']]); _isotr += iso['TT']
        for c in shell_order: _pcs[c] += pc[c]
    _acc /= len(_rng); _isotr /= len(_rng)
    _pcs = {c: v/len(_rng) for c, v in _pcs.items()}
    print(f"   {_lab:7s} RR {_acc[0]:.2f}   TR {_acc[1]:.2f}   TT {_acc[2]:.2f}   "
          f"isotropic fraction of TT {_isotr:.2f}")
    _rank = sorted(_pcs.items(), key=lambda kv: -kv[1])
    print(f"   {'':7s} contacts: " +
          "  ".join(f"{str(c)} {v:.2f}" for c, v in _rank[:3]))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].semilogy(np.arange(1, N_PARAMS+1), np.maximum(_f, 1e-12), 'o-', ms=3)
ax[0].axhline(1.0, color='r', ls='--', lw=1, label='data = prior')
ax[0].set_xlabel('direction index'); ax[0].set_ylabel('$f$ = data / prior curvature')
ax[0].set_title('Spectrum'); ax[0].legend(fontsize=8)
ax[1].plot(np.arange(1, N_PARAMS+1), _shrink, 'o-', ms=3)
ax[1].axhline(0.5, color='r', ls='--', lw=1, label='half recovered')
ax[1].axvline(_ne+0.5, color='k', ls=':', lw=1, label=f'$n_{{eff}}$ = {_ne}')
ax[1].set_xlabel('direction index'); ax[1].set_ylabel('recovered fraction $f/(1+f)$')
ax[1].set_ylim(-0.02, 1.02); ax[1].set_title('Shrinkage: what comes back')
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig('stiff_sloppy_anatomy.png', dpi=150); plt.show()


In [ ]:
gen_spec, n_eff = report_sloppiness(theta_fit, q_fit, G_fit, w_fit, scale=S_STAR)
print(f"\nAt the selected LAMBDA_PRIOR = {LAMBDA_PRIOR:g}, the data determines "
      f"{n_eff} of {N_PARAMS}")
print("parameter combinations more tightly than the prior does.")


## Watching the Band Structure Converge

There is no ground-truth $K$ here — the crystal's dispersion is what the data is
being used to infer — so the thick grey reference line is the final refined fit
and the coloured line is the current iterate.

Two things to watch. The acoustic branches near Γ should lock on quickly while the
librational branches drift: that asymmetry is the visible face of the sloppiness
quantified below. And a branch that collapses toward zero, or rises by an order of
magnitude relative to the prior, indicates the optimizer has walked to a boundary
of the search space rather than an interior optimum; a check below prints a
warning if that happens.

In [ ]:
freqs_final_bs = bandstructure_freqs(K_LAB_FIT)
freqs_prior_bs = bandstructure_freqs(K_LAB_PRIOR)
ymax = max(freqs_final_bs.max(), freqs_prior_bs.max())*1.15

frame_idx = np.unique(np.linspace(0, len(theta_history)-1,
                                  min(60, len(theta_history))).astype(int))
legend_handles = [Line2D([0],[0], color='0.6', lw=3.5, label='Final refined fit'),
                  Line2D([0],[0], color='steelblue', lw=1.3, label='Current iteration')]

images_bs = []
for fi in frame_idx:
    _, K_lab_fi = build_K(theta_to_Lmap(theta_history[fi]))
    freqs_fi = bandstructure_freqs(K_lab_fi)*S_STAR**0.5
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x, freqs_final_bs, color='0.6', lw=3.5, zorder=1)
    ax.plot(x, freqs_fi, color='steelblue', lw=1.3, zorder=2)
    for ti in tick_idx: ax.axvline(x[ti], color='k', lw=0.7)
    ax.set_xticks([x[ti] for ti in tick_idx]); ax.set_xticklabels(path_labels)
    ax.set_ylabel('Frequency (THz)'); ax.set_xlim(x[0], x[-1]); ax.set_ylim(0, ymax)
    ax.set_title(f'Band-structure convergence — iteration {fi}/{len(theta_history)-1}')
    ax.legend(handles=legend_handles, loc='upper right', fontsize=8)
    plt.tight_layout()
    images_bs.append(fig_to_image(fig))
    plt.close(fig)

imageio.mimsave('band_structure_convergence.gif', images_bs, fps=8, loop=0)
del images_bs; gc.collect()
print(f"Saved band_structure_convergence.gif ({len(frame_idx)} frames)")

# --- Boundary-seeking check ------------------------------------------------
zone_bnd = B_recip @ np.array([0.5, 0.5, 0.5])
f_fit   = bandstructure_freqs(K_LAB_FIT,   zone_bnd[None])[0]
f_prior = bandstructure_freqs(K_LAB_PRIOR, zone_bnd[None])[0]
ratio = f_fit/np.maximum(f_prior, 1e-12)
print(f"\nZone-boundary frequencies (THz):")
print(f"  prior: {f_prior.round(4)}")
print(f"  fit:   {f_fit.round(4)}")
print(f"  ratio: {ratio.round(3)}")
if ratio.min() < 0.1 or ratio.max() > 10:
    print("\n*** WARNING: a branch moved by more than an order of magnitude from the")
    print("    prior. That is the signature of the optimizer reaching a boundary")
    print("    rather than an interior optimum. Check the sloppiness spectrum and")
    print("    consider raising LAMBDA_PRIOR. ***")
else:
    print("\nAll branches within an order of magnitude of the prior: no boundary-seeking.")

## Experimental vs. Fitted Diffuse Scattering

The experimental $l\approx0$ plane next to the model's prediction on exactly the
same $(h,k)$ grid points. The refinement only ever saw the fitting points; this
compares the two everywhere in the plane.

The intensity panels are $\log_{10}(1+I)$; the residual panels are not
log-transformed. The absolute residual is dominated by the brightest halo pixels
even when the fit is good elsewhere, which is what the relative panel corrects for.
NaN voxels, outside the measured region, are masked from all four panels.

In [ ]:
PLANE_STRIDE = 3   # the separable plane transform makes this affordable

h_pts_cmp0 = h_pts_bg0[::PLANE_STRIDE]
k_pts_cmp0 = k_pts_bg0[::PLANE_STRIDE]
I_bg0_cmp  = I_bg0[::PLANE_STRIDE, ::PLANE_STRIDE]
print(f"Comparison plane: {len(h_pts_cmp0)}×{len(k_pts_cmp0)} points "
      f"(stride={PLANE_STRIDE}, full plane {len(h_pts_bg0)}×{len(k_pts_bg0)})")

def predicted_plane(K_lab, h_pts, k_pts, l_layer, scale=1.0):
    """G is evaluated with the separable plane transform, so a full plane costs
    about three orders of magnitude less than a dense phase matrix would."""
    HH, KK = np.meshgrid(h_pts, k_pts, indexing='ij')
    h_flat, k_flat = HH.ravel(), KK.ravel()
    l_flat = np.full_like(h_flat, l_layer)
    G_all = G_plane(np.asarray(h_pts, float), np.asarray(k_pts, float),
                    float(l_layer)).reshape(-1, 6)
    return (scale*diffuse_intensity_batch(K_lab, h_flat, k_flat, l_flat,
                                          G_arr=G_all)).reshape(HH.shape)

I_fit_map0 = predicted_plane(K_LAB_FIT, h_pts_cmp0, k_pts_cmp0, l_layer_0, scale=S_FIT)
exp_mask0  = np.isfinite(I_bg0_cmp)

def plot_diffuse_comparison(h_pts, k_pts, I_exp, I_fit, mask, fname, rel_clip=1.0):
    I_exp_m = np.where(mask, I_exp, np.nan)
    I_fit_m = np.where(mask, I_fit, np.nan)
    pos_vals = np.concatenate([I_exp_m[mask & (I_exp_m > 0)], I_fit_m[mask & (I_fit_m > 0)]])
    vmax = np.percentile(pos_vals, 97) if len(pos_vals) else 1.0
    resid = np.where(mask, I_fit_m - I_exp_m, np.nan)
    rmax = np.nanpercentile(np.abs(resid), 99) if np.isfinite(resid).any() else 1.0
    rel = resid/np.maximum(np.abs(I_exp_m), 0.02*vmax)
    extent = [h_pts[0], h_pts[-1], k_pts[0], k_pts[-1]]

    fig, axes = plt.subplots(2, 2, figsize=(11, 10))
    im0 = axes[0,0].imshow((np.log1p(np.clip(I_exp_m, 0, vmax))/np.log(10)).T, origin='lower',
                           cmap='inferno', extent=extent, aspect='auto')
    axes[0,0].set_title('Experimental (CXIDB 128)')
    plt.colorbar(im0, ax=axes[0,0], label=r'$\log_{10}(1+I)$')
    im1 = axes[0,1].imshow((np.log1p(np.clip(I_fit_m, 0, vmax))/np.log(10)).T, origin='lower',
                           cmap='inferno', extent=extent, aspect='auto')
    axes[0,1].set_title('Fitted $K$')
    plt.colorbar(im1, ax=axes[0,1], label=r'$\log_{10}(1+I)$')
    im2 = axes[1,0].imshow(resid.T, origin='lower', cmap='RdBu_r',
                           extent=extent, aspect='auto', vmin=-rmax, vmax=rmax)
    axes[1,0].set_title('Absolute residual (fit − experimental)')
    plt.colorbar(im2, ax=axes[1,0], label=r'$\Delta I$ (raw units, not log)')
    im3 = axes[1,1].imshow(np.clip(rel, -rel_clip, rel_clip).T, origin='lower', cmap='RdBu_r',
                           extent=extent, aspect='auto', vmin=-rel_clip, vmax=rel_clip)
    axes[1,1].set_title(f'Relative residual, clipped to ±{100*rel_clip:.0f}%')
    plt.colorbar(im3, ax=axes[1,1], label=r'$\Delta I / I_{\rm exp}$')
    for ax in axes.flat:
        ax.set_xlabel('h'); ax.set_ylabel('k')
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

    ok = np.isfinite(resid) & np.isfinite(I_exp_m)
    cc = np.corrcoef(I_fit_m[ok], I_exp_m[ok])[0,1]
    print(f"Saved {fname}")
    print(f"Plane CC (fit vs. experimental): {cc:.4f}")
    print(f"Absolute residual RMS: {np.sqrt(np.nanmean(resid**2)):.4g}  "
          f"(I_exp RMS: {np.sqrt(np.nanmean(I_exp_m**2)):.4g})")
    print(f"Fraction of the experimental variance explained: {cc**2:.1%}")

plot_diffuse_comparison(h_pts_cmp0, k_pts_cmp0, I_bg0_cmp, I_fit_map0, exp_mask0,
                        'diffuse_map_experimental_vs_fit_hk0.png')

## Predicted vs. Deposited Atomic Displacement Parameters

$\Sigma=\langle(\Omega,v)(\Omega,v)^{\mathsf T}\rangle$ from integrating
$D(\mathbf q)^{-1}$ over the Brillouin zone; projecting onto an atom via
$\delta\mathbf r=\mathbf v+\boldsymbol\Omega\times(\mathbf r-\mathbf r_{\rm cm})$
gives its full anisotropic tensor $U=J_r\Sigma J_r^{\mathsf T}$, with
$B=\tfrac{8\pi^2}{3}\mathrm{Tr}(U)$.

This is the independent check, and nothing above uses it: $K$ was fixed absolutely
by the unit argument, not by matching a deposited $B$. Three genuine predictions
follow, in this order.

1. **The mean.** It should come out **below** the deposited mean, because the
   deposited $B$ also absorbs internal motion, side-chain disorder and
   substitutional heterogeneity that a single-rigid-body lattice model cannot
   reproduce. A prediction above the deposited mean means $K$ is too soft; a
   prediction exactly at it would mean the lattice accounts for all of the
   disorder, which is not physical.
2. **The per-atom shape.** How $B$ varies across the molecule. Note the floor: a
   rigid body predicts $B$ growing with distance from the centre of mass, which
   correlates with real $B$ at $r\approx0.4$–$0.6$ from geometry alone, so a
   correlation in that range is not by itself evidence the dynamics are right. The
   geometry-only baseline is computed and printed for comparison.
3. **The anisotropy.** The shape of each displacement ellipsoid, which geometry
   alone does not supply, and the most informative of the three.

In [ ]:
# Σ, the ADP projection, and the convergence check are evaluated here;
# their definitions sit with the other machinery further up, because the
# posterior propagation needs them too.
Sigma_fit = bz_covariance(K_LAB_FIT)
U_fit = atom_adp_tensors(Sigma_fit, apos)
B_fit = isotropic_B(U_fit)

check_bz_convergence(K_LAB_FIT)
print()
print(f"PREDICTED mean B (fitted K):  {B_fit.mean():8.3f} Å²")
print(f"Deposited mean B (6o2h):      {b_exp.mean():8.3f} Å²")
ratio = B_fit.mean()/b_exp.mean()
print(f"ratio predicted/deposited:    {ratio:8.3f}")
if ratio > 1.0:
    print("\n  > 1: the model predicts MORE displacement than was refined from the")
    print("  Bragg data, so the fitted contacts are too soft. Something is wrong --")
    print("  check s*, the resolution range, and the sloppiness spectrum.")
else:
    print(f"\n  < 1, as it should be: the lattice model accounts for {100*ratio:.0f}% of the")
    print("  deposited B, with the remainder from internal and substitutional")
    print("  disorder outside this model. That split is the result, not a defect.")

In [ ]:
res_ids = []
for ch in st[0]:
    for res in ch:
        for atom in res:
            res_ids.append((ch.name, res.seqid.num))
res_index = {}
res_inverse = np.empty(len(res_ids), dtype=int)
for i, key_ in enumerate(res_ids):
    res_inverse[i] = res_index.setdefault(key_, len(res_index))
n_res = len(res_index)

def per_residue_mean(v):
    out = np.zeros(n_res); counts = np.zeros(n_res)
    np.add.at(out, res_inverse, v); np.add.at(counts, res_inverse, 1)
    return out/counts

B_exp_res, B_fit_res = per_residue_mean(b_exp), per_residue_mean(B_fit)

# Geometry-only baseline: what correlation does a rigid body give from the lever
# arm alone, with no dynamical content at all? Anything at or below this is not
# evidence the fitted dynamics are right.
lever = np.linalg.norm(apos - r_cm_at, axis=1)**2
r_geom = np.corrcoef(b_exp, lever)[0,1]
r_fit  = np.corrcoef(b_exp, B_fit)[0,1]

fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
ax = axes[0]
ax.scatter(b_exp, B_fit, s=6, alpha=0.4)
lim = [0, max(b_exp.max(), B_fit.max())*1.05]
ax.plot(lim, lim, 'k--', lw=1); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('$B$ — deposited 6o2h (Å²)'); ax.set_ylabel('$B$ — fitted $K$ (Å²)')
ax.set_title(f'Per-atom isotropic $B$   (r = {r_fit:.3f})')

ax = axes[1]
ax.plot(B_exp_res, label='deposited', lw=1.3)
ax.plot(B_fit_res, label='fitted $K$', lw=1.3, alpha=0.8)
ax.set_xlabel('residue index'); ax.set_ylabel('$B$ (Å²)')
ax.set_title('Per-residue mean $B$'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('adp_comparison.png', dpi=150); plt.show()

print(f"Pearson r, per-atom B (deposited vs. fitted):     {r_fit:.3f}")
print(f"Pearson r, deposited B vs. |r - r_cm|² alone:     {r_geom:.3f}")
print(f"  -> the model beats the geometry-only baseline by {r_fit - r_geom:+.3f}.")
print("  A fitted correlation at or below the baseline means the dynamics are")
print("  contributing nothing beyond the rigid-body lever arm.")

### Anisotropic Comparison

The isotropic trace discards the shape of each displacement ellipsoid. Where 6o2h
carries ANISOU records, the full tensor comparison checks whether the refinement
recovers that shape: the six independent components of $U$ against each other, and
each atom's anisotropy ratio (largest/smallest principal displacement).

This is the most informative of the ADP tests, because unlike the isotropic trace
it cannot be reproduced by the lever-arm geometry alone. A model that matches on
the trace but gets the anisotropy wrong would look acceptable above and fail here.

In [ ]:
def anisotropy_ratio(U):
    ev = np.linalg.eigvalsh(U)
    return ev[:, -1] / np.clip(ev[:, 0], 1e-12, None)

comp_idx   = [(0,0),(1,1),(2,2),(0,1),(0,2),(1,2)]
comp_names = ['$U_{11}$','$U_{22}$','$U_{33}$','$U_{12}$','$U_{13}$','$U_{23}$']

if has_aniso.mean() > 0.3:
    fig, axes = plt.subplots(1, 2, figsize=(11,4.5))

    ax = axes[0]
    for (i,j), name in zip(comp_idx, comp_names):
        ax.scatter(u_exp[has_aniso,i,j], U_fit[has_aniso,i,j], s=5, alpha=0.4, label=name)
    lim = [min(u_exp[has_aniso].min(), U_fit[has_aniso].min()),
           max(u_exp[has_aniso].max(), U_fit[has_aniso].max())]
    ax.plot(lim, lim, 'k--', lw=1)
    ax.set_xlabel('$U$ component — deposited 6o2h (Å²)'); ax.set_ylabel('$U$ component — fitted $K$ (Å²)')
    ax.set_title('Anisotropic tensor components: deposited vs. fitted')
    ax.legend(fontsize=7, ncol=2)

    ax = axes[1]
    aniso_exp = anisotropy_ratio(u_exp[has_aniso])
    aniso_fit = anisotropy_ratio(U_fit[has_aniso])
    ax.scatter(aniso_exp, aniso_fit, s=6, alpha=0.4)
    lim2 = [1, max(aniso_exp.max(), aniso_fit.max())*1.05]
    ax.plot(lim2, lim2, 'k--', lw=1)
    ax.set_xlabel('anisotropy ratio — deposited'); ax.set_ylabel('anisotropy ratio — fitted $K$')
    ax.set_title('Displacement-ellipsoid anisotropy: deposited vs. fitted')

    plt.tight_layout(); plt.savefig('adp_anisotropic_comparison.png', dpi=150); plt.show()
else:
    print(f"Only {has_aniso.mean():.1%} of atoms in 6o2h carry ANISOU records; "
          f"skipping the anisotropic comparison. Isotropic B-factors are "
          f"compared above regardless.")


## Phonon Mode Visualization

Each phonon mode at a chosen $\mathbf q$-point is rendered as a looping GIF
of the rigid-body motion, using the model **fitted to the experimental
data**. $|\Omega|$ and $|v|$ are the rotational and translational norms of
the mass-weighted eigenvector (normalized so
$\Omega^{\mathsf T}J\Omega+v^{\mathsf T}mv=1$); their ratio gives the
rotational kinetic-energy fraction, used below to label each mode
librational, translational, or mixed. Displacement amplitude is defined as
the largest per-atom displacement in Å, not the center-of-mass displacement.


In [ ]:
Q_VIZ     = np.array([0.5, 0.25, 0.0])  # fractional — zone-boundary, off-axis
AMPLITUDE = 3.0    # Å — maximum per-atom displacement
N_FRAMES  = 30
GIF_FPS   = 12

q_viz  = B_recip @ Q_VIZ
D_viz  = dynamical_matrix(q_viz, unique, K_LAB_FIT)
# Complex Hermitian, as everywhere else -- Re(D) has different eigenvalues and the
# acoustic (smallest) ones are worst affected, so a .real here would animate a
# different set of modes from the ones that were fit.
Dw_viz = np.einsum('ij,jk,lk->il', Msq_inv, D_viz, Msq_inv)
ev_v, evec_v = np.linalg.eigh(Dw_viz)
ev_v      = np.maximum(ev_v, 0)
freqs_viz = np.sqrt(ev_v) * freq_unit
evecs_lab = Msq_inv.T @ evec_v          # complex, one column per mode

# A phonon eigenvector at general q is genuinely complex: the physical motion is
# Re(e e^{i(q.R - wt)}), so components lead each other in phase. For a single-cell
# animation we rotate each mode by the global phase that makes it as real as
# possible (exact for a standing mode, a good approximation otherwise) and record
# how much amplitude the residual imaginary part carries.
_phase = np.exp(-1j*np.angle(evecs_lab[np.abs(evecs_lab).argmax(axis=0),
                                       np.arange(6)]))
evecs_lab = evecs_lab * _phase[None, :]
_im_frac = np.linalg.norm(evecs_lab.imag, axis=0)/np.linalg.norm(evecs_lab, axis=0)
print(f"Residual out-of-phase amplitude per mode: {_im_frac.round(3)}")
print("  (0 = a pure standing mode the single-cell animation represents exactly;")
print("   large values mean the components genuinely lead each other in phase.)")

print(f"Modes at q={Q_VIZ}:")
hdr = f"{'Mode':>5} {'Freq (THz)':>14} {'|Ω|':>9} {'|v|':>9}  {'rot KE %':>9}  character"
print(hdr); print('-'*len(hdr))
for s in range(6):
    Om = evecs_lab[:3, s].real; v = evecs_lab[3:, s].real
    rot_KE, trans_KE = float(Om @ J @ Om), float(m_total * (v@v))
    rot_pct = 100 * rot_KE / max(rot_KE + trans_KE, 1e-30)
    char = 'librational' if rot_pct > 60 else ('translational' if rot_pct < 40 else 'mixed')
    print(f"  {s:3d}  {freqs_viz[s]:14.6f}  {np.linalg.norm(Om):9.4f}  "
          f"{np.linalg.norm(v):9.4f}  {rot_pct:9.1f}%  {char}")


In [ ]:
# Molecular isosurface: marching cubes on the CALCULATED model density
# (rho_model, built alongside the phases -- the transform used for G runs on
# the unwrapped hybrid density, not on this grid), downsampled
# for speed, keeping only the largest connected component so periodic-image
# and solvent-void fragments are discarded, then Lambert-shaded per face so
# shape and roughness stay visible even at partial transparency.
base_pos = apos.copy()
pts_cm   = base_pos - r_cm_at

MC_DS = 0.5
rho_mc = nd_zoom(rho_model.astype(np.float32), MC_DS, order=1)
nu_mc, nv_mc, nw_mc = rho_mc.shape
_iso_level = rho_mc.max() * 0.05
mc_verts_grid, mc_faces_raw, _normals, _vals = marching_cubes(rho_mc, level=_iso_level)
_frac = mc_verts_grid / np.array([nu_mc, nv_mc, nw_mc])
mc_verts_raw = (A_orth @ _frac.T).T

def _largest_component(verts, faces):
    n = len(verts)
    idx_i = np.concatenate([faces[:,0], faces[:,1], faces[:,2]])
    idx_j = np.concatenate([faces[:,1], faces[:,2], faces[:,0]])
    adj   = csr_matrix((np.ones(len(idx_i), dtype=np.int8), (idx_i, idx_j)), shape=(n, n))
    _, labels = sc_connected_components(adj, directed=False)
    keep_label = np.bincount(labels).argmax()
    keep = np.where(labels == keep_label)[0]
    remap = np.full(n, -1, dtype=int); remap[keep] = np.arange(len(keep))
    face_ok = (labels[faces] == keep_label).all(axis=1)
    return verts[keep], remap[faces[face_ok]]

mc_verts_cart, mc_faces = _largest_component(mc_verts_raw, mc_faces_raw)
mc_verts_cm = mc_verts_cart - r_cm_at
print(f"Isosurface: {len(mc_verts_cm)} verts, {len(mc_faces)} tris "
      f"(largest connected component, level={_iso_level:.3f} e/Å³)")

_LIGHT = np.array([0.5, 0.8, 1.0]); _LIGHT /= np.linalg.norm(_LIGHT)

def _shade(tri_array, hex_color, alpha, ambient=0.35):
    v0, v1, v2 = tri_array[:,0], tri_array[:,1], tri_array[:,2]
    n = np.cross(v1 - v0, v2 - v0)
    mag = np.linalg.norm(n, axis=1, keepdims=True)
    n /= np.where(mag > 1e-12, mag, 1.0)
    intensity = ambient + (1 - ambient) * np.abs(n @ _LIGHT)
    r = int(hex_color[1:3], 16) / 255
    g = int(hex_color[3:5], 16) / 255
    b = int(hex_color[5:7], 16) / 255
    return np.column_stack([intensity*r, intensity*g, intensity*b, np.full(len(tri_array), alpha)])

def displaced_mc(mode_idx, scale):
    Om_raw = evecs_lab[:3, mode_idx].real.copy()
    v_raw  = evecs_lab[3:, mode_idx].real.copy()
    norm_vec = np.sqrt(Om_raw @ Om_raw + v_raw @ v_raw)
    if norm_vec < 1e-10:
        return mc_verts_cm + r_cm_at
    Om_u, v_u = Om_raw/norm_vec, v_raw/norm_vec
    delta_unit = v_u[None,:] + np.cross(Om_u[None,:], pts_cm)
    max_d = np.linalg.norm(delta_unit, axis=1).max()
    if max_d < 1e-12:
        return mc_verts_cm + r_cm_at
    fac = scale / max_d
    Om_s, v_s = fac*Om_u, fac*v_u
    if abs(fac)*np.linalg.norm(Om_u) > 1e-10:
        return Rot.from_rotvec(Om_s).apply(mc_verts_cm) + r_cm_at + v_s
    return mc_verts_cm + r_cm_at + v_s

eq_bbox_min = mc_verts_cart.min(axis=0)
eq_bbox_max = mc_verts_cart.max(axis=0)


In [ ]:
MODE_COLORS = ['#4e9de0', '#e05c5c', '#4fba74', '#e0b14e', '#a56be0', '#e07e4e']
SURF_ALPHA  = 0.65
ELEV, AZIM  = 20, -60

def make_mode_gif(mode_idx, amplitude=AMPLITUDE, n_frames=N_FRAMES, fps=GIF_FPS, fname=None, figsize=(6,5)):
    """Render one phonon mode as a looping GIF and save it to disk (no
    in-notebook display, and memory is freed after each mode)."""
    if fname is None:
        qstr = '_'.join(f'{x:.2f}' for x in Q_VIZ)
        fname = f'mode_{mode_idx}_q{qstr}.gif'
    color  = MODE_COLORS[mode_idx % len(MODE_COLORS)]
    scales = amplitude * np.sin(2*np.pi*np.arange(n_frames)/n_frames)
    pad = amplitude * 2.5
    xlim = (eq_bbox_min[0]-pad, eq_bbox_max[0]+pad)
    ylim = (eq_bbox_min[1]-pad, eq_bbox_max[1]+pad)
    zlim = (eq_bbox_min[2]-pad, eq_bbox_max[2]+pad)

    Om_r = evecs_lab[:3, mode_idx].real
    rot_KE   = float(Om_r @ J @ Om_r)
    trans_KE = m_total * float(np.linalg.norm(evecs_lab[3:, mode_idx].real)**2)
    rot_pct  = 100*rot_KE/(rot_KE+trans_KE+1e-30)

    images = []
    for sc in scales:
        fig = plt.figure(figsize=figsize, facecolor='#0d1117')
        ax  = fig.add_subplot(111, projection='3d', facecolor='#0d1117')
        try:
            verts = displaced_mc(mode_idx, sc)
            tris  = verts[mc_faces]
            rgba  = _shade(tris, color, SURF_ALPHA)
            poly  = Poly3DCollection(tris, facecolors=rgba, linewidths=0, edgecolor='none')
            ax.add_collection3d(poly)
            ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
            ax.set_axis_off(); ax.view_init(elev=ELEV, azim=AZIM)
            ax.set_title(f'Mode {mode_idx}   {freqs_viz[mode_idx]:.4f} THz\n'
                         f'q={list(Q_VIZ)}   rot {rot_pct:.0f}%  trans {100-rot_pct:.0f}%',
                         color='white', fontsize=8.5, pad=3)
            plt.tight_layout(pad=0.2)
            images.append(fig_to_image(fig))
        finally:
            plt.close(fig)

    imageio.mimsave(fname, images, fps=fps, loop=0)
    del images; gc.collect()
    return fname

print(f"Generating surface GIFs at q={Q_VIZ} (amplitude={AMPLITUDE} Å max per-atom)...")
for s in range(6):
    print(f"Mode {s}: {freqs_viz[s]:.6f} THz", end="  ... ")
    try:
        path = make_mode_gif(s)
        print(f"saved -> {path}")
    except Exception as e:
        print(f"FAILED: {e}")
    gc.collect()
print("Load any GIF with: display(IPImage(filename='mode_N_q....gif'))")


In [ ]:
# Supercell wave: each cell (i,j,k) is a rigid copy displaced by
# amplitude * cos(2*pi*q_frac.[i,j,k] + phase). The central cell is
# colored distinctly from its neighbors so the propagating wave pattern
# can be read off against a fixed reference point.
SUPER_N1, SUPER_N2, SUPER_N3 = 3, 3, 1   # a full 3x3x3 renders 27 isosurfaces per
                                          # frame and can exhaust notebook-kernel memory;
                                          # 3x3x1 already shows the in-plane wave pattern
MODE_SUPER   = 0
SUPER_COLOR  = '#4e9de0'
SUPER_ALPHA  = 0.38
CENTER_COLOR = '#e05c5c'
CENTER_ALPHA = 0.55

def make_supercell_gif(mode_idx=MODE_SUPER, amplitude=AMPLITUDE, n_frames=N_FRAMES, fps=GIF_FPS,
                       n1=SUPER_N1, n2=SUPER_N2, n3=SUPER_N3, fname=None, figsize=(10,9)):
    if fname is None:
        qstr = '_'.join(f'{x:.2f}' for x in Q_VIZ)
        fname = f'supercell_mode{mode_idx}_q{qstr}.gif'

    cells_ijk = [(i,j,k) for i in range(n1) for j in range(n2) for k in range(n3)]
    center_ijk = (n1//2, n2//2, n3//2)
    phase0 = np.array([2*np.pi*(Q_VIZ[0]*i + Q_VIZ[1]*j + Q_VIZ[2]*k) for i,j,k in cells_ijk])
    T_cell = np.array([i*a1 + j*a2 + k*a3 for i,j,k in cells_ijk])

    all_eq = np.vstack([mc_verts_cart + T for T in T_cell])
    pad = amplitude * 2.5
    xlim = (all_eq[:,0].min()-pad, all_eq[:,0].max()+pad)
    ylim = (all_eq[:,1].min()-pad, all_eq[:,1].max()+pad)
    zlim = (all_eq[:,2].min()-pad, all_eq[:,2].max()+pad)

    images = []
    for fi in range(n_frames):
        t = 2*np.pi*fi/n_frames
        fig = plt.figure(figsize=figsize, facecolor='#0d1117')
        ax  = fig.add_subplot(111, projection='3d', facecolor='#0d1117')
        try:
            for (i,j,k), T, ph0 in zip(cells_ijk, T_cell, phase0):
                sc = amplitude * np.cos(ph0 + t)
                verts = displaced_mc(mode_idx, sc) + T
                tris_v = verts[mc_faces]
                is_center = (i,j,k) == center_ijk
                color = CENTER_COLOR if is_center else SUPER_COLOR
                alpha = CENTER_ALPHA if is_center else SUPER_ALPHA
                rgba = _shade(tris_v, color, alpha)
                poly = Poly3DCollection(tris_v, facecolors=rgba, linewidths=0, edgecolor='none')
                ax.add_collection3d(poly)

            ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
            ax.set_axis_off(); ax.view_init(elev=25, azim=-55)
            ax.set_title(f'Supercell phonon — Mode {mode_idx}  {freqs_viz[mode_idx]:.4f} THz\n'
                         f'q={list(Q_VIZ)}   {n1}×{n2}×{n3} cells (center cell highlighted)',
                         color='white', fontsize=8.5, pad=3)
            plt.tight_layout(pad=0.2)
            images.append(fig_to_image(fig))
        finally:
            plt.close(fig)

    imageio.mimsave(fname, images, fps=fps, loop=0)
    del images; gc.collect()
    return fname

print(f"Generating {SUPER_N1}×{SUPER_N2}×{SUPER_N3} supercell GIF for mode {MODE_SUPER}...")
try:
    sc_path = make_supercell_gif()
    print(f"Saved -> {sc_path}")
except Exception as e:
    print(f"Supercell GIF failed: {e}")
    import traceback; traceback.print_exc()


## Summary

| Output | File | Description |
|--------|------|-------------|
| Solvent contrast | `solvent_contrast.png` | Density profile into the protein, contrast profile, contrast histogram |
| Band structure (prior) | `band_structure_prior.png` | Phonon dispersion Γ–X–Y–Z–Γ before fitting |
| Resolution diagnostics | `resolution_snr_scan.png` | SNR, coverage and negative fraction vs. resolution |
| Bragg-distance diagnostics | `bragg_distance_diagnostic.png` | Negative fraction, mean $I$, and population vs. halo distance |
| Sampled q-points | `sampled_q_points.png` | 3D view plus slab overlays on the experimental map |
| Refinement fit quality | `refinement_fit_quality.png` | $I_{\rm obs}$ vs $I_{\rm model}$, plus held-out CC by resolution |
| Band-structure convergence | `band_structure_convergence.gif` | Convergence toward the final fit |
| Sloppiness spectrum | `sloppiness_spectrum.png` | Gauss-Newton eigenvalues; how many parameters the data determines |
| Experimental vs. fitted map | `diffuse_map_experimental_vs_fit_hk0.png` | Data, fit, and residuals |
| ADP comparison (isotropic) | `adp_comparison.png` | Per-atom and per-residue $B$, with a geometry-only baseline |
| ADP comparison (anisotropic) | `adp_anisotropic_comparison.png` | Full $U$ components and anisotropy ratio |
| Mode GIFs / supercell | `mode_N_q*.gif`, `supercell_mode*.gif` | Fitted-model phonon modes |

**Molecular transform.** $F$ and $L$ come from the hybrid electron density —
measured amplitudes with model phases — reduced to the solvent-contrast density
$\rho_{\rm eff}=\rho-\rho_{\rm sol}M$ and unwrapped onto a single molecule, each
voxel carrying the phase $e^{+2\pi i(h,k,l)\cdot\mathbf n_v}$ of the lattice image
it belongs to. Checked against the measured amplitudes at integer $hkl$ binned by
resolution, against brute-force summation over unwrapped coordinates at fractional
$hkl$, and plane path against scattered path.

**Contact stiffness and prior.** Six distinct contacts — $P1$ has no point
symmetry, so they are not symmetry-related — each a free $6\times6$ Cholesky
factor in a frame anchored at the measured contact centroid. The prior is the
stiffness of $n$ isotropic point springs at the atom-pair midpoints, supplying the
contact-count scaling, the $\kappa_R/\kappa_T\sim\rho_g^2$ ratio and the patch
anisotropy. Refinement is regularized toward it under the geodesic metric on
positive-definite matrices, which respects the mixed units of $K$ and is invariant
to the choice of reference point and axes.

**Scale.** The $K$/scale degeneracy is removed from the parameter space by
refining the normalized shape of $K$ with $s$ profiled in closed form, then closed
with units ($s\equiv1$, since $I_{\rm model}$ is in electrons² per unit cell). No
ADP information enters the fit, so the predicted $B$-factors and ANISOU tensors
remain an independent check — and the predicted mean $B$ is expected to fall below
the deposited mean.

**Data selection.** A halo profile (shell × sector averages around every
reciprocal-lattice point) plus a stratified mid-zone sample, over a resolution
range chosen for model validity. Negative intensities are kept, and the effective
sample size is reported alongside every nominal count.

**Refinement.** L-BFGS-B with an analytic gradient checked against finite
differences, vectorized. $R$ and $CC$ are reported per resolution shell, and the
Gauss-Newton spectrum states how many of the 126 parameters the data determines.